<a href="https://colab.research.google.com/github/WangariLM/Fine-Tuning-Qwen2.5-using-HotPot-QA-dataset/blob/main/Final_HotpotQA_Qwen2_5_1_5b_FineTuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Cell 1: Mount Drive and Create a Folder for the Project and Data
from google.colab import drive
#drive.mount('/content/drive')
drive.mount('/content/drive', force_remount=True)
import os
os.makedirs('/content/drive/MyDrive/hotpotqa-rag/data', exist_ok=True)

Mounted at /content/drive


In [ ]:
import json
import re
from pathlib import Path

if not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")

CLEAN_B = Path("/content/drive/MyDrive/hotpotqa-rag/data/distributions/clean/distribution_B.jsonl")
assert CLEAN_B.exists(), f"Missing {CLEAN_B}. Check that the Drive folder path is right."


def base_title(t):
    return re.sub(r"\s*\(.*?\)", "", t).strip().lower()


total = visible = 0
not_visible_examples = []
with open(CLEAN_B, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        rec = json.loads(line)
        withheld = rec["provenance"]["withheld_title"]
        context, _, question = rec["messages"][0]["content"].rpartition("\n\nQuestion:")
        total += 1
        if base_title(withheld) in context.lower():
            visible += 1
        elif len(not_visible_examples) < 6:
            not_visible_examples.append((question.strip(), withheld, rec["messages"][1]["content"]))

print(f"B examples: {total:,}")
print(f"Entity named in the Search: target is visible in the shown context: {visible:,} ({100 * visible / total:.1f}%)")
print(f"Not visible anywhere in the context: {total - visible:,} ({100 * (total - visible) / total:.1f}%)")
print("\nExamples where it is not visible:")
for q, w, target in not_visible_examples:
    print(f"  Q: {q}\n     withheld title: {w} | training target: {target}")

B examples: 24,456
Entity named in the Search: target is visible in the shown context: 21,119 (86.4%)
Not visible anywhere in the context: 3,337 (13.6%)

Examples where it is not visible:
  Q: Noritada Saneyoshi, is a former Japanese football player and manager, in 1993, he was a member of which Japan national football team?
     withheld title: Football at the Summer Universiade | training target: Search: Football at the Summer Universiade saneyoshi japanese noritada
  Q: What film for Disney was a 1992 American animated comedy musical romantic fantasy adventure film produced by Walt Disney Feature Animation that had an American animation studio founded by Terry and Sue Shakespeare with David Molina in 1989 that was an uncredited additional animation facility for the film?
     withheld title: Creative Capers Entertainment | training target: Search: Creative Capers Entertainment shakespeare uncredited additional
  Q: Deez Nuts was a satirical candidate, portrayed by Brady C. Olson, wh

In [ ]:
import json
from pathlib import Path
from collections import Counter

TRAIN_SUBSAMPLE_PATH = Path("/content/drive/MyDrive/hotpotqa-rag/data/train_subsample.json")
with open(TRAIN_SUBSAMPLE_PATH, "r", encoding="utf-8") as f:
    train_subsample = json.load(f)

def title_named_in_question(title: str, question: str) -> bool:
    q_lower = question.lower()
    words = [w for w in title.lower().split() if len(w) > 3]
    return any(w in q_lower for w in words)

comparison_qs = [rec for rec in train_subsample.values() if rec["type"] == "comparison"]

both_named = one_named = none_named = 0
not_fully_named_by_level = Counter()

for rec in comparison_qs:
    hits = sum(title_named_in_question(t, rec["question"]) for t in rec["gold_titles"])
    if hits == 2:
        both_named += 1
    elif hits == 1:
        one_named += 1
        not_fully_named_by_level[rec["level"]] += 1
    else:
        none_named += 1
        not_fully_named_by_level[rec["level"]] += 1

total = len(comparison_qs)
print(f"Comparison questions: {total:,}")
print(f"  Both gold titles named:  {both_named:,} ({both_named/total*100:.1f}%)")
print(f"  Only one named:          {one_named:,} ({one_named/total*100:.1f}%)")
print(f"  Neither named:           {none_named:,} ({none_named/total*100:.1f}%)")
print(f"\n'Not fully named' by level: {dict(not_fully_named_by_level)}")

Comparison questions: 5,640
  Both gold titles named:  5,529 (98.0%)
  Only one named:          110 (2.0%)
  Neither named:           1 (0.0%)

'Not fully named' by level: {'easy': 26, 'hard': 35, 'medium': 50}


In [ ]:
#Cell 2: Download HotpotQA Dataset from Huggingface and save to Drive
from datasets import load_dataset
import json

#Download both splits
#dataset = load_dataset("hotpot_qa","distractor")
dataset = load_dataset("hotpotqa/hotpot_qa", "distractor")

#Save raw files to Drive as JSON
dataset["train"].to_json("/content/drive/MyDrive/hotpotqa-rag/data/hotpot_train.json")
dataset["validation"].to_json("/content/drive/MyDrive/hotpotqa-rag/data/hotpot_dev.json")

print("Train", len(dataset["train"]))
print("Dev", len(dataset["validation"]))



README.md:   0%|          | 0.00/9.52k [00:00<?, ?B/s]

distractor/train-00000-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  166MB            

distractor/train-00000-of-00002.parquet: downloading bytes:           |  0.00B            

distractor/train-00001-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  166MB            

distractor/train-00001-of-00002.parquet: downloading bytes:           |  0.00B            

...

Generating train split:   0%|          | 0/90447 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/7405 [00:00<?, ? examples/s]

Creating json from Arrow format:   0%|          | 0/91 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/8 [00:00<?, ?ba/s]

Train 90447
Dev 7405


In [ ]:
#3. Verify the Download
import json

# Load from Drive (this is how we will load it from now on)
with open("/content/drive/MyDrive/hotpotqa-rag/data/hotpot_train.json", "r") as f:
    first_line = json.loads(f.readline())

print("Keys:", first_line.keys())
print("Question:", first_line["question"][:100])
print("Answer:", first_line["answer"])
print("Type:", first_line["type"])
print("Level:", first_line["level"])
print("Supporting facts:", first_line["supporting_facts"])
#print("Context articles:", [c[0] for c in first_line["context"][:5]])
print("Context articles:", first_line["context"]["title"][:8])

Keys: dict_keys(['id', 'question', 'answer', 'type', 'level', 'supporting_facts', 'context'])
Question: Which magazine was started first Arthur's Magazine or First for Women?
Answer: Arthur's Magazine
Type: comparison
Level: medium
Supporting facts: {'title': ["Arthur's Magazine", 'First for Women'], 'sent_id': [0, 0]}
Context articles: ['Radio City (Indian radio station)', 'History of Albanian football', 'Echosmith', "Women's colleges in the Southern United States", 'First Arthur County Courthouse and Jail', "Arthur's Magazine", '2014–15 Ukrainian Hockey Championship', 'First for Women']


In [ ]:
# Cell 4.0: Clean Slate (run this first)
from pathlib import Path

for f in ["gold_lookup.json", "dev_gold_lookup.json",
          "train_subsample.json", "dev_subsample.json", "val_subsample.json",
          "master_title_list.json"]:
    p = Path(f"/content/drive/MyDrive/hotpotqa-rag/data/{f}")
    if p.exists():
        p.unlink()
        print(f"Deleted stale: {f}")
    else:
        print(f"Already absent: {f}")

Deleted stale: gold_lookup.json
Deleted stale: dev_gold_lookup.json
Deleted stale: train_subsample.json
Deleted stale: dev_subsample.json
Deleted stale: val_subsample.json
Deleted stale: master_title_list.json


In [ ]:
# Cell 5: Profile the Corpus
# Profile data to understand it before restructuring it.
import json
from pathlib import Path
from collections import Counter
from typing import Dict, List, Set

RAW_PATH = Path("/content/drive/MyDrive/hotpotqa-rag/data/hotpot_train.json")

# ── Metrics containers ────────────────────────────────────────────────────────
type_counter = Counter()
level_counter = Counter()
unique_gold_articles: Set[str] = set()
gold_article_sentence_counts: List[int] = []   # per-occurrence sentence counts
unique_article_sentences: Dict[str, int] = {}  # deduplicated by title

total_examples = 0
malformed = 0

print(f"Profiling {RAW_PATH.name} ...")

# ── Single pass over raw data ─────────────────────────────────────────────────
with open(RAW_PATH, "r", encoding="utf-8") as f:
    for line_num, line in enumerate(f, start=1):
        line = line.strip()
        if not line:
            continue

        try:
            ex = json.loads(line)
        except json.JSONDecodeError:
            malformed += 1
            continue

        total_examples += 1

        # 1. Type & level distribution
        type_counter[ex.get("type", "unknown")] += 1
        level_counter[ex.get("level", "unknown")] += 1

        # 2. Build title → sentence list mapping from context
        #    HF datasets schema: context = {"title": [...], "sentences": [...]}
        ctx_titles = ex["context"]["title"]
        ctx_sentences = ex["context"]["sentences"]
        title_to_sents = {t: sents for t, sents in zip(ctx_titles, ctx_sentences)}

        # 3. Extract unique gold titles for this example (preserve order, dedup)
        #    supporting_facts = {"title": [...], "sent_id": [...]}
        sf_titles = ex["supporting_facts"]["title"]
        seen = set()
        example_gold_titles = []
        for t in sf_titles:
            if t not in seen:
                seen.add(t)
                example_gold_titles.append(t)

        # 4. Accumulate gold article stats
        for title in example_gold_titles:
            unique_gold_articles.add(title)

            if title in title_to_sents:
                n_sents = len(title_to_sents[title])
                gold_article_sentence_counts.append(n_sents)

                # Record sentence count once per unique article
                if title not in unique_article_sentences:
                    unique_article_sentences[title] = n_sents

# ── Compute derived statistics ────────────────────────────────────────────────
total = sum(type_counter.values())

bridge_n = type_counter.get("bridge", 0)
comp_n = type_counter.get("comparison", 0)
bridge_pct = (bridge_n / total * 100) if total else 0
comp_pct = (comp_n / total * 100) if total else 0

avg_sents_occurrence = (
    sum(gold_article_sentence_counts) / len(gold_article_sentence_counts)
    if gold_article_sentence_counts else 0
)
avg_sents_unique = (
    sum(unique_article_sentences.values()) / len(unique_article_sentences)
    if unique_article_sentences else 0
)

# ── Report ────────────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print("CORPUS PROFILE REPORT")
print(f"{'='*60}")
print(f"Total examples parsed : {total_examples:,}")
if malformed:
    print(f"Malformed records     : {malformed:,}")

print(f"\n--- 1. Question Type Distribution ---")
print(f"  Bridge     : {bridge_n:>6,} ({bridge_pct:>5.1f}%)")
print(f"  Comparison : {comp_n:>6,} ({comp_pct:>5.1f}%)")

print(f"\n--- 2. Difficulty Level Distribution ---")
for lvl in ["easy", "medium", "hard"]:
    cnt = level_counter.get(lvl, 0)
    pct = (cnt / total * 100) if total else 0
    print(f"  {lvl.capitalize():>8} : {cnt:>6,} ({pct:>5.1f}%)")

print(f"\n--- 3. Unique Wikipedia Articles (from supporting_facts) ---")
print(f"  Unique gold articles : {len(unique_gold_articles):,}")
print(f"  (This is the size of your subset corpus for RAG indexing.)")

print(f"\n--- 4. Sentence Count per Gold Article ---")
print(f"  Average (all occurrences)  : {avg_sents_occurrence:.1f} sentences")
print(f"  Average (unique articles)  : {avg_sents_unique:.1f} sentences")
if gold_article_sentence_counts:
    sorted_counts = sorted(gold_article_sentence_counts)
    print(f"  Minimum                    : {min(gold_article_sentence_counts)} sentences")
    print(f"  Maximum                    : {max(gold_article_sentence_counts)} sentences")
    print(f"  Median                     : {sorted_counts[len(sorted_counts)//2]} sentences")

print(f"\n{'='*60}")
print("DESIGN IMPLICATIONS FOR DOWNSTREAM PHASES")
print(f"{'='*60}")
print(f"• Subset corpus: ~{len(unique_gold_articles):,} unique articles to index.")
print(f"  At ~{avg_sents_unique:.0f} sentences/article, expect ~{len(unique_gold_articles) * avg_sents_unique / 2:,.0f}–{len(unique_gold_articles) * avg_sents_unique:,.0f} chunks")
print(f"  after paragraph-level splitting with 512-token ceiling.")

if bridge_pct > 50:
    print(f"• Bridge-heavy corpus ({bridge_pct:.1f}%). Query reconstruction (Distribution B)")
    print(f"  is the critical skill; comparison questions are secondary.")
else:
    print(f"• Balanced type distribution. Both bridge and comparison reasoning matter.")

hard_pct = (level_counter.get("hard", 0) / total * 100) if total else 0
if hard_pct < 20:
    print(f"• WARNING: Hard questions are only {hard_pct:.1f}% of data.")
    print(f"  → Consider oversampling hard examples during stratification (Step 14).")
elif hard_pct > 35:
    print(f"• Hard questions are {hard_pct:.1f}% of data. Strong signal for complex reasoning.")

Profiling hotpot_train.json ...

CORPUS PROFILE REPORT
Total examples parsed : 90,447

--- 1. Question Type Distribution ---
  Bridge     : 72,991 ( 80.7%)
  Comparison : 17,456 ( 19.3%)

--- 2. Difficulty Level Distribution ---
      Easy : 17,972 ( 19.9%)
    Medium : 56,814 ( 62.8%)
      Hard : 15,661 ( 17.3%)

--- 3. Unique Wikipedia Articles (from supporting_facts) ---
  Unique gold articles : 105,570
  (This is the size of your subset corpus for RAG indexing.)

--- 4. Sentence Count per Gold Article ---
  Average (all occurrences)  : 3.2 sentences
  Average (unique articles)  : 3.2 sentences
  Minimum                    : 1 sentences
  Maximum                    : 26 sentences
  Median                     : 3 sentences

DESIGN IMPLICATIONS FOR DOWNSTREAM PHASES
• Subset corpus: ~105,570 unique articles to index.
  At ~3 sentences/article, expect ~170,194–340,388 chunks
  after paragraph-level splitting with 512-token ceiling.
• Bridge-heavy corpus (80.7%). Query reconstruction (

In [ ]:
# Cell 6: Stratified Question Subsampling — 30K Train Target
# Train (hard-skewed), Val (natural, from train holdout), Dev (official hard-only)

import json
import hashlib
import random
from pathlib import Path
from typing import Dict, List, Set
from collections import defaultdict, Counter

# ── Configuration ─────────────────────────────────────────────────────────────
RAW_TRAIN_PATH = Path("/content/drive/MyDrive/hotpotqa-rag/data/hotpot_train.json")
RAW_DEV_PATH = Path("/content/drive/MyDrive/hotpotqa-rag/data/hotpot_dev.json")
TRAIN_LOOKUP_PATH = Path("/content/drive/MyDrive/hotpotqa-rag/data/gold_lookup.json")
DEV_LOOKUP_PATH = Path("/content/drive/MyDrive/hotpotqa-rag/data/dev_gold_lookup.json")

TRAIN_SUBSAMPLE_PATH = Path("/content/drive/MyDrive/hotpotqa-rag/data/train_subsample.json")
DEV_SUBSAMPLE_PATH = Path("/content/drive/MyDrive/hotpotqa-rag/data/dev_subsample.json")
VAL_SUBSAMPLE_PATH = Path("/content/drive/MyDrive/hotpotqa-rag/data/val_subsample.json")
MASTER_TITLE_PATH = Path("/content/drive/MyDrive/hotpotqa-rag/data/master_title_list.json")

RANDOM_SEED = 42
TRAIN_TARGET = 30_000
VAL_TARGET = 1000

TRAIN_LEVEL_DISTRIBUTION = {"easy": 0.25, "medium": 0.45, "hard": 0.30}
PRESERVE_NATURAL_TYPE_SPLIT = True

EXPECTED_GOLD_COUNT = 2
EXPECTED_DISTRACTOR_COUNT = 8
VALID_LEVELS: Set[str] = {"easy", "medium", "hard"}

random.seed(RANDOM_SEED)

# ── Format-agnostic raw file reader ───────────────────────────────────────────
def _iter_raw_examples(raw_path: Path):
    with open(raw_path, "r", encoding="utf-8") as f:
        first_char = ""
        while True:
            ch = f.read(1)
            if not ch:
                raise ValueError(f"{raw_path} is empty.")
            if not ch.isspace():
                first_char = ch
                break

    if first_char == "[":
        print(f"  Detected JSON array format for {raw_path.name}")
        with open(raw_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        for ex in data:
            yield ex
    else:
        print(f"  Detected JSONL format for {raw_path.name}")
        with open(raw_path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    yield json.loads(line)
                except json.JSONDecodeError:
                    yield {"__malformed_line__": True}

# ── Reusable validated lookup builder ─────────────────────────────────────────
def stable_qid(record: Dict) -> str:
    if "_id" in record:
        return str(record["_id"])
    return hashlib.md5(record["question"].encode("utf-8")).hexdigest()

def build_or_load_gold_lookup(raw_path: Path, lookup_path: Path, label: str) -> Dict[str, Dict]:
    if lookup_path.exists():
        print(f"Loading existing {label} lookup: {lookup_path.name}")
        with open(lookup_path, "r", encoding="utf-8") as f:
            lookup = json.load(f)
        if lookup:
            sample = next(iter(lookup.values()))
            sft = sample.get("supporting_fact_texts")
            schema_valid = isinstance(sft, dict) and all(isinstance(v, dict) for v in sft.values())
            if schema_valid:
                print(f"  {label}: {len(lookup):,} records (schema valid)")
                return lookup
            print(f"  [WARN] Existing lookup uses the old list-based supporting_fact_texts schema. Rebuilding...")

    print(f"Building {label} lookup from {raw_path.name} ...")
    gold_lookup: Dict[str, Dict] = {}
    malformed = 0

    for ex in _iter_raw_examples(raw_path):
        if ex.get("__malformed_line__"):
            malformed += 1
            continue

        try:
            qid = stable_qid(ex)
            level = ex.get("level")
            if level not in VALID_LEVELS:
                malformed += 1
                continue

            gold_titles = list(dict.fromkeys(ex["supporting_facts"]["title"]))
            if len(gold_titles) != EXPECTED_GOLD_COUNT:
                malformed += 1
                continue

            gold_set = set(gold_titles)
            all_titles = ex["context"]["title"]
            distractor_titles = [t for t in all_titles if t not in gold_set]
            if len(distractor_titles) != EXPECTED_DISTRACTOR_COUNT:
                malformed += 1
                continue

            title_to_sentences = {}
            for t, sents in zip(ex["context"]["title"], ex["context"]["sentences"]):
                title_to_sentences[t] = sents

            # Keyed by real sent_id, not by append order, so a downstream
            # lookup by sent_id resolves to the correct sentence.
            supporting_fact_texts: Dict[str, Dict[int, str]] = {}
            for sf_title, sf_sid in zip(ex["supporting_facts"]["title"], ex["supporting_facts"]["sent_id"]):
                if sf_title in title_to_sentences and sf_sid < len(title_to_sentences[sf_title]):
                    sent_text = title_to_sentences[sf_title][sf_sid]
                    supporting_fact_texts.setdefault(sf_title, {})[sf_sid] = sent_text

            if any(t not in supporting_fact_texts or not supporting_fact_texts[t] for t in gold_titles):
                malformed += 1
                continue

            gold_lookup[qid] = {
                "question": ex["question"],
                "answer": ex["answer"],
                "type": ex["type"],
                "level": level,
                "gold_titles": gold_titles,
                "distractor_titles": distractor_titles,
                "supporting_facts": ex["supporting_facts"],
                "supporting_fact_texts": supporting_fact_texts,
                "context": ex["context"],
            }
        except (KeyError, TypeError, IndexError):
            malformed += 1
            continue

    print(f"  {label}: {len(gold_lookup):,} clean records, {malformed:,} skipped")
    with open(lookup_path, "w", encoding="utf-8") as f:
        json.dump(gold_lookup, f, ensure_ascii=False)

    return gold_lookup

train_lookup = build_or_load_gold_lookup(RAW_TRAIN_PATH, TRAIN_LOOKUP_PATH, "train")
dev_lookup = build_or_load_gold_lookup(RAW_DEV_PATH, DEV_LOOKUP_PATH, "dev")

# ── Compute dev target proportionally ─────────────────────────────────────────
full_ratio = len(dev_lookup) / len(train_lookup) if train_lookup else 0
DEV_TARGET = max(1, round(TRAIN_TARGET * full_ratio))
print(f"\nTrain pool: {len(train_lookup):,} | Dev pool: {len(dev_lookup):,} "
      f"(natural ratio {full_ratio:.4f})")
print(f"Targets — train: {TRAIN_TARGET:,} | val: {VAL_TARGET:,} | dev: {DEV_TARGET:,}")

# ── Stratified sampler ────────────────────────────────────────────────────────
def stratified_sample(
    lookup: Dict[str, Dict],
    target_total: int,
    label: str,
    level_distribution: Dict[str, float],
    exclude_qids: Set[str] = None,
) -> List[str]:
    exclude_qids = exclude_qids or set()
    available_lookup = {qid: rec for qid, rec in lookup.items() if qid not in exclude_qids}

    strata: Dict[tuple, List[str]] = defaultdict(list)
    for qid, rec in available_lookup.items():
        strata[(rec["type"], rec["level"])].append(qid)

    type_counts = Counter(rec["type"] for rec in available_lookup.values())
    total = sum(type_counts.values())
    type_proportions = {t: c / total for t, c in type_counts.items()} if total else {}

    selected: List[str] = []

    print(f"\n{label} stratum targets vs. availability:")
    for (qtype, level), qids in sorted(strata.items()):
        type_frac = type_proportions.get(qtype, 0) if PRESERVE_NATURAL_TYPE_SPLIT else 0.5
        level_frac = level_distribution.get(level, 0)
        stratum_target = round(target_total * type_frac * level_frac)

        available = len(qids)
        take = min(stratum_target, available)
        random.shuffle(qids)
        selected.extend(qids[:take])

        print(f"  {qtype:>10} / {level:>6}: target {stratum_target:>5,} | "
              f"available {available:>6,} | took {take:>5,}")

    current = len(selected)
    needed = target_total - current
    if needed > 0:
        remaining_pool = [qid for qid in available_lookup if qid not in set(selected)]
        random.shuffle(remaining_pool)
        backfill = remaining_pool[:needed]
        selected.extend(backfill)
        print(f"  Backfilled {len(backfill):,} to hit target {target_total:,}.")

    print(f"\n{label} achieved distribution:")
    achieved_level = Counter(available_lookup[qid]["level"] for qid in selected)
    achieved_type = Counter(available_lookup[qid]["type"] for qid in selected)
    for lvl in sorted(VALID_LEVELS):
        cnt = achieved_level.get(lvl, 0)
        print(f"  {lvl:>6}: {cnt:>5,} ({cnt/len(selected)*100:>5.1f}%)")
    for t in sorted({"bridge", "comparison"}):
        cnt = achieved_type.get(t, 0)
        print(f"  {t:>10}: {cnt:>5,} ({cnt/len(selected)*100:>5.1f}%)")

    return selected

# ── 1. Train: hard-skewed ─────────────────────────────────────────────────────
train_qids = stratified_sample(
    train_lookup, TRAIN_TARGET, "Train", TRAIN_LEVEL_DISTRIBUTION
)
train_subsample = {qid: train_lookup[qid] for qid in train_qids}

# ── 2. Val: natural stratification from train holdout ─────────────────────────
natural_level_dist = Counter(rec["level"] for rec in train_lookup.values())
total_levels = sum(natural_level_dist.values())
natural_level_fractions = {
    lvl: natural_level_dist[lvl] / total_levels for lvl in VALID_LEVELS
}
print(f"\nVal will use natural level distribution: "
      f"{ {k: f'{v:.3f}' for k, v in natural_level_fractions.items()} }")

val_qids = stratified_sample(
    train_lookup,
    VAL_TARGET,
    "Val",
    natural_level_fractions,
    exclude_qids=set(train_qids),
)
val_subsample = {qid: train_lookup[qid] for qid in val_qids}

# ── 3. Dev: uniform random from official hard-only dev set ────────────────────
print(f"\nDev: official HotpotQA dev set is hard-only by design.")
dev_qids = random.sample(list(dev_lookup.keys()), DEV_TARGET)
dev_subsample = {qid: dev_lookup[qid] for qid in dev_qids}

print(f"\nDev achieved distribution:")
dev_achieved_level = Counter(dev_lookup[qid]["level"] for qid in dev_qids)
dev_achieved_type = Counter(dev_lookup[qid]["type"] for qid in dev_qids)
for lvl in sorted(VALID_LEVELS):
    cnt = dev_achieved_level.get(lvl, 0)
    print(f"  {lvl:>6}: {cnt:>5,} ({cnt/len(dev_qids)*100:>5.1f}%)")
for t in sorted({"bridge", "comparison"}):
    cnt = dev_achieved_type.get(t, 0)
    print(f"  {t:>10}: {cnt:>5,} ({cnt/len(dev_qids)*100:>5.1f}%)")

# ── Persist ───────────────────────────────────────────────────────────────────
with open(TRAIN_SUBSAMPLE_PATH, "w", encoding="utf-8") as f:
    json.dump(train_subsample, f, ensure_ascii=False)
with open(VAL_SUBSAMPLE_PATH, "w", encoding="utf-8") as f:
    json.dump(val_subsample, f, ensure_ascii=False)
with open(DEV_SUBSAMPLE_PATH, "w", encoding="utf-8") as f:
    json.dump(dev_subsample, f, ensure_ascii=False)

print(f"\nSaved train subsample: {len(train_subsample):,} -> {TRAIN_SUBSAMPLE_PATH.name}")
print(f"Saved val subsample:   {len(val_subsample):,} -> {VAL_SUBSAMPLE_PATH.name}")
print(f"Saved dev subsample:   {len(dev_subsample):,} -> {DEV_SUBSAMPLE_PATH.name}")

# ── Master title list from all three sets ─────────────────────────────────────
all_titles: Set[str] = set()
for rec in train_subsample.values():
    all_titles.update(rec["gold_titles"])
    all_titles.update(rec["distractor_titles"])
for rec in val_subsample.values():
    all_titles.update(rec["gold_titles"])
    all_titles.update(rec["distractor_titles"])
for rec in dev_subsample.values():
    all_titles.update(rec["gold_titles"])
    all_titles.update(rec["distractor_titles"])

master_title_list = sorted(all_titles)
with open(MASTER_TITLE_PATH, "w", encoding="utf-8") as f:
    json.dump(master_title_list, f, ensure_ascii=False)

print(f"\nUnique articles required (train + val + dev): {len(master_title_list):,}")
print(f"Saved: {MASTER_TITLE_PATH}")

Building train lookup from hotpot_train.json ...
  Detected JSONL format for hotpot_train.json
  train: 89,602 clean records, 838 skipped
Building dev lookup from hotpot_dev.json ...
  Detected JSONL format for hotpot_dev.json
  dev: 7,345 clean records, 60 skipped

Train pool: 89,602 | Dev pool: 7,345 (natural ratio 0.0820)
Targets — train: 30,000 | val: 1,000 | dev: 2,459

Train stratum targets vs. availability:
      bridge /   easy: target 6,090 | available 14,381 | took 6,090
      bridge /   hard: target 7,308 | available 12,413 | took 7,308
      bridge / medium: target 10,962 | available 45,962 | took 10,962
  comparison /   easy: target 1,410 | available  3,326 | took 1,410
  comparison /   hard: target 1,692 | available  3,135 | took 1,692
  comparison / medium: target 2,538 | available 10,385 | took 2,538

Train achieved distribution:
    easy: 7,500 ( 25.0%)
    hard: 9,000 ( 30.0%)
  medium: 13,500 ( 45.0%)
      bridge: 24,360 ( 81.2%)
  comparison: 5,640 ( 18.8%)

Val wi

In [ ]:
# Install the OpenMP library prerequisite
!apt-get install -y libomp-dev

# Install the CPU-optimized FAISS package
!pip install faiss-cpu

# Install the GPU-optimized FAISS package
!pip install faiss-gpu

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libllvm18 libomp-18-dev libomp5-18
Suggested packages:
  libomp-18-doc
The following NEW packages will be installed:
  libllvm18 libomp-18-dev libomp-dev libomp5-18
0 upgraded, 4 newly installed, 0 to remove and 52 not upgraded.
Need to get 29.1 MB of archives.
After this operation, 154 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 libllvm18 amd64 1:18.1.3-1ubuntu1 [27.5 MB]
Get:2 http://archive.ubuntu.com/ubuntu noble-updates/universe amd64 libomp5-18 amd64 1:18.1.3-1ubuntu1 [608 kB]
Get:3 http://archive.ubuntu.com/ubuntu noble-updates/universe amd64 libomp-18-dev amd64 1:18.1.3-1ubuntu1 [968 kB]
Get:4 http://archive.ubuntu.com/ubuntu noble/universe amd64 libomp-dev amd64 1:18.0-59~exp2 [5,366 B]
Fetched 29.1 MB in 3s (9,410 kB/s)
Selecting previously unselected package libllv

In [ ]:
# Cell 7 Reset: Remove previous chunking and embedding artifacts
# Deletes artifacts generated by the previous chunking/embedding pipeline.
# Source subsamples from Cell 6 are preserved.

from pathlib import Path
import shutil


# ── Paths ─────────────────────────────────────────────────────────────────────
CHUNK_METADATA_PATH = Path(
    "/content/drive/MyDrive/hotpotqa-rag/data/chunk_metadata.jsonl"
)

CHUNK_INDEX_PATH = Path(
    "/content/drive/MyDrive/hotpotqa-rag/data/chunk_index.json"
)

EMBED_CHECKPOINT_DIR = Path(
    "/content/drive/MyDrive/hotpotqa-rag/data/embedding_checkpoints"
)

FAISS_INDEX_PATH = Path(
    "/content/drive/MyDrive/hotpotqa-rag/data/faiss_index.ivfpq"
)

CHUNK_ID_MAP_PATH = Path(
    "/content/drive/MyDrive/hotpotqa-rag/data/chunk_id_map.json"
)


# ── Remove chunk artifacts ────────────────────────────────────────────────────
files_removed = 0

for path in [
    CHUNK_METADATA_PATH,
    CHUNK_INDEX_PATH,
    FAISS_INDEX_PATH,
    CHUNK_ID_MAP_PATH,
]:
    if path.exists():
        path.unlink()
        print(f"Removed: {path}")
        files_removed += 1


# ── Remove embedding checkpoints ──────────────────────────────────────────────
if EMBED_CHECKPOINT_DIR.exists():
    checkpoint_files = list(
        EMBED_CHECKPOINT_DIR.glob("batch_*.npy")
    )

    for path in checkpoint_files:
        path.unlink()

    print(
        f"Removed {len(checkpoint_files):,} embedding checkpoints: "
        f"{EMBED_CHECKPOINT_DIR}"
    )

    # Remove the directory itself if now empty.
    try:
        EMBED_CHECKPOINT_DIR.rmdir()
    except OSError:
        pass


print("\nReset complete.")
print(f"Files removed: {files_removed:,}")
print("Cell 6 subsample files were not modified.")
print("Cell 8 can now be rerun from a clean state.")


Reset complete.
Files removed: 0
Cell 6 subsample files were not modified.
Cell 8 can now be rerun from a clean state.


In [ ]:
# Cell 7: Context-Based Chunker
# Build chunks from HotpotQA's provided context sentences.
#
# Aggregates all sentences per title across train/val/dev subsamples,
# deduplicates while preserving order, then chunks using BGE's tokenizer.
# BGE-large-en-v1.5 accepts 512 input tokens including [CLS] and [SEP],
# so chunks are limited to 510 content tokens.
#
# Normal chunks use sentence boundaries with 1-sentence overlap.
# Sentences exceeding the content-token limit are split at the token level
# with 50-token overlap.
#
# Changed  design from havong to fetch Full Wikipedis Articles coz of space constraints
# even though I had worked around rate limiting
# No Wikipedia fetching. No HTML parsing.
# Chunk metadata is written incrementally to support resumable execution.

import json
import os
from pathlib import Path
from typing import Dict, List, Set
from collections import defaultdict
from concurrent.futures import ProcessPoolExecutor, wait, FIRST_COMPLETED
import itertools

from tqdm.auto import tqdm
from transformers import AutoTokenizer


# ── Configuration ─────────────────────────────────────────────────────────────
TRAIN_SUBSAMPLE_PATH = Path("/content/drive/MyDrive/hotpotqa-rag/data/train_subsample.json")
VAL_SUBSAMPLE_PATH = Path("/content/drive/MyDrive/hotpotqa-rag/data/val_subsample.json")
DEV_SUBSAMPLE_PATH = Path("/content/drive/MyDrive/hotpotqa-rag/data/dev_subsample.json")

CHUNK_METADATA_PATH = Path("/content/drive/MyDrive/hotpotqa-rag/data/chunk_metadata.jsonl")
CHUNK_INDEX_PATH = Path("/content/drive/MyDrive/hotpotqa-rag/data/chunk_index.json")

TOKENIZER_NAME = "BAAI/bge-large-en-v1.5"

BGE_MAX_SEQ_LENGTH = 512
BGE_SPECIAL_TOKEN_COUNT = 2
MAX_CHUNK_TOKENS = BGE_MAX_SEQ_LENGTH - BGE_SPECIAL_TOKEN_COUNT

OVERLAP_SENTENCES = 1
OVERSIZED_SENTENCE_OVERLAP = 50

MAX_WORKERS = max(1, (os.cpu_count() or 2) - 1)
MAX_IN_FLIGHT = MAX_WORKERS * 4


# ── Step 1: Aggregate sentences per title ─────────────────────────────────────
print("Loading subsamples and aggregating sentences per title...")

title_to_sentences: Dict[str, List[str]] = defaultdict(list)
title_qids: Dict[str, Set[str]] = defaultdict(set)

for path, label in [
    (TRAIN_SUBSAMPLE_PATH, "train"),
    (VAL_SUBSAMPLE_PATH, "val"),
    (DEV_SUBSAMPLE_PATH, "dev"),
]:
    print(f"  Loading {label}...")

    with open(path, "r", encoding="utf-8") as f:
        subsample = json.load(f)

    for qid, rec in subsample.items():
        for title, sentences in zip(
            rec["context"]["title"],
            rec["context"]["sentences"],
        ):
            title_qids[title].add(qid)
            title_to_sentences[title].extend(sentences)


# Deduplicate while preserving first-occurrence order.
for title in title_to_sentences:
    title_to_sentences[title] = list(
        dict.fromkeys(title_to_sentences[title])
    )

total_titles = len(title_to_sentences)
total_sents = sum(
    len(sentences)
    for sentences in title_to_sentences.values()
)

print(f"Unique titles: {total_titles:,}")
print(f"Total sentences (post-dedup): {total_sents:,}")


# ── Step 2: Validate BGE tokenization assumptions ─────────────────────────────
print("\nLoading BGE tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    TOKENIZER_NAME
)

print("\nBGE tokenizer boundary check...")
print(f"  Tokenizer: {TOKENIZER_NAME}")
print(f"  BGE maximum sequence length: {BGE_MAX_SEQ_LENGTH}")
print(f"  CLS token: {tokenizer.cls_token!r}")
print(f"  SEP token: {tokenizer.sep_token!r}")

assert tokenizer.cls_token is not None, (
    "BGE tokenizer does not define a CLS token."
)

assert tokenizer.sep_token is not None, (
    "BGE tokenizer does not define a SEP token."
)

test_text = "BGE tokenizer boundary validation."

content_ids = tokenizer(
    test_text,
    add_special_tokens=False,
)["input_ids"]

full_ids = tokenizer(
    test_text,
    add_special_tokens=True,
)["input_ids"]

special_token_count = len(full_ids) - len(content_ids)

assert special_token_count == BGE_SPECIAL_TOKEN_COUNT, (
    f"Expected {BGE_SPECIAL_TOKEN_COUNT} special tokens, "
    f"but tokenizer adds {special_token_count}."
)

boundary_content_ids = content_ids[:]

while len(boundary_content_ids) < MAX_CHUNK_TOKENS:
    boundary_content_ids.extend(content_ids)

boundary_content_ids = boundary_content_ids[:MAX_CHUNK_TOKENS]

boundary_text = tokenizer.decode(
    boundary_content_ids,
    skip_special_tokens=True,
)

boundary_content_check = tokenizer(
    boundary_text,
    add_special_tokens=False,
)["input_ids"]

boundary_full_check = tokenizer(
    boundary_text,
    add_special_tokens=True,
)["input_ids"]

assert len(boundary_content_check) <= MAX_CHUNK_TOKENS, (
    "Configured BGE content-token limit is not valid."
)

assert len(boundary_full_check) <= BGE_MAX_SEQ_LENGTH, (
    "Configured chunk limit exceeds BGE maximum sequence length."
)

if len(boundary_content_check) == MAX_CHUNK_TOKENS:
    assert len(boundary_full_check) == BGE_MAX_SEQ_LENGTH, (
        "BGE boundary validation failed: "
        f"{MAX_CHUNK_TOKENS} content tokens do not produce "
        f"{BGE_MAX_SEQ_LENGTH} total input tokens."
    )

print(f"  Special tokens added: {special_token_count}")
print(f"  Content-token limit: {MAX_CHUNK_TOKENS}")
print(f"  ✓ Validated: {MAX_CHUNK_TOKENS} + "
      f"{special_token_count} = {BGE_MAX_SEQ_LENGTH}")


# ── Step 3: Resume support ────────────────────────────────────────────────────
already_done: Set[str] = set()

if CHUNK_METADATA_PATH.exists():
    with open(CHUNK_METADATA_PATH, "r", encoding="utf-8") as f:
        for line in f:
            try:
                already_done.add(json.loads(line)["title"])
            except (json.JSONDecodeError, KeyError):
                continue

    print(f"Resuming: {len(already_done):,} titles already chunked.")

pending_items = [
    (title, sentences)
    for title, sentences in title_to_sentences.items()
    if title not in already_done
]

print(f"Pending: {len(pending_items):,} titles")


# ── Worker setup ──────────────────────────────────────────────────────────────
_worker_tokenizer = None


def _init_worker(tokenizer_name: str):
    global _worker_tokenizer
    _worker_tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)


# ── Chunk construction ────────────────────────────────────────────────────────
def _chunk_one(title: str, sentences: List[str]) -> List[Dict]:
    """
    Chunk a title's sentences using BGE's content-token limit.

    Normal sentences accumulate on sentence boundaries with a
    guaranteed-to-fit carryover overlap. Sentences exceeding
    MAX_CHUNK_TOKENS are split into token-level pieces; no overlap is
    carried across an oversized-sentence boundary in either direction,
    since the piece-chunks on one side and the sentence-chunks on the
    other belong to unrelated spans of text.
    """

    global _worker_tokenizer

    if _worker_tokenizer is None:
        raise RuntimeError("Worker tokenizer was not initialized.")

    chunks: List[Dict] = []
    buf_sents: List[str] = []
    sent_tok_counts: List[int] = []
    buf_sent_indices: List[int] = []
    buf_total = 0
    chunk_idx = 0
    sent_cursor = 0

    def flush() -> None:
        nonlocal chunk_idx
        if not buf_sents:
            return
        chunks.append({
            "chunk_id": f"{title}::{chunk_idx}",
            "title": title,
            "text": " ".join(buf_sents),
            "token_count": buf_total,
            "sent_start_idx": buf_sent_indices[0],
            "sent_end_idx": buf_sent_indices[-1],
        })
        chunk_idx += 1

    def carryover_fit(new_sent_len: int):
        """
        Build a carryover from the tail of the just-flushed buffer that
        is guaranteed to leave room for a sentence of new_sent_len.
        Trims sentence-by-sentence from the front of the carryover
        until it fits, down to an empty carryover if necessary.
        """
        c_sents = list(buf_sents[-OVERLAP_SENTENCES:]) if OVERLAP_SENTENCES > 0 else []
        c_counts = list(sent_tok_counts[-OVERLAP_SENTENCES:]) if OVERLAP_SENTENCES > 0 else []
        c_idx = list(buf_sent_indices[-OVERLAP_SENTENCES:]) if OVERLAP_SENTENCES > 0 else []
        while c_sents and sum(c_counts) + new_sent_len > MAX_CHUNK_TOKENS:
            c_sents.pop(0)
            c_counts.pop(0)
            c_idx.pop(0)
        return c_sents, c_counts, c_idx

    for sent in sentences:
        sent_tok = _worker_tokenizer.encode(
            sent,
            add_special_tokens=False,
        )
        sent_len = len(sent_tok)

        if sent_len > MAX_CHUNK_TOKENS:
            # Oversized sentence: flush and clear the buffer outright,
            # no carryover across this boundary.
            flush()
            buf_sents, sent_tok_counts, buf_sent_indices = [], [], []
            buf_total = 0

            step = MAX_CHUNK_TOKENS - OVERSIZED_SENTENCE_OVERLAP

            if step <= 0:
                raise ValueError(
                    "OVERSIZED_SENTENCE_OVERLAP must be smaller than "
                    "MAX_CHUNK_TOKENS."
                )

            start = 0

            while start < sent_len:
                end = min(
                    start + MAX_CHUNK_TOKENS,
                    sent_len,
                )

                piece_ids = sent_tok[start:end]

                piece_text = _worker_tokenizer.decode(
                    piece_ids,
                    skip_special_tokens=True,
                    clean_up_tokenization_spaces=True,
                )

                piece_check_ids = _worker_tokenizer.encode(
                    piece_text,
                    add_special_tokens=False,
                )

                piece_token_count = len(piece_check_ids)

                if piece_token_count > MAX_CHUNK_TOKENS:
                    raise ValueError(
                        f"Chunk generated from oversized sentence exceeds "
                        f"BGE content limit: {piece_token_count} > "
                        f"{MAX_CHUNK_TOKENS}."
                    )

                chunks.append({
                    "chunk_id": f"{title}::{chunk_idx}",
                    "title": title,
                    "text": piece_text,
                    "token_count": piece_token_count,
                    "sent_start_idx": sent_cursor,
                    "sent_end_idx": sent_cursor,
                })

                chunk_idx += 1

                if end >= sent_len:
                    break

                start += step

            sent_cursor += 1
            continue

        # Normal sentence: flush and reseed with a fit-checked carryover
        # if it wouldn't fit in the current buffer.
        if buf_sents and buf_total + sent_len > MAX_CHUNK_TOKENS:
            flush()
            buf_sents, sent_tok_counts, buf_sent_indices = carryover_fit(sent_len)
            buf_total = sum(sent_tok_counts)

        buf_sents.append(sent)
        sent_tok_counts.append(sent_len)
        buf_sent_indices.append(sent_cursor)
        buf_total += sent_len

        sent_cursor += 1

    flush()
    return chunks


# ── Parallel execution with bounded in-flight work ────────────────────────────
chunked_this_run = 0
chunks_this_run = 0

with open(CHUNK_METADATA_PATH, "a", encoding="utf-8") as out_f:
    with ProcessPoolExecutor(
        max_workers=MAX_WORKERS,
        initializer=_init_worker,
        initargs=(TOKENIZER_NAME,),
    ) as executor:

        items_iter = iter(pending_items)

        in_flight = {
            executor.submit(_chunk_one, title, sentences): title
            for title, sentences in itertools.islice(
                items_iter,
                MAX_IN_FLIGHT,
            )
        }

        with tqdm(
            total=len(pending_items),
            desc="Chunking titles",
        ) as pbar:

            while in_flight:
                done, _ = wait(
                    in_flight,
                    return_when=FIRST_COMPLETED,
                )

                for future in done:
                    chunks = future.result()

                    for chunk in chunks:
                        if chunk["token_count"] > MAX_CHUNK_TOKENS:
                            raise ValueError(
                                f"Chunk {chunk['chunk_id']} exceeds "
                                f"BGE content limit: "
                                f"{chunk['token_count']} > "
                                f"{MAX_CHUNK_TOKENS}."
                            )

                        out_f.write(
                            json.dumps(
                                chunk,
                                ensure_ascii=False,
                            ) + "\n"
                        )

                    out_f.flush()

                    chunks_this_run += len(chunks)
                    chunked_this_run += 1

                    del in_flight[future]
                    pbar.update(1)

                    nxt = next(items_iter, None)

                    if nxt is not None:
                        title, sentences = nxt
                        in_flight[
                            executor.submit(
                                _chunk_one,
                                title,
                                sentences,
                            )
                        ] = title


# ── Rebuild title → chunks index ───────────────────────────────────────────────
print("\nRebuilding title → chunks index...")

title_to_chunks: Dict[str, List[Dict]] = defaultdict(list)

with open(CHUNK_METADATA_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()

        if not line:
            continue

        record = json.loads(line)
        title_to_chunks[record["title"]].append(record)

with open(CHUNK_INDEX_PATH, "w", encoding="utf-8") as f:
    json.dump(
        dict(title_to_chunks),
        f,
        ensure_ascii=False,
    )


# ── Final statistics ──────────────────────────────────────────────────────────
total_chunks = sum(
    len(chunks)
    for chunks in title_to_chunks.values()
)

print("\nChunking complete.")
print(f"  Titles chunked (this run): {chunked_this_run:,}")
print(f"  Chunks created (this run): {chunks_this_run:,}")
print(f"  Total unique titles:        {len(title_to_chunks):,}")
print(f"  Total chunks:               {total_chunks:,}")
print(
    f"  Avg chunks/title:           "
    f"{total_chunks / len(title_to_chunks):.2f}"
    if title_to_chunks
    else "  Avg chunks/title:           N/A"
)
print(f"\nMetadata: {CHUNK_METADATA_PATH}")
print(f"Index:    {CHUNK_INDEX_PATH}")

Loading subsamples and aggregating sentences per title...
  Loading train...
  Loading val...
  Loading dev...
Unique titles: 240,376
Total sentences (post-dedup): 927,934

Loading BGE tokenizer...


config.json:   0%|          | 0.00/779 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]


BGE tokenizer boundary check...
  Tokenizer: BAAI/bge-large-en-v1.5
  BGE maximum sequence length: 512
  CLS token: '[CLS]'
  SEP token: '[SEP]'
  Special tokens added: 2
  Content-token limit: 510
  ✓ Validated: 510 + 2 = 512
Pending: 240,376 titles


Chunking titles:   0%|          | 0/240376 [00:00<?, ?it/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (588 > 512). Running this sequence through the model will result in indexing errors



Rebuilding title → chunks index...

Chunking complete.
  Titles chunked (this run): 240,376
  Chunks created (this run): 240,571
  Total unique titles:        240,376
  Total chunks:               240,571
  Avg chunks/title:           1.00

Metadata: /content/drive/MyDrive/hotpotqa-rag/data/chunk_metadata.jsonl
Index:    /content/drive/MyDrive/hotpotqa-rag/data/chunk_index.json


In [ ]:
# One-off cleanup: clear stale embedding checkpoints and FAISS artifacts
# left over from before the Cell 6/7 fixes ie fromchange of design No Wikipedia artile fetching
# and html parsing.
# Safe to run even if some of
# these paths don't exist yet.

import shutil
from pathlib import Path

EMBED_CHECKPOINT_DIR = Path("/content/drive/MyDrive/hotpotqa-rag/data/embedding_checkpoints")
FAISS_INDEX_PATH = Path("/content/drive/MyDrive/hotpotqa-rag/data/faiss_index.ivfpq")
CHUNK_ID_MAP_PATH = Path("/content/drive/MyDrive/hotpotqa-rag/data/chunk_id_map.json")
FAISS_METADATA_PATH = Path("/content/drive/MyDrive/hotpotqa-rag/data/faiss_index_metadata.json")

if EMBED_CHECKPOINT_DIR.exists():
    n = len(list(EMBED_CHECKPOINT_DIR.glob("*.npy")))
    shutil.rmtree(EMBED_CHECKPOINT_DIR)
    print(f"Removed {EMBED_CHECKPOINT_DIR} ({n} checkpoint files).")
else:
    print(f"{EMBED_CHECKPOINT_DIR} does not exist, nothing to remove.")

for path in [FAISS_INDEX_PATH, CHUNK_ID_MAP_PATH, FAISS_METADATA_PATH]:
    if path.exists():
        path.unlink()
        print(f"Removed {path}")
    else:
        print(f"{path} does not exist, nothing to remove.")

/content/drive/MyDrive/hotpotqa-rag/data/embedding_checkpoints does not exist, nothing to remove.
/content/drive/MyDrive/hotpotqa-rag/data/faiss_index.ivfpq does not exist, nothing to remove.
/content/drive/MyDrive/hotpotqa-rag/data/chunk_id_map.json does not exist, nothing to remove.
/content/drive/MyDrive/hotpotqa-rag/data/faiss_index_metadata.json does not exist, nothing to remove.


In [ ]:
# Cell 8: BGE Embedding & FAISS IVFPQ Index Builder
# Embed all chunks with BGE-large-en-v1.5 and build IVFPQ index.
#
# Part 1: Embed chunks with per-batch checkpointing and resumable execution.
# Part 2: Train IVFPQ from a deterministic sample of checkpointed embeddings,
#         stream embeddings into the trained index, validate, and persist artifacts.
#
# Embedding is executed as a single GPU workload. Throughput is controlled
# primarily by BATCH_SIZE rather than process count.

import json
import hashlib
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import faiss
import torch
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer


# ── Configuration ─────────────────────────────────────────────────────────────

CHUNK_METADATA_PATH = Path(
    "/content/drive/MyDrive/hotpotqa-rag/data/chunk_metadata.jsonl"
)
FAISS_INDEX_PATH = Path(
    "/content/drive/MyDrive/hotpotqa-rag/data/faiss_index.ivfpq"
)
CHUNK_ID_MAP_PATH = Path(
    "/content/drive/MyDrive/hotpotqa-rag/data/chunk_id_map.json"
)
FAISS_METADATA_PATH = Path(
    "/content/drive/MyDrive/hotpotqa-rag/data/faiss_index_metadata.json"
)
EMBED_CHECKPOINT_DIR = Path(
    "/content/drive/MyDrive/hotpotqa-rag/data/embedding_checkpoints"
)
EMBED_FINGERPRINT_PATH = EMBED_CHECKPOINT_DIR / "_fingerprint.json"

EMBED_MODEL = "BAAI/bge-large-en-v1.5"

BATCH_SIZE = 256
EMBED_DIM = 1024
RANDOM_SEED = 42

# IVFPQ configuration.
NLIST_MULTIPLIER = 4
MIN_TRAIN_VECTORS_PER_CLUSTER = 39
PQ_M = 64
PQ_NBITS = 8

ADD_BATCH_SIZE = 10_000

TRAIN_VECTORS_PER_CLUSTER = MIN_TRAIN_VECTORS_PER_CLUSTER


# ── Utility functions ─────────────────────────────────────────────────────────

def checkpoint_path(start: int, count: int) -> Path:
    return EMBED_CHECKPOINT_DIR / f"batch_{start:08d}_{count:04d}.npy"


def validate_embedding_array(
    emb: np.ndarray,
    expected_count: int,
    source: str,
) -> None:
    if emb.ndim != 2:
        raise ValueError(
            f"{source}: expected a 2D embedding matrix, got shape {emb.shape}."
        )

    if emb.shape != (expected_count, EMBED_DIM):
        raise ValueError(
            f"{source}: expected shape "
            f"({expected_count}, {EMBED_DIM}), got {emb.shape}."
        )

    if emb.dtype != np.float32:
        raise ValueError(
            f"{source}: expected dtype float32, got {emb.dtype}."
        )

    if not np.isfinite(emb).all():
        raise ValueError(
            f"{source}: embedding matrix contains NaN or infinite values."
        )


def validate_checkpoint(
    path: Path,
    expected_start: int,
    expected_count: int,
) -> bool:
    if not path.exists():
        return False

    try:
        emb = np.load(path, allow_pickle=False)
        validate_embedding_array(
            emb,
            expected_count,
            str(path),
        )
    except Exception as exc:
        print(f"[WARN] Invalid checkpoint {path.name}: {exc}")
        return False

    return True


def compute_chunk_fingerprint(chunk_ids: List[str]) -> str:
    hasher = hashlib.sha256()

    for chunk_id in chunk_ids:
        hasher.update(chunk_id.encode("utf-8"))
        hasher.update(b"\n")

    return hasher.hexdigest()


def load_chunk_metadata() -> Tuple[List[Dict], List[str]]:
    chunks: List[Dict] = []

    with open(CHUNK_METADATA_PATH, "r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, start=1):
            line = line.strip()

            if not line:
                continue

            try:
                record = json.loads(line)
            except json.JSONDecodeError as exc:
                raise ValueError(
                    f"Invalid JSON in {CHUNK_METADATA_PATH} at line {line_number}."
                ) from exc

            if "chunk_id" not in record or "text" not in record:
                raise ValueError(
                    f"Missing required fields in {CHUNK_METADATA_PATH} "
                    f"at line {line_number}."
                )

            chunks.append(record)

    chunk_ids = [record["chunk_id"] for record in chunks]

    if len(chunk_ids) != len(set(chunk_ids)):
        raise ValueError("Duplicate chunk_id values detected in chunk metadata.")

    return chunks, chunk_ids


def validate_checkpoint_sequence(
    total_chunks: int,
) -> List[Tuple[Path, int, int]]:
    checkpoints: List[Tuple[Path, int, int]] = []
    offset = 0

    while offset < total_chunks:
        count = min(BATCH_SIZE, total_chunks - offset)
        path = checkpoint_path(offset, count)

        if not validate_checkpoint(path, offset, count):
            break

        checkpoints.append((path, offset, count))
        offset += count

    return checkpoints


def load_embed_fingerprint() -> Dict:
    if not EMBED_FINGERPRINT_PATH.exists():
        return None
    try:
        with open(EMBED_FINGERPRINT_PATH, "r", encoding="utf-8") as f:
            return json.load(f)
    except (json.JSONDecodeError, OSError):
        return None


def write_embed_fingerprint(fingerprint: str, total: int) -> None:
    with open(EMBED_FINGERPRINT_PATH, "w", encoding="utf-8") as f:
        json.dump(
            {"chunk_fingerprint": fingerprint, "total_chunks": total},
            f,
            ensure_ascii=False,
        )


# ── Part 1: Embed chunks with checkpointing ────────────────────────────────────

print("=" * 60)
print("PART 1: Embedding chunks")
print("=" * 60)

print("Loading chunk metadata ...")
chunks, chunk_ids = load_chunk_metadata()

total_chunks = len(chunks)

if total_chunks == 0:
    raise ValueError("No chunks found in chunk metadata.")

print(f"Loaded {total_chunks:,} chunks.")

chunk_fingerprint = compute_chunk_fingerprint(chunk_ids)
print(f"Chunk dataset fingerprint: {chunk_fingerprint}")


print(f"Loading embedding model: {EMBED_MODEL}")
device = "cuda" if torch.cuda.is_available() else "cpu"

if device == "cuda":
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
else:
    print("Using CPU.")

model = SentenceTransformer(
    EMBED_MODEL,
    device=device,
)


# ── BGE embedding validation ──────────────────────────────────────────────────

model_embedding_dim = model.get_sentence_embedding_dimension()

if model_embedding_dim != EMBED_DIM:
    raise ValueError(
        f"BGE embedding dimension mismatch: configuration expects "
        f"{EMBED_DIM}, but the loaded model produces {model_embedding_dim}."
    )

if model.max_seq_length != 512:
    raise ValueError(
        f"Unexpected BGE maximum sequence length: "
        f"{model.max_seq_length}. Expected 512."
    )

print(f"BGE max_seq_length: {model.max_seq_length}")
print(f"BGE embedding dimension: {model_embedding_dim}")


# ── Checkpoint directory ──────────────────────────────────────────────────────

EMBED_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)


# ── Fingerprint-based staleness check ──────────────────────────────────────────
# Unlike Part 2's FAISS artifacts, embedding checkpoints previously had no
# record of which chunk dataset produced them, so a resume after re-chunking
# could silently reuse embeddings for text that no longer matches the current
# chunk boundaries. This makes Part 1 self-protecting the same way Part 2
# already is: store the fingerprint alongside the checkpoints, and if it
# doesn't match the current chunk dataset, wipe stale checkpoints and
# re-embed from scratch automatically.

stored_fingerprint = load_embed_fingerprint()

if stored_fingerprint is not None and (
    stored_fingerprint.get("chunk_fingerprint") != chunk_fingerprint
    or stored_fingerprint.get("total_chunks") != total_chunks
):
    print("\n[WARN] Embedding checkpoints were built from a different chunk dataset.")
    print(f"  Stored fingerprint:  {stored_fingerprint.get('chunk_fingerprint')}")
    print(f"  Current fingerprint: {chunk_fingerprint}")
    print("  Clearing stale checkpoints; embedding will restart from scratch.\n")

    for p in EMBED_CHECKPOINT_DIR.glob("batch_*.npy"):
        p.unlink()

    stored_fingerprint = None

if stored_fingerprint is None:
    write_embed_fingerprint(chunk_fingerprint, total_chunks)
else:
    print("Embedding checkpoint fingerprint matches current chunk dataset.")


# ── Resume from the first missing or invalid checkpoint ────────────────────────

valid_checkpoints = validate_checkpoint_sequence(total_chunks)

covered_upto = sum(count for _, _, count in valid_checkpoints)

if covered_upto == total_chunks:
    print(
        f"All embedding checkpoints are valid: "
        f"{len(valid_checkpoints):,} batches."
    )
else:
    print(
        f"Resuming from chunk {covered_upto:,} / {total_chunks:,} "
        f"({len(valid_checkpoints):,} valid batches already checkpointed)."
    )

    for path in EMBED_CHECKPOINT_DIR.glob("batch_*.npy"):
        try:
            parts = path.stem.split("_")
            start = int(parts[1])

            if start >= covered_upto:
                path.unlink()
        except (ValueError, IndexError):
            print(f"[WARN] Ignoring unexpected checkpoint filename: {path.name}")


# ── Embed remaining chunks ────────────────────────────────────────────────────

for i in tqdm(
    range(covered_upto, total_chunks, BATCH_SIZE),
    desc="Embedding",
):
    batch = chunks[i : i + BATCH_SIZE]
    texts = [record["text"] for record in batch]

    embeddings = model.encode(
        texts,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False,
    )

    embeddings = np.asarray(embeddings, dtype=np.float32)

    validate_embedding_array(
        embeddings,
        len(batch),
        f"Embedding batch starting at {i}",
    )

    cp_path = checkpoint_path(i, len(batch))

    tmp_path = cp_path.with_suffix(".tmp.npy")

    np.save(tmp_path, embeddings)
    tmp_path.replace(cp_path)


print("Part 1 complete: all embedding batches checkpointed.")


# ── Part 2: Build FAISS IVFPQ index ────────────────────────────────────────────

print("\n" + "=" * 60)
print("PART 2: Building FAISS index")
print("=" * 60)


# ── Existing index validation ──────────────────────────────────────────────────

index_needs_build = True

if (
    FAISS_INDEX_PATH.exists()
    and CHUNK_ID_MAP_PATH.exists()
    and FAISS_METADATA_PATH.exists()
):
    print("Existing FAISS artifacts detected.")

    try:
        with open(FAISS_METADATA_PATH, "r", encoding="utf-8") as f:
            existing_metadata = json.load(f)

        existing_fingerprint = existing_metadata.get("chunk_fingerprint")

        existing_index = faiss.read_index(str(FAISS_INDEX_PATH))

        with open(CHUNK_ID_MAP_PATH, "r", encoding="utf-8") as f:
            existing_chunk_ids = json.load(f)

        index_matches = (
            existing_index.ntotal == total_chunks
            and existing_index.d == EMBED_DIM
            and len(existing_chunk_ids) == total_chunks
            and existing_fingerprint == chunk_fingerprint
        )

        if index_matches:
            print(
                f"Existing index matches the current chunk dataset "
                f"({existing_index.ntotal:,} vectors)."
            )
            print("Skipping FAISS index construction.")
            index_needs_build = False
        else:
            print(
                "[WARN] Existing FAISS artifacts do not match the current "
                "chunk dataset or configuration. Rebuilding..."
            )

    except Exception as exc:
        print(f"[WARN] Existing FAISS artifacts could not be validated: {exc}")
        print("Rebuilding index...")


if index_needs_build:

    nlist_heuristic = int(NLIST_MULTIPLIER * np.sqrt(total_chunks))

    nlist_max = max(
        1,
        total_chunks // MIN_TRAIN_VECTORS_PER_CLUSTER,
    )

    nlist = min(nlist_heuristic, nlist_max)
    nlist = max(1, nlist)

    if EMBED_DIM % PQ_M != 0:
        raise ValueError(
            f"Embedding dimension {EMBED_DIM} must be divisible by "
            f"PQ_M={PQ_M}."
        )

    subvectors_per_dimension = EMBED_DIM // PQ_M

    print(f"\nIndex parameters (computed from N={total_chunks:,}):")
    print(f"  nlist (clusters):        {nlist:,}")
    print(f"  m (sub-quantizers):      {PQ_M}")
    print(f"  dimensions/subvector:    {subvectors_per_dimension}")
    print(f"  nbits:                   {PQ_NBITS}")
    print("  metric:                  INNER_PRODUCT")
    print("  embeddings:              normalized")
    print("  similarity:              cosine")


    quantizer = faiss.IndexFlatIP(EMBED_DIM)

    index = faiss.IndexIVFPQ(
        quantizer,
        EMBED_DIM,
        nlist,
        PQ_M,
        PQ_NBITS,
        faiss.METRIC_INNER_PRODUCT,
    )


    train_size = min(
        nlist * TRAIN_VECTORS_PER_CLUSTER,
        total_chunks,
    )

    rng = np.random.RandomState(RANDOM_SEED)

    train_indices = rng.choice(
        total_chunks,
        size=train_size,
        replace=False,
    )

    train_indices.sort()

    print(
        f"\nTraining IVFPQ on {train_size:,} vectors "
        f"({train_size / total_chunks * 100:.1f}% of corpus)..."
    )


    train_sample = np.empty(
        (train_size, EMBED_DIM),
        dtype=np.float32,
    )

    checkpoint_cache: Dict[int, np.ndarray] = {}

    for train_position, global_index in enumerate(
        tqdm(train_indices, desc="Preparing training sample")
    ):
        batch_start = (global_index // BATCH_SIZE) * BATCH_SIZE
        batch_count = min(
            BATCH_SIZE,
            total_chunks - batch_start,
        )

        if batch_start not in checkpoint_cache:
            cp_path = checkpoint_path(
                batch_start,
                batch_count,
            )

            if not validate_checkpoint(
                cp_path,
                batch_start,
                batch_count,
            ):
                raise RuntimeError(
                    f"Required training checkpoint is missing or invalid: "
                    f"{cp_path}"
                )

            checkpoint_cache[batch_start] = np.load(
                cp_path,
                allow_pickle=False,
            )

        local_index = global_index - batch_start

        train_sample[train_position] = (
            checkpoint_cache[batch_start][local_index]
        )

    checkpoint_cache.clear()


    index.train(train_sample)

    del train_sample

    if not index.is_trained:
        raise RuntimeError("FAISS IVFPQ index failed to train.")

    print("IVFPQ training complete.")


    print("\nAdding embeddings to FAISS index...")

    cp_files = []
    offset = 0

    while offset < total_chunks:
        count = min(
            BATCH_SIZE,
            total_chunks - offset,
        )

        cp_path = checkpoint_path(offset, count)

        if not validate_checkpoint(
            cp_path,
            offset,
            count,
        ):
            raise RuntimeError(
                f"Missing or invalid checkpoint during index construction: "
                f"{cp_path}"
            )

        cp_files.append((cp_path, offset, count))
        offset += count

    if offset != total_chunks:
        raise RuntimeError(
            f"Checkpoint coverage mismatch: {offset} vs {total_chunks}."
        )


    for cp_path, offset, count in tqdm(
        cp_files,
        desc="Adding to index",
    ):
        embeddings = np.load(
            cp_path,
            allow_pickle=False,
        )

        validate_embedding_array(
            embeddings,
            count,
            str(cp_path),
        )

        for start in range(0, count, ADD_BATCH_SIZE):
            end = min(start + ADD_BATCH_SIZE, count)
            index.add(embeddings[start:end])

        del embeddings


    if index.ntotal != total_chunks:
        raise RuntimeError(
            f"FAISS index size mismatch: "
            f"{index.ntotal:,} vectors indexed, "
            f"{total_chunks:,} expected."
        )

    if index.d != EMBED_DIM:
        raise RuntimeError(
            f"FAISS dimension mismatch: "
            f"{index.d} vs expected {EMBED_DIM}."
        )

    if not index.is_trained:
        raise RuntimeError("FAISS index is not marked as trained.")


    FAISS_INDEX_PATH.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    faiss.write_index(
        index,
        str(FAISS_INDEX_PATH),
    )


    chunk_id_map = chunk_ids

    with open(
        CHUNK_ID_MAP_PATH,
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            chunk_id_map,
            f,
            ensure_ascii=False,
        )


    index_metadata = {
        "embedding_model": EMBED_MODEL,
        "embedding_dimension": EMBED_DIM,
        "normalize_embeddings": True,
        "metric": "INNER_PRODUCT",
        "similarity": "COSINE",
        "index_type": "IVFPQ",
        "nlist": nlist,
        "m": PQ_M,
        "nbits": PQ_NBITS,
        "dimensions_per_subvector": subvectors_per_dimension,
        "training_vectors": train_size,
        "training_vectors_per_cluster": TRAIN_VECTORS_PER_CLUSTER,
        "random_seed": RANDOM_SEED,
        "num_vectors": total_chunks,
        "chunk_fingerprint": chunk_fingerprint,
        "batch_size": BATCH_SIZE,
    }

    with open(
        FAISS_METADATA_PATH,
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            index_metadata,
            f,
            indent=2,
            ensure_ascii=False,
        )


    persisted_index = faiss.read_index(
        str(FAISS_INDEX_PATH)
    )

    if persisted_index.ntotal != total_chunks:
        raise RuntimeError(
            "Persisted FAISS index failed post-write validation: "
            f"{persisted_index.ntotal:,} vs {total_chunks:,} vectors."
        )

    if persisted_index.d != EMBED_DIM:
        raise RuntimeError(
            "Persisted FAISS index has unexpected dimensionality: "
            f"{persisted_index.d} vs {EMBED_DIM}."
        )


    print(f"\nIndex saved: {FAISS_INDEX_PATH}")
    print(f"ID map saved: {CHUNK_ID_MAP_PATH}")
    print(f"Metadata saved: {FAISS_METADATA_PATH}")
    print(f"Total indexed vectors: {persisted_index.ntotal:,}")

else:
    print("\nFAISS index construction skipped.")

PART 1: Embedding chunks
Loading chunk metadata ...
Loaded 240,571 chunks.
Chunk dataset fingerprint: 6b8ac62e04f5488640999bc7e557e55bf7f3471796184432d0f725a4841e2199
Loading embedding model: BAAI/bge-large-en-v1.5
Using GPU: Tesla T4


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

Part 1 complete: all embedding batches checkpointed.

PART 2: Building FAISS index

Index parameters (computed from N=240,571):
  nlist (clusters):        1,961
  m (sub-quantizers):      64
  dimensions/subvector:    16
  nbits:                   8
  metric:                  INNER_PRODUCT
  embeddings:              normalized
  similarity:              cosine

Training IVFPQ on 76,479 vectors (31.8% of corpus)...


Preparing training sample:   0%|          | 0/76479 [00:00<?, ?it/s]

IVFPQ training complete.

Adding embeddings to FAISS index...


Adding to index:   0%|          | 0/940 [00:00<?, ?it/s]


Index saved: /content/drive/MyDrive/hotpotqa-rag/data/faiss_index.ivfpq
ID map saved: /content/drive/MyDrive/hotpotqa-rag/data/chunk_id_map.json
Metadata saved: /content/drive/MyDrive/hotpotqa-rag/data/faiss_index_metadata.json
Total indexed vectors: 240,571


In [ ]:
!pip install rapidfuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 72.1 MB/s eta 0:00:00


In [ ]:
# Cell 9: Pilot Validation —Context-Based Chunk Routing
# Verify that supporting-fact sentences are findable in the
# chunks produced by Cell 8. Samples from train_subsample.json (not the raw
# 90K file), uses pre-extracted supporting_fact_texts (keyed by real
# sent_id as of the Cell 6 fix), and validates exact + fuzzy matching
# against the chunk_index.
#
# This is a go/no-go gate before Phase 3. With context-based chunking,
# exact-match should dominate; fuzzy is a safety net for whitespace drift.

import json
import random
from pathlib import Path
from typing import Dict, List, Set, Tuple
from collections import Counter, defaultdict

try:
    from rapidfuzz import fuzz
    HAS_RAPIDFUZZ = True
except ImportError:
    from difflib import SequenceMatcher
    HAS_RAPIDFUZZ = False
    print("[WARN] rapidfuzz not installed. Run: pip install rapidfuzz")
    print("       Falling back to difflib (slower, slightly different scores).")

# ── Configuration ─────────────────────────────────────────────────────────────
TRAIN_SUBSAMPLE_PATH = Path("/content/drive/MyDrive/hotpotqa-rag/data/train_subsample.json")
CHUNK_INDEX_PATH = Path("/content/drive/MyDrive/hotpotqa-rag/data/chunk_index.json")
PILOT_REPORT_PATH = Path("/content/drive/MyDrive/hotpotqa-rag/data/pilot_report.json")

PILOT_SAMPLE_SIZE = 500
FUZZY_THRESHOLD = 85.0

random.seed(42)

# ── Load subsample and chunk index ────────────────────────────────────────────
print("Loading train subsample ...")
with open(TRAIN_SUBSAMPLE_PATH, "r", encoding="utf-8") as f:
    train_subsample: Dict[str, Dict] = json.load(f)

print("Loading chunk index ...")
with open(CHUNK_INDEX_PATH, "r", encoding="utf-8") as f:
    chunk_index: Dict[str, List[Dict]] = json.load(f)

# ── Stratified sample from SUBSAMPLE ──────────────────────────────────────────
strata = defaultdict(list)
for qid, rec in train_subsample.items():
    strata[(rec["type"], rec["level"])].append(qid)

target_per_stratum = PILOT_SAMPLE_SIZE // max(1, len(strata))
sampled_qids: List[str] = []
for key, qids in strata.items():
    n = min(target_per_stratum, len(qids))
    sampled_qids.extend(random.sample(qids, n))

if len(sampled_qids) < PILOT_SAMPLE_SIZE:
    remaining = [qid for qid in train_subsample if qid not in set(sampled_qids)]
    random.shuffle(remaining)
    sampled_qids.extend(remaining[:PILOT_SAMPLE_SIZE - len(sampled_qids)])

random.shuffle(sampled_qids)
print(f"Pilot sample: {len(sampled_qids)} questions from subsample "
      f"(out of {len(train_subsample):,} available).")

# ── Target-sentence lookup ─────────────────────────────────────────────────────
def get_target_sentences(
    rec: Dict, title: str, sf_titles: List[str], sf_sent_ids: List[int]
) -> List[str]:
    """
    Pull this title's real supporting-fact sentence text(s) by direct
    sent_id lookup. supporting_fact_texts is stored as
    {title: {sent_id: text}}; JSON round-tripping always turns dict keys
    into strings, so sent_id is cast with str() here regardless of how
    it started out in memory.
    """
    sft = rec.get("supporting_fact_texts", {})
    title_map = sft.get(title, {})

    targets = []
    for t, sid in zip(sf_titles, sf_sent_ids):
        if t != title:
            continue
        text = title_map.get(str(sid))
        if text is not None:
            targets.append(text)
    return targets

# ── Resolution engine ─────────────────────────────────────────────────────────
def resolve_gold_chunk(title: str, target_sentences: List[str]) -> Tuple[str, float, str]:
    """
    Find target supporting-fact sentence(s) in the chunks for this title.
    Returns: (match_type, best_score, chunk_id)
    match_type ∈ {"exact", "fuzzy", "not_found", "no_chunks", "no_target_text"}
    """
    if not target_sentences:
        # Should be rare to nonexistent post-Cell-6-fix, since malformed
        # records with an empty supporting_fact_texts entry for a gold
        # title are now filtered out before reaching the subsample. A
        # nonzero count here is a signal to check for a stale
        # train_subsample.json, not a chunking problem.
        return "no_target_text", 0.0, ""

    if title not in chunk_index or not chunk_index[title]:
        return "no_chunks", 0.0, ""

    chunks = chunk_index[title]
    best_chunk_id = ""
    best_score = 0.0

    for target in target_sentences:
        target_clean = target.strip()
        if not target_clean:
            continue

        for chunk in chunks:
            if target_clean in chunk["text"]:
                return "exact", 100.0, chunk["chunk_id"]

        for chunk in chunks:
            if HAS_RAPIDFUZZ:
                score = fuzz.partial_ratio(target_clean, chunk["text"])
            else:
                s = SequenceMatcher(None, target_clean, chunk["text"])
                match = s.find_longest_match(0, len(target_clean), 0, len(chunk["text"]))
                score = (match.size / len(target_clean)) * 100.0 if len(target_clean) > 0 else 0.0

            if score > best_score:
                best_score = score
                best_chunk_id = chunk["chunk_id"]

    if best_score >= FUZZY_THRESHOLD:
        return "fuzzy", best_score, best_chunk_id

    return "not_found", best_score, best_chunk_id

# ── Run pilot ─────────────────────────────────────────────────────────────────
per_article_stats = Counter()
per_question_stats = Counter()
fuzzy_scores: List[float] = []
chunk_counts: List[int] = []

for qid in sampled_qids:
    rec = train_subsample[qid]
    sf_titles = rec["supporting_facts"]["title"]
    sf_sent_ids = rec["supporting_facts"]["sent_id"]

    seen = set()
    unique_gold_titles = [t for t in sf_titles if not (t in seen or seen.add(t))]

    question_resolved = True
    for title in unique_gold_titles:
        targets = get_target_sentences(rec, title, sf_titles, sf_sent_ids)
        match_type, score, chunk_id = resolve_gold_chunk(title, targets)

        per_article_stats[match_type] += 1
        if match_type == "fuzzy":
            fuzzy_scores.append(score)

        if title in chunk_index:
            chunk_counts.append(len(chunk_index[title]))

        if match_type in ("no_chunks", "not_found", "no_target_text"):
            question_resolved = False

    per_question_stats["survived" if question_resolved else "blacklisted"] += 1

# ── Metrics ───────────────────────────────────────────────────────────────────
total_articles = sum(per_article_stats.values())
exact_rate = per_article_stats["exact"] / total_articles * 100 if total_articles else 0
fuzzy_rate = per_article_stats["fuzzy"] / total_articles * 100 if total_articles else 0
combined_rate = exact_rate + fuzzy_rate
survival_rate = per_question_stats["survived"] / len(sampled_qids) * 100 if sampled_qids else 0

# ── Report ────────────────────────────────────────────────────────────────────
report = {
    "sample_size": len(sampled_qids),
    "per_article": dict(per_article_stats),
    "exact_match_rate_pct": round(exact_rate, 2),
    "fuzzy_match_rate_pct": round(fuzzy_rate, 2),
    "combined_resolution_rate_pct": round(combined_rate, 2),
    "per_question": dict(per_question_stats),
    "question_survival_rate_pct": round(survival_rate, 2),
    "fuzzy_score_distribution": {
        "count": len(fuzzy_scores),
        "min": round(min(fuzzy_scores), 2) if fuzzy_scores else None,
        "max": round(max(fuzzy_scores), 2) if fuzzy_scores else None,
        "mean": round(sum(fuzzy_scores) / len(fuzzy_scores), 2) if fuzzy_scores else None,
        "median": round(sorted(fuzzy_scores)[len(fuzzy_scores) // 2], 2) if fuzzy_scores else None,
    },
    "avg_chunks_per_title": round(sum(chunk_counts) / len(chunk_counts), 2) if chunk_counts else 0,
    "threshold_used": FUZZY_THRESHOLD,
    "used_rapidfuzz": HAS_RAPIDFUZZ,
}

with open(PILOT_REPORT_PATH, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

print(f"\n{'='*60}")
print("PILOT REPORT")
print(f"{'='*60}")
print(f"Sample size: {report['sample_size']}")
print(f"\nPer-article resolution:")
print(f"  Exact match:      {per_article_stats['exact']:,} ({exact_rate:.1f}%)")
print(f"  Fuzzy match:      {per_article_stats['fuzzy']:,} ({fuzzy_rate:.1f}%)")
print(f"  Combined:         {combined_rate:.1f}%")
print(f"  Not found:        {per_article_stats['not_found']:,}")
print(f"  No chunks:        {per_article_stats['no_chunks']:,}")
print(f"  No target text:   {per_article_stats['no_target_text']:,}")

print(f"\nPer-question survival:")
print(f"  Survived:    {per_question_stats['survived']:,} ({survival_rate:.1f}%)")
print(f"  Blacklisted: {per_question_stats['blacklisted']:,}")

if fuzzy_scores:
    print(f"\nFuzzy score distribution (n={len(fuzzy_scores)}):")
    print(f"  Min: {report['fuzzy_score_distribution']['min']}")
    print(f"  Max: {report['fuzzy_score_distribution']['max']}")
    print(f"  Mean: {report['fuzzy_score_distribution']['mean']}")
    print(f"  Median: {report['fuzzy_score_distribution']['median']}")

if chunk_counts:
    print(f"\nAvg chunks per resolved title: {report['avg_chunks_per_title']}")

print(f"\n{'='*60}")
print("DECISION GATES")
print(f"{'='*60}")

gate_failed = False

if per_article_stats.get("no_target_text", 0) > 0:
    print(f"❌ GATE FAILED: {per_article_stats['no_target_text']} gold titles had no "
          f"recoverable target sentence text.")
    print(f"   This should not happen after the Cell 6 fix. Check whether "
          f"train_subsample.json is stale and regenerate it from Cell 6.")
    gate_failed = True

if survival_rate < 99:
    print(f"❌ GATE FAILED: Question survival {survival_rate:.1f}% < 99%")
    print(f"   Context-based chunking should yield near-perfect survival.")
    print(f"   Investigate Cell 8 for chunking bugs before Phase 3.")
    gate_failed = True

if per_article_stats["no_chunks"] > 0:
    print(f"❌ GATE FAILED: {per_article_stats['no_chunks']} titles have zero chunks.")
    print(f"   Check Cell 8 output for missing titles.")
    gate_failed = True

if not gate_failed:
    print(f"✅ All gates passed. Proceed to Phase 3.")

print(f"\nFull report: {PILOT_REPORT_PATH}")

Loading train subsample ...
Loading chunk index ...
Pilot sample: 500 questions from subsample (out of 30,000 available).

PILOT REPORT
Sample size: 500

Per-article resolution:
  Exact match:      1,000 (100.0%)
  Fuzzy match:      0 (0.0%)
  Combined:         100.0%
  Not found:        0
  No chunks:        0
  No target text:   0

Per-question survival:
  Survived:    500 (100.0%)
  Blacklisted: 0

Avg chunks per resolved title: 1.0

DECISION GATES
✅ All gates passed. Proceed to Phase 3.

Full report: /content/drive/MyDrive/hotpotqa-rag/data/pilot_report.json


In [ ]:
import json
import random
import numpy as np
import faiss
import torch
from pathlib import Path
from sentence_transformers import SentenceTransformer

CHUNK_METADATA_PATH = Path("/content/drive/MyDrive/hotpotqa-rag/data/chunk_metadata.jsonl")
CHUNK_ID_MAP_PATH = Path("/content/drive/MyDrive/hotpotqa-rag/data/chunk_id_map.json")
FAISS_INDEX_PATH = Path("/content/drive/MyDrive/hotpotqa-rag/data/faiss_index.ivfpq")

with open(CHUNK_ID_MAP_PATH, "r", encoding="utf-8") as f:
    chunk_id_map = json.load(f)

chunk_text_by_id = {}
with open(CHUNK_METADATA_PATH, "r", encoding="utf-8") as f:
    for line in f:
        rec = json.loads(line)
        chunk_text_by_id[rec["chunk_id"]] = rec["text"]

index = faiss.read_index(str(FAISS_INDEX_PATH))
index.make_direct_map()  # required for reconstruct() on an IVFPQ index

device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer("BAAI/bge-large-en-v1.5", device=device)

random.seed(123)
sample_positions = random.sample(range(len(chunk_id_map)), 20)

print(f"{'chunk_id':<40} {'cosine_sim':>10}")
sims = []
for pos in sample_positions:
    cid = chunk_id_map[pos]
    text = chunk_text_by_id[cid]

    stored_vec = index.reconstruct(pos)  # PQ-decompressed approximation of what was added
    stored_vec = stored_vec / (np.linalg.norm(stored_vec) + 1e-9)

    fresh_vec = model.encode([text], normalize_embeddings=True, convert_to_numpy=True)[0]

    sim = float(np.dot(stored_vec, fresh_vec))
    sims.append(sim)
    print(f"{cid:<40} {sim:>10.4f}")

print(f"\nMean cosine similarity over {len(sims)} samples: {np.mean(sims):.4f}")
print("Expect high (roughly 0.9+, PQ compression is lossy so not exactly 1.0).")
print("A low or scattered result would mean the stored vector at that position")
print("doesn't correspond to that chunk_id's actual text.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/779 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

chunk_id                                 cosine_sim
Pacific Coast Highway station (Blue Line)::0     0.8236
William Herbert, 3rd Earl of Pembroke::0     0.8777
List of Jewish American economists::0        0.7956
David Xavier Cifu::0                         0.8210
Love Blossoms::0                             0.8382
Bengalized::0                                0.7841
Punta Gorda Woman's Club::0                  0.8025
University of Ruhuna::0                      0.8396
Alex Clark::0                                0.8205
Hey Mama!::0                                 0.8665
Pure Pumpkin Flower::0                       0.8766
Movie theaters in the Netherlands::0         0.8214
Kiss Me, Kate::0                             0.8706
Athletics at the 2017 Southeast Asian Games::0     0.8454
Khaali Haath::0                              0.8829
Woodie Smalls::0                             0.8412
Khalid bin Sultan Al Qasimi::0               0.8473
Oliver Sipple::0                             0.8416
It

In [ ]:
# One-off cleanup: remove Distribution A/B/C/D files and the blacklist
# from the earlier Cell 12 run before rerunning with the updated code.
# Safe to run even if some of these don't exist yet.

from pathlib import Path

DIST_DIR = Path("/content/drive/MyDrive/hotpotqa-rag/data/distributions")

files_to_remove = [
    DIST_DIR / "distribution_A.jsonl",
    DIST_DIR / "distribution_B.jsonl",
    DIST_DIR / "distribution_C.jsonl",
    DIST_DIR / "distribution_D.jsonl",
    DIST_DIR / "blacklist.json",
]

for path in files_to_remove:
    if path.exists():
        path.unlink()
        print(f"Removed {path}")
    else:
        print(f"{path} does not exist, nothing to remove.")

/content/drive/MyDrive/hotpotqa-rag/data/distributions/distribution_A.jsonl does not exist, nothing to remove.
/content/drive/MyDrive/hotpotqa-rag/data/distributions/distribution_B.jsonl does not exist, nothing to remove.
/content/drive/MyDrive/hotpotqa-rag/data/distributions/distribution_C.jsonl does not exist, nothing to remove.
/content/drive/MyDrive/hotpotqa-rag/data/distributions/distribution_D.jsonl does not exist, nothing to remove.
/content/drive/MyDrive/hotpotqa-rag/data/distributions/blacklist.json does not exist, nothing to remove.


In [ ]:
from pathlib import Path

DIST_DIR = Path("/content/drive/MyDrive/hotpotqa-rag/data/distributions")

files_to_remove = [
    DIST_DIR / "distribution_A.jsonl",
    DIST_DIR / "distribution_B.jsonl",
    DIST_DIR / "distribution_C.jsonl",
    DIST_DIR / "distribution_D.jsonl",
    DIST_DIR / "blacklist.json",   # old format, from before this update
    DIST_DIR / "blacklist.txt",    # current format
    DIST_DIR / "processed_qids.txt",
]

for path in files_to_remove:
    if path.exists():
        path.unlink()
        print(f"Removed {path}")
    else:
        print(f"{path} does not exist, nothing to remove.")

Removed /content/drive/MyDrive/hotpotqa-rag/data/distributions/distribution_A.jsonl
Removed /content/drive/MyDrive/hotpotqa-rag/data/distributions/distribution_B.jsonl
Removed /content/drive/MyDrive/hotpotqa-rag/data/distributions/distribution_C.jsonl
Removed /content/drive/MyDrive/hotpotqa-rag/data/distributions/distribution_D.jsonl
/content/drive/MyDrive/hotpotqa-rag/data/distributions/blacklist.json does not exist, nothing to remove.
Removed /content/drive/MyDrive/hotpotqa-rag/data/distributions/blacklist.txt
Removed /content/drive/MyDrive/hotpotqa-rag/data/distributions/processed_qids.txt


In [ ]:
from google.colab import drive

# 1. Force Colab to push all cached files to Google Drive
drive.flush_and_unmount()
print("Sync complete. All files are safely on Google Drive.")

# 2. Re-mount the drive (only if you need to keep working)
drive.mount('/content/drive')


Sync complete. All files are safely on Google Drive.
Mounted at /content/drive


In [ ]:
!ls -la /content/drive/MyDrive/hotpotqa-rag/data/distributions/
!wc -l /content/drive/MyDrive/hotpotqa-rag/data/distributions/*.txt /content/drive/MyDrive/hotpotqa-rag/data/distributions/*.jsonl

total 222199
drwx------ 2 root root     4096 Sep 18 13:20 .
drwx------ 3 root root     4096 Aug  5 07:00 ..
-rw------- 1 root root        0 Sep 12 07:03 blacklist.txt
-rw------- 1 root root 54719654 Sep 18 13:20 distribution_A.jsonl
-rw------- 1 root root 59790769 Sep 18 13:20 distribution_B.jsonl
-rw------- 1 root root 96839126 Sep 18 13:20 distribution_C.jsonl
-rw------- 1 root root 15665922 Sep 18 13:20 distribution_D.jsonl
-rw------- 1 root root   506781 Sep 18 13:20 processed_qids.txt
        0 /content/drive/MyDrive/hotpotqa-rag/data/distributions/blacklist.txt
    15357 /content/drive/MyDrive/hotpotqa-rag/data/distributions/processed_qids.txt
    15366 /content/drive/MyDrive/hotpotqa-rag/data/distributions/distribution_A.jsonl
    15366 /content/drive/MyDrive/hotpotqa-rag/data/distributions/distribution_B.jsonl
    13332 /content/drive/MyDrive/hotpotqa-rag/data/distributions/distribution_C.jsonl
     2025 /content/drive/MyDrive/hotpotqa-rag/data/distributions/distribution_D.json

In [ ]:
!ls -la /content/drive/MyDrive/hotpotqa-rag/data/distributions/
!wc -l /content/drive/MyDrive/hotpotqa-rag/data/distributions/*.txt /content/drive/MyDrive/hotpotqa-rag/data/distributions/*.jsonl

total 178793
drwx------ 2 root root     4096 Sep 17 14:46 .
drwx------ 3 root root     4096 Aug  5 07:00 ..
-rw------- 1 root root        0 Sep 12 07:03 blacklist.txt
-rw------- 1 root root 44045059 Sep 17 14:46 distribution_A.jsonl
-rw------- 1 root root 48143797 Sep 17 14:46 distribution_B.jsonl
-rw------- 1 root root 77926601 Sep 17 14:46 distribution_C.jsonl
-rw------- 1 root root 12551502 Sep 17 14:46 distribution_D.jsonl
-rw------- 1 root root   407517 Sep 17 14:46 processed_qids.txt
        0 /content/drive/MyDrive/hotpotqa-rag/data/distributions/blacklist.txt
    12349 /content/drive/MyDrive/hotpotqa-rag/data/distributions/processed_qids.txt
    12357 /content/drive/MyDrive/hotpotqa-rag/data/distributions/distribution_A.jsonl
    12357 /content/drive/MyDrive/hotpotqa-rag/data/distributions/distribution_B.jsonl
    10729 /content/drive/MyDrive/hotpotqa-rag/data/distributions/distribution_C.jsonl
     1620 /content/drive/MyDrive/hotpotqa-rag/data/distributions/distribution_D.json

In [ ]:
# Cell 12: Distribution Synthesis
# Phase 3: Build Distributions A, B, C, D from train_subsample.json.
#
# Uses the FAISS index and BGE model for live retrieval, plus a
# cross-encoder reranker for Distribution C/D, to match the planned
# two-stage production pipeline (ANN candidate pool -> rerank -> top-k).
#
# Hop-style B/C/D processing (a shown entity + a withheld entity to
# reconstruct a search query for) applies to all bridge questions, plus
# any comparison question where exactly one gold title is directly
# named in the question text.
#
# Every first-turn context is fixed at exactly 5 chunks total, matching
# what real retrieval always returns (TOP_K=5), so chunk count carries
# no signal about which distribution an example came from:
#   A:            2 gold + 3 distractors = 5
#   B / C / D-hop: 1 gold (hop1) + 4 distractors = 5
#   D (comparison, no gold shown): 0 gold + 5 distractors = 5
# Gold position within that 5 is shuffled per question, so "the answer
# is always chunk [1]" isn't a second shortcut alongside chunk count.
# The retrieval turn inside C/D was already fixed at TOP_K=5 and already
# reflects genuine cross-encoder rank order, no change needed there.
#
# Checkpointed: writes each question's output immediately (flushed) and
# tracks processed qids, so a disconnect only costs work since the last
# flush. Random draws are seeded per-question (not from one global
# sequential stream), so results are identical regardless of how many
# times the run is interrupted and resumed.

import json
import random
import re
import hashlib
from pathlib import Path
from typing import Dict, List, Set, Tuple, Optional
from collections import defaultdict, Counter
from contextlib import ExitStack

import numpy as np
import faiss
import torch
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer, CrossEncoder

# ── Configuration ─────────────────────────────────────────────────────────────
TRAIN_SUBSAMPLE_PATH = Path("/content/drive/MyDrive/hotpotqa-rag/data/train_subsample.json")
CHUNK_INDEX_PATH = Path("/content/drive/MyDrive/hotpotqa-rag/data/chunk_index.json")
FAISS_INDEX_PATH = Path("/content/drive/MyDrive/hotpotqa-rag/data/faiss_index.ivfpq")
CHUNK_ID_MAP_PATH = Path("/content/drive/MyDrive/hotpotqa-rag/data/chunk_id_map.json")
DIST_DIR = Path("/content/drive/MyDrive/hotpotqa-rag/data/distributions")

DIST_DIR.mkdir(parents=True, exist_ok=True)

DIST_A_PATH = DIST_DIR / "distribution_A.jsonl"
DIST_B_PATH = DIST_DIR / "distribution_B.jsonl"
DIST_C_PATH = DIST_DIR / "distribution_C.jsonl"
DIST_D_PATH = DIST_DIR / "distribution_D.jsonl"
BLACKLIST_PATH = DIST_DIR / "blacklist.txt"
PROCESSED_QIDS_PATH = DIST_DIR / "processed_qids.txt"

BGE_MODEL = "BAAI/bge-large-en-v1.5"
RERANKER_MODEL = "BAAI/bge-reranker-large"
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "

TOP_K = 5
ANN_CANDIDATE_POOL = 50
CROSS_ENCODER_BATCH_SIZE = 32

# Fixed context sizes: total chunks always sum to 5, matching real
# retrieval's TOP_K=5, so chunk count never signals distribution type.
A_DISTRACTOR_COUNT = 3        # + 2 gold = 5
HOP_DISTRACTOR_COUNT = 4       # + 1 gold (hop1) = 5
COMPARISON_D_DISTRACTOR_COUNT = 5  # + 0 gold = 5
DISTRACTOR_POOL_SIZE = 5      # select_distractor_chunks must supply at least this many

IMPERFECT_C_FRACTION = 0.0
COMPARISON_D_PROB = 0.30

FAISS_NPROBE = 32
BASE_SEED = 42

def qid_seed(qid: str) -> int:
    """Deterministic per-question seed, independent of processing order."""
    h = hashlib.md5(f"{BASE_SEED}:{qid}".encode("utf-8")).hexdigest()
    return int(h, 16) % (2**32)

# ── Load data ─────────────────────────────────────────────────────────────────
print("Loading train subsample ...")
with open(TRAIN_SUBSAMPLE_PATH, "r", encoding="utf-8") as f:
    train_subsample: Dict[str, Dict] = json.load(f)

print("Loading chunk index ...")
with open(CHUNK_INDEX_PATH, "r", encoding="utf-8") as f:
    chunk_index: Dict[str, List[Dict]] = json.load(f)

print("Loading chunk ID map ...")
with open(CHUNK_ID_MAP_PATH, "r", encoding="utf-8") as f:
    chunk_id_map: List[str] = json.load(f)

chunk_lookup: Dict[str, Dict] = {
    c["chunk_id"]: c
    for chunks_for_title in chunk_index.values()
    for c in chunks_for_title
}

print("Loading FAISS index ...")
faiss_index = faiss.read_index(str(FAISS_INDEX_PATH))
faiss_index.nprobe = FAISS_NPROBE
print(f"FAISS nprobe set to {FAISS_NPROBE} (of {faiss_index.nlist:,} clusters).")

print(f"Loading BGE embedding model: {BGE_MODEL}")
device = "cuda" if torch.cuda.is_available() else "cpu"
bge_model = SentenceTransformer(BGE_MODEL, device=device)

print(f"Loading cross-encoder reranker: {RERANKER_MODEL}")
reranker_model = CrossEncoder(RERANKER_MODEL, max_length=512, device=device)

print(f"\nConfig: TOP_K={TOP_K}, ANN_CANDIDATE_POOL={ANN_CANDIDATE_POOL}, "
      f"A_DISTRACTOR_COUNT={A_DISTRACTOR_COUNT}, HOP_DISTRACTOR_COUNT={HOP_DISTRACTOR_COUNT}, "
      f"COMPARISON_D_DISTRACTOR_COUNT={COMPARISON_D_DISTRACTOR_COUNT}, "
      f"IMPERFECT_C_FRACTION={IMPERFECT_C_FRACTION}, COMPARISON_D_PROB={COMPARISON_D_PROB}, "
      f"FAISS_NPROBE={FAISS_NPROBE}")

# ── Helper: stopwords for query synthesis ───────────────────────────────────────
_STOPWORDS = {
    "the","a","an","is","are","was","were","be","been","being","have","has","had",
    "do","does","did","will","would","could","should","may","might","must","shall",
    "can","need","dare","ought","used","to","of","in","for","on","with","at","by",
    "from","as","into","through","during","before","after","above","below","between",
    "under","and","but","or","yet","so","if","because","although","though","while",
    "where","when","that","which","who","whom","whose","what","how","this","these",
    "those","i","you","he","she","it","we","they","me","him","her","us","them","my",
    "your","his","its","our","their","mine","yours","hers","ours","theirs","all",
    "each","every","both","few","more","most","other","some","such","no","nor","not",
    "only","own","same","than","too","very","just","now","then","here","there","up",
    "down","out","off","over","under","again","further","once","also"
}

# ── Helper functions ──────────────────────────────────────────────────────────
def find_gold_chunk(title: str, sf_texts: Dict[str, Dict[str, str]], chunk_index: Dict) -> Optional[Dict]:
    """
    Deterministic: supporting-fact text must be a substring of the chunk.
    sf_texts is {title: {sent_id: text}} as of the Cell 6 schema.
    """
    targets = list(sf_texts.get(title, {}).values())
    for chunk in chunk_index.get(title, []):
        for target in targets:
            if target in chunk["text"]:
                return chunk
    return None

def get_hop_order(gold_titles: List[str], question: str) -> Tuple[List[str], bool]:
    """
    Heuristic: the gold title whose words appear in the question is hop 1.
    Returns (hop_order, eligible). eligible is True only when exactly one
    gold title's words are found in the question. For bridge questions
    this flag is informational only, hop-style processing applies to all
    of them regardless. For comparison questions, eligible gates entry
    into hop-style B/C/D processing at all.
    """
    q_lower = question.lower()
    hits = []
    for i, title in enumerate(gold_titles):
        words = [w for w in title.lower().split() if len(w) > 3]
        if any(w in q_lower for w in words):
            hits.append(i)

    eligible = len(hits) == 1
    if hits:
        i = hits[0]
        ordered = list(gold_titles)
        ordered.pop(i)
        ordered.insert(0, gold_titles[i])
        return ordered, eligible
    return list(gold_titles), eligible

def synthesize_query(question: str, withheld_title: str, first_chunk_text: str) -> str:
    """
    Rule-based query synthesis: withheld title + attribute words from question
    that do not appear in the first gold chunk.
    """
    q_words = set(re.findall(r'\b[a-zA-Z]+\b', question.lower()))
    chunk_words = set(re.findall(r'\b[a-zA-Z]+\b', first_chunk_text.lower()))
    title_words = set(re.findall(r'\b[a-zA-Z]+\b', withheld_title.lower()))

    candidates = [w for w in q_words
                  if w not in chunk_words and w not in title_words and w not in _STOPWORDS]

    if len(candidates) < 2:
        candidates = [w for w in q_words if w not in title_words and w not in _STOPWORDS]
    if len(candidates) < 1:
        return withheld_title

    candidates = sorted(candidates, key=lambda w: len(w), reverse=True)
    attribute = " ".join(candidates[:3])
    return f"{withheld_title} {attribute}".strip()

def format_context(chunks: List[Dict], start_idx: int = 1) -> str:
    lines = []
    for i, chunk in enumerate(chunks, start=start_idx):
        lines.append(f"[{i}] {chunk['title']}:\n{chunk['text']}")
    return "\n\n".join(lines)

def select_distractor_chunks(
    question: str,
    distractor_titles: List[str],
    top_n: int,
    rng: random.Random,
) -> List[Dict]:
    """
    Use FAISS to find the most seductive distractor chunks (topically
    adjacent but factually insufficient), restricted to this question's
    own official distractor_titles. Single-stage ANN, no reranking: this
    pool feeds A/B/D-comparison, which are fixed-size context slots, not
    simulations of the live retrieval pipeline the way retrieve_for_c is.
    Falls back to random chunks if FAISS returns too few.
    """
    q_emb = bge_model.encode(
        [QUERY_PREFIX + question],
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype("float32")

    scores, indices = faiss_index.search(q_emb, 20)
    distractor_set = set(distractor_titles)

    results = []
    seen_ids = set()

    for idx in indices[0]:
        if idx == -1:
            continue
        cid = chunk_id_map[idx]
        title = cid.split("::")[0]
        if title in distractor_set and cid not in seen_ids:
            results.append(chunk_lookup[cid])
            seen_ids.add(cid)
        if len(results) >= top_n:
            break

    if len(results) < top_n:
        pool = []
        for t in distractor_titles:
            for chunk in chunk_index.get(t, []):
                if chunk["chunk_id"] not in seen_ids:
                    pool.append(chunk)
        rng.shuffle(pool)
        results.extend(pool[:top_n - len(results)])

    return results

def retrieve_for_c(
    query: str, gold_chunk: Dict, ann_pool_size: int = 50, final_k: int = 5
) -> Tuple[List[Dict], bool, bool]:
    """
    Two-stage retrieval: FAISS ANN candidate pool, then cross-encoder
    rerank down to final_k. Returns (final_chunks, gold_in_final_top_k,
    gold_in_ann_pool). final_chunks is already in genuine descending
    rerank-score order, gold's position here is whatever the real score
    produced, not artificially forced.
    """
    q_emb = bge_model.encode(
        [QUERY_PREFIX + query],
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype("float32")

    scores, indices = faiss_index.search(q_emb, ann_pool_size)
    candidate_ids = [chunk_id_map[idx] for idx in indices[0] if idx != -1]
    candidate_chunks = [chunk_lookup[cid] for cid in candidate_ids]

    gold_in_ann_pool = gold_chunk["chunk_id"] in candidate_ids

    if not candidate_chunks:
        return [], False, gold_in_ann_pool

    pairs = [[query, c["text"]] for c in candidate_chunks]
    rerank_scores = reranker_model.predict(
        pairs,
        batch_size=CROSS_ENCODER_BATCH_SIZE,
        show_progress_bar=False,
    )

    ranked = sorted(
        zip(candidate_chunks, rerank_scores),
        key=lambda x: x[1],
        reverse=True,
    )
    final_chunks = [c for c, _ in ranked[:final_k]]
    final_ids = {c["chunk_id"] for c in final_chunks}

    gold_in_final_top_k = gold_chunk["chunk_id"] in final_ids

    return final_chunks, gold_in_final_top_k, gold_in_ann_pool

# ── Resume support ─────────────────────────────────────────────────────────────
processed_qids: Set[str] = set()
if PROCESSED_QIDS_PATH.exists():
    with open(PROCESSED_QIDS_PATH, "r", encoding="utf-8") as f:
        processed_qids = {line.strip() for line in f if line.strip()}

pending_qids = [qid for qid in train_subsample if qid not in processed_qids]

if processed_qids:
    print(f"Resuming: {len(processed_qids):,} questions already processed.")
print(f"Pending: {len(pending_qids):,} questions")

# ── Main synthesis loop ───────────────────────────────────────────────────────
with ExitStack() as stack:
    f_a = stack.enter_context(open(DIST_A_PATH, "a", encoding="utf-8"))
    f_b = stack.enter_context(open(DIST_B_PATH, "a", encoding="utf-8"))
    f_c = stack.enter_context(open(DIST_C_PATH, "a", encoding="utf-8"))
    f_d = stack.enter_context(open(DIST_D_PATH, "a", encoding="utf-8"))
    f_blacklist = stack.enter_context(open(BLACKLIST_PATH, "a", encoding="utf-8"))
    f_progress = stack.enter_context(open(PROCESSED_QIDS_PATH, "a", encoding="utf-8"))

    def write_record(f, record: Dict) -> None:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")
        f.flush()

    for qid in tqdm(pending_qids):
        rec = train_subsample[qid]
        rng = random.Random(qid_seed(qid))

        # ── Resolve gold chunks ───────────────────────────────────────────────
        gold_chunks = []
        is_blacklisted = False
        for title in rec["gold_titles"]:
            chunk = find_gold_chunk(title, rec.get("supporting_fact_texts", {}), chunk_index)
            if chunk is None:
                is_blacklisted = True
                break
            gold_chunks.append(chunk)

        if is_blacklisted:
            f_blacklist.write(qid + "\n")
            f_blacklist.flush()
            f_progress.write(qid + "\n")
            f_progress.flush()
            continue

        all_distractors = select_distractor_chunks(
            rec["question"], rec["distractor_titles"], top_n=DISTRACTOR_POOL_SIZE, rng=rng
        )

        # ── Distribution A: Direct Answer ─────────────────────────────────────
        # Fixed at 2 gold + 3 distractors = 5 total, position shuffled so
        # gold isn't always chunk [1].
        a_distractors = all_distractors[:A_DISTRACTOR_COUNT]
        a_chunks = gold_chunks + a_distractors
        rng.shuffle(a_chunks)
        a_context = format_context(a_chunks)

        write_record(f_a, {
            "messages": [
                {"role": "user", "content": f"{a_context}\n\nQuestion: {rec['question']}"},
                {"role": "assistant", "content": f"Answer: {rec['answer']}"}
            ],
            "provenance": {
                "qid": qid,
                "distribution": "A",
                "question_type": rec["type"],
                "level": rec["level"],
                "gold_chunks": [{"title": c["title"], "chunk_id": c["chunk_id"]} for c in gold_chunks],
                "distractor_chunks": [{"title": d["title"], "chunk_id": d["chunk_id"]} for d in a_distractors],
                "num_distractors": len(a_distractors),
            }
        })

        # ── Determine hop-style eligibility ───────────────────────────────────
        hop_order, hop_eligible = get_hop_order(rec["gold_titles"], rec["question"])
        route_hop_style = True if rec["type"] == "bridge" else hop_eligible

        if not route_hop_style:
            # Comparison question with both (or neither) gold title directly
            # named: no natural withheld entity, skips B/C entirely.
            # Fixed at 0 gold + 5 distractors = 5 total.
            if rng.random() < COMPARISON_D_PROB:
                d_distractors = all_distractors[:COMPARISON_D_DISTRACTOR_COUNT]
                d_chunks = list(d_distractors)
                rng.shuffle(d_chunks)
                d_context = format_context(d_chunks, start_idx=1)

                write_record(f_d, {
                    "messages": [
                        {"role": "user", "content": f"{d_context}\n\nQuestion: {rec['question']}"},
                        {"role": "assistant", "content": "Answer: I cannot determine the answer from the provided context."}
                    ],
                    "provenance": {
                        "qid": qid,
                        "distribution": "D",
                        "question_type": rec["type"],
                        "level": rec["level"],
                        "d_type": "comparison_no_gold",
                        "context_chunks": [{"title": d["title"], "chunk_id": d["chunk_id"]} for d in d_distractors],
                    }
                })

            f_progress.write(qid + "\n")
            f_progress.flush()
            continue

        # ── Hop-style processing ───────────────────────────────────────────────
        # First turn fixed at 1 gold (hop1) + 4 distractors = 5 total,
        # position shuffled. Shared unchanged across B and C/D's first turn.
        hop1_title, hop2_title = hop_order[0], hop_order[1]
        hop1_chunk = next(c for c in gold_chunks if c["title"] == hop1_title)
        hop2_chunk = next(c for c in gold_chunks if c["title"] == hop2_title)

        b_distractors = all_distractors[:HOP_DISTRACTOR_COUNT]
        b_chunks = [hop1_chunk] + b_distractors
        rng.shuffle(b_chunks)
        b_context = format_context(b_chunks)

        query = synthesize_query(rec["question"], hop2_title, hop1_chunk["text"])

        write_record(f_b, {
            "messages": [
                {"role": "user", "content": f"{b_context}\n\nQuestion: {rec['question']}"},
                {"role": "assistant", "content": f"Search: {query}"}
            ],
            "provenance": {
                "qid": qid,
                "distribution": "B",
                "question_type": rec["type"],
                "level": rec["level"],
                "hop_order": hop_order,
                "withheld_title": hop2_title,
                "synthesized_query": query,
                "gold_chunk": {"title": hop1_chunk["title"], "chunk_id": hop1_chunk["chunk_id"]},
                "distractor_chunks": [{"title": d["title"], "chunk_id": d["chunk_id"]} for d in b_distractors],
            }
        })

        retrieved_chunks, gold_in_top_k, gold_in_ann_pool = retrieve_for_c(
            query, hop2_chunk, ann_pool_size=ANN_CANDIDATE_POOL, final_k=TOP_K
        )

        if gold_in_top_k or (not gold_in_top_k and rng.random() < IMPERFECT_C_FRACTION):
            write_record(f_c, {
                "messages": [
                    {"role": "user", "content": f"{b_context}\n\nQuestion: {rec['question']}"},
                    {"role": "assistant", "content": f"Search: {query}"},
                    {"role": "user", "content": "Retrieved information:\n" + format_context(retrieved_chunks, start_idx=1)},
                    {"role": "assistant", "content": f"Synthesis: {rec['answer']}"}
                ],
                "provenance": {
                    "qid": qid,
                    "distribution": "C",
                    "question_type": rec["type"],
                    "level": rec["level"],
                    "hop_order": hop_order,
                    "synthesized_query": query,
                    "retrieved_chunks": [{"title": c["title"], "chunk_id": c["chunk_id"]} for c in retrieved_chunks],
                    "gold_in_top_k": gold_in_top_k,
                    "gold_in_ann_pool": gold_in_ann_pool,
                    "ann_pool_size": ANN_CANDIDATE_POOL,
                    "gold_chunk": {"title": hop2_chunk["title"], "chunk_id": hop2_chunk["chunk_id"]},
                }
            })
        else:
            write_record(f_d, {
                "messages": [
                    {"role": "user", "content": f"{b_context}\n\nQuestion: {rec['question']}"},
                    {"role": "assistant", "content": f"Search: {query}"},
                    {"role": "user", "content": "Retrieved information:\n" + format_context(retrieved_chunks, start_idx=1)},
                    {"role": "assistant", "content": "Answer: I cannot determine the answer from the provided context."}
                ],
                "provenance": {
                    "qid": qid,
                    "distribution": "D",
                    "question_type": rec["type"],
                    "level": rec["level"],
                    "d_type": "bridge_failed_retrieval",
                    "hop_order": hop_order,
                    "synthesized_query": query,
                    "retrieved_chunks": [{"title": c["title"], "chunk_id": c["chunk_id"]} for c in retrieved_chunks],
                    "gold_in_top_k": gold_in_top_k,
                    "gold_in_ann_pool": gold_in_ann_pool,
                    "ann_pool_size": ANN_CANDIDATE_POOL,
                    "gold_chunk": {"title": hop2_chunk["title"], "chunk_id": hop2_chunk["chunk_id"]},
                }
            })

        f_progress.write(qid + "\n")
        f_progress.flush()

print("\nSynthesis loop complete.")

# ── Recompute summary stats from the saved files (robust to resumes) ──────────
def load_jsonl(path: Path) -> List[Dict]:
    if not path.exists():
        return []
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

def load_lines(path: Path) -> List[str]:
    if not path.exists():
        return []
    with open(path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]

dist_A_all = load_jsonl(DIST_A_PATH)
dist_B_all = load_jsonl(DIST_B_PATH)
dist_C_all = load_jsonl(DIST_C_PATH)
dist_D_all = load_jsonl(DIST_D_PATH)
blacklist_all = load_lines(BLACKLIST_PATH)

total_q = len(train_subsample)
blacklist_rate = len(blacklist_all) / total_q * 100 if total_q else 0

type_counts = Counter(r["provenance"]["question_type"] for r in dist_A_all)
comparison_count = type_counts.get("comparison", 0)
bridge_count = type_counts.get("bridge", 0)
comparison_hop_style_count = sum(1 for r in dist_B_all if r["provenance"]["question_type"] == "comparison")

hop_style_D = [r for r in dist_D_all if r["provenance"].get("d_type") == "bridge_failed_retrieval"]
comparison_D = [r for r in dist_D_all if r["provenance"].get("d_type") == "comparison_no_gold"]

total_hop_retrievals = len(dist_B_all)
gold_retrieval_hits = sum(1 for r in dist_C_all if r["provenance"]["gold_in_top_k"]) + \
                       sum(1 for r in hop_style_D if r["provenance"]["gold_in_top_k"])
gold_ann_pool_hits = sum(1 for r in dist_C_all if r["provenance"]["gold_in_ann_pool"]) + \
                      sum(1 for r in hop_style_D if r["provenance"]["gold_in_ann_pool"])

gold_hit_rate = gold_retrieval_hits / total_hop_retrievals * 100 if total_hop_retrievals else 0
ann_pool_hit_rate = gold_ann_pool_hits / total_hop_retrievals * 100 if total_hop_retrievals else 0
comparison_no_gold_denom = comparison_count - comparison_hop_style_count
comparison_D_rate = len(comparison_D) / comparison_no_gold_denom * 100 if comparison_no_gold_denom else 0

# Verify the fixed-5-chunk design actually held, count chunks in each
# distribution's first turn directly from the saved text.
def count_chunks_in_first_turn(record: Dict) -> int:
    first_user_msg = record["messages"][0]["content"]
    return len(re.findall(r'^\[\d+\]', first_user_msg, re.MULTILINE))

a_chunk_counts = Counter(count_chunks_in_first_turn(r) for r in dist_A_all)
b_chunk_counts = Counter(count_chunks_in_first_turn(r) for r in dist_B_all)
comparison_d_chunk_counts = Counter(count_chunks_in_first_turn(r) for r in comparison_D)

print(f"\n{'='*60}")
print("DISTRIBUTION SYNTHESIS SUMMARY (cumulative across all runs)")
print(f"{'='*60}")
print(f"Total questions in subsample: {total_q:,}")
print(f"Blacklisted (no resolvable gold chunk): {len(blacklist_all):,} ({blacklist_rate:.2f}%)")
print(f"\nQuestion types processed: {comparison_count:,} comparison "
      f"({comparison_hop_style_count:,} routed through hop-style B/C), {bridge_count:,} bridge")

print(f"\nDistribution A (Direct Answer):              {len(dist_A_all):,}")
print(f"Distribution B (Query Reconstruction):        {len(dist_B_all):,}")
print(f"Distribution C (Synthesis After Retrieval):   {len(dist_C_all):,}")
print(f"Distribution D (Insufficient Evidence):       {len(dist_D_all):,}")
print(f"  - from comparison (no gold shown):          {len(comparison_D):,} "
      f"({comparison_D_rate:.1f}% of {comparison_no_gold_denom:,} non-hop-eligible comparison Qs, "
      f"target {COMPARISON_D_PROB*100:.0f}%)")
print(f"  - from hop-style (failed retrieval):        {len(hop_style_D):,}")

print(f"\nHop-style retrieval quality (bridge + eligible comparison, n={total_hop_retrievals:,}):")
print(f"  In ANN candidate pool (top {ANN_CANDIDATE_POOL}):    {gold_ann_pool_hits:,} ({ann_pool_hit_rate:.1f}%)")
print(f"  In final reranked top {TOP_K}:                {gold_retrieval_hits:,} ({gold_hit_rate:.1f}%)")
if gold_ann_pool_hits > gold_retrieval_hits:
    lost_to_rerank = gold_ann_pool_hits - gold_retrieval_hits
    lost_pct = lost_to_rerank / total_hop_retrievals * 100 if total_hop_retrievals else 0
    print(f"  -> {lost_to_rerank:,} cases ({lost_pct:.1f}%) had gold in the candidate pool "
          f"but the reranker didn't keep it in the final top {TOP_K}.")

print(f"\nFixed-5-chunk check (first turn, chunk count distribution):")
print(f"  Distribution A:            {dict(a_chunk_counts)}")
print(f"  Distribution B:            {dict(b_chunk_counts)}")
print(f"  Distribution D (comp-no-gold): {dict(comparison_d_chunk_counts)}")

print(f"\n{'='*60}")
print("SANITY CHECKS")
print(f"{'='*60}")
if blacklist_rate > 2:
    print(f"⚠ {blacklist_rate:.2f}% blacklisted is higher than expected.")
else:
    print(f"✓ Blacklist rate ({blacklist_rate:.2f}%) is consistent with the pilot's near-100% resolution.")

if total_hop_retrievals and gold_hit_rate < 50:
    if ann_pool_hit_rate - gold_hit_rate > 20:
        print(f"⚠ Final hit rate ({gold_hit_rate:.1f}%) is low, but ANN pool hit rate "
              f"({ann_pool_hit_rate:.1f}%) is much higher, worth spot-checking the reranker.")
    else:
        print(f"⚠ Gold retrieval hit rate ({gold_hit_rate:.1f}%) looks low even before "
              f"reranking. Double-check FAISS_NPROBE.")
else:
    print(f"✓ Gold retrieval hit rate ({gold_hit_rate:.1f}%) looks reasonable.")

if set(a_chunk_counts.keys()) not in ({5}, set()):
    print(f"⚠ Distribution A has first-turn chunk counts other than 5: {dict(a_chunk_counts)}. "
          f"This means the fixed-5 design didn't hold everywhere, check for questions with "
          f"fewer than {A_DISTRACTOR_COUNT} available distractors.")
else:
    print(f"✓ Distribution A is fixed at exactly 5 chunks per first turn, as intended.")

if set(b_chunk_counts.keys()) not in ({5}, set()):
    print(f"⚠ Distribution B has first-turn chunk counts other than 5: {dict(b_chunk_counts)}. "
          f"Check for questions with fewer than {HOP_DISTRACTOR_COUNT} available distractors.")
else:
    print(f"✓ Distribution B is fixed at exactly 5 chunks per first turn, as intended.")

for name, dist in [("A", dist_A_all), ("B", dist_B_all), ("C", dist_C_all), ("D", dist_D_all)]:
    if not dist:
        print(f"⚠ Distribution {name} is empty, check upstream logic for that branch.")

covered = len(dist_A_all) + len(blacklist_all)
if covered != total_q:
    print(f"⚠ {covered:,} questions accounted for (A records + blacklist) vs {total_q:,} "
          f"total, if this run was interrupted, rerun this cell to pick up where it left off.")
else:
    print(f"✓ All {total_q:,} questions accounted for.")

print(f"\nSaved to: {DIST_DIR}")

Loading train subsample ...
Loading chunk index ...
Loading chunk ID map ...
Loading FAISS index ...
FAISS nprobe set to 32 (of 1,961 clusters).
Loading BGE embedding model: BAAI/bge-large-en-v1.5


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]


Config: TOP_K=5, ANN_CANDIDATE_POOL=50, A_DISTRACTOR_COUNT=3, HOP_DISTRACTOR_COUNT=4, COMPARISON_D_DISTRACTOR_COUNT=5, IMPERFECT_C_FRACTION=0.0, COMPARISON_D_PROB=0.3, FAISS_NPROBE=32
Resuming: 30,000 questions already processed.
Pending: 0 questions


0it [00:00, ?it/s]


Synthesis loop complete.

DISTRIBUTION SYNTHESIS SUMMARY (cumulative across all runs)
Total questions in subsample: 30,000
Blacklisted (no resolvable gold chunk): 0 (0.00%)

Question types processed: 5,640 comparison (110 routed through hop-style B/C), 24,372 bridge

Distribution A (Direct Answer):              30,012
Distribution B (Query Reconstruction):        24,482
Distribution C (Synthesis After Retrieval):   21,220
Distribution D (Insufficient Evidence):       4,981
  - from comparison (no gold shown):          1,628 (29.4% of 5,530 non-hop-eligible comparison Qs, target 30%)
  - from hop-style (failed retrieval):        3,353

Hop-style retrieval quality (bridge + eligible comparison, n=24,482):
  In ANN candidate pool (top 50):    21,531 (87.9%)
  In final reranked top 5:                21,220 (86.7%)
  -> 311 cases (1.3%) had gold in the candidate pool but the reranker didn't keep it in the final top 5.

Fixed-5-chunk check (first turn, chunk count distribution):
  Distribut

In [ ]:
!ls -la /content/drive/MyDrive/hotpotqa-rag/data/distributions/
!wc -l /content/drive/MyDrive/hotpotqa-rag/data/distributions/*.txt /content/drive/MyDrive/hotpotqa-rag/data/distributions/*.jsonl

total 142587
drwx------ 2 root root     4096 Sep  1 09:20 .
drwx------ 3 root root     4096 Aug  5 07:00 ..
-rw------- 1 root root        0 Sep 12 07:03 blacklist.txt
-rw------- 1 root root 35142455 Sep 14 13:26 distribution_A.jsonl
-rw------- 1 root root 38408865 Sep 14 13:26 distribution_B.jsonl
-rw------- 1 root root 61978918 Sep 14 13:26 distribution_C.jsonl
-rw------- 1 root root 10144555 Sep 14 11:34 distribution_D.jsonl
-rw------- 1 root root   324423 Sep 14 13:26 processed_qids.txt
        0 /content/drive/MyDrive/hotpotqa-rag/data/distributions/blacklist.txt
     9831 /content/drive/MyDrive/hotpotqa-rag/data/distributions/processed_qids.txt
     9837 /content/drive/MyDrive/hotpotqa-rag/data/distributions/distribution_A.jsonl
     9837 /content/drive/MyDrive/hotpotqa-rag/data/distributions/distribution_B.jsonl
     8524 /content/drive/MyDrive/hotpotqa-rag/data/distributions/distribution_C.jsonl
     1307 /content/drive/MyDrive/hotpotqa-rag/data/distributions/distribution_D.json

In [ ]:
# Cell 13: Phase 4 Audit
# Five structural/behavioral checks plus a sixth (chunk-count) before any
# training. Loads distribution JSONL files, validates every record in a
# single pass per record, splits clean vs. contaminated, and reports.

import json
import re
from pathlib import Path
from typing import Dict, List, Tuple
from collections import Counter

# ── Configuration ─────────────────────────────────────────────────────────────
DIST_DIR = Path("/content/drive/MyDrive/hotpotqa-rag/data/distributions")
AUDIT_REPORT_PATH = Path("/content/drive/MyDrive/hotpotqa-rag/data/audit_report.json")
CLEAN_DIR = DIST_DIR / "clean"
CONTAM_DIR = DIST_DIR / "contaminated"

CLEAN_DIR.mkdir(parents=True, exist_ok=True)
CONTAM_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_CHUNK_COUNT = 5  # matches the fixed-5-chunk design in Cell 12

# ── Heuristic patterns ────────────────────────────────────────────────────────
QUERY_PROSE_PATTERNS = [
    r"\bi need\b", r"\bi want\b", r"\bplease\b", r"\bfind information\b",
    r"\bsearch for\b", r"\blook for\b", r"\baccording to\b", r"\bi think\b",
    r"\bi believe\b", r"\bcan you\b", r"\bcould you\b", r"\bhelp me\b",
    r"\btell me\b", r"\bgive me\b", r"\bwhat is\b", r"\bwhere is\b",
]

ANSWER_INDICATORS = [r"\bis\s+[A-Z]", r"\bare\s+[A-Z]", r"\bwas\s+[A-Z]", r"\bwere\s+[A-Z]"]

REFUSAL_PHRASES = ["cannot determine", "i cannot", "not enough information", "unable to answer"]

# ── Loaders ───────────────────────────────────────────────────────────────────
def load_dist(name: str) -> List[Dict]:
    # Cell 12 writes distribution_A.jsonl / _B / _C / _D, not dist_A.jsonl.
    path = DIST_DIR / f"distribution_{name}.jsonl"
    if not path.exists():
        print(f"[WARN] {path} does not exist yet, treating Distribution {name} as empty.")
        return []
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

# ── Check: Chunk Count Consistency (independent re-check of the fixed-5 design) ──
def count_chunks_in_first_turn(record: Dict) -> int:
    messages = record.get("messages", [])
    if not messages:
        return -1
    first_turn = messages[0].get("content", "")
    return len(re.findall(r'^\[\d+\]', first_turn, re.MULTILINE))

def check_chunk_count(record: Dict, expected: int = EXPECTED_CHUNK_COUNT) -> Tuple[bool, str]:
    count = count_chunks_in_first_turn(record)
    if count != expected:
        return False, f"expected_{expected}_got_{count}"
    return True, "ok"

# ── Check: Prefix Adherence ───────────────────────────────────────────────────
def check_prefix(record: Dict, dist: str) -> Tuple[bool, str]:
    msgs = [m for m in record.get("messages", []) if m.get("role") == "assistant"]

    if dist == "A":
        if not msgs or not msgs[0].get("content", "").startswith("Answer:"):
            return False, "missing_Answer_prefix"

    elif dist == "B":
        if not msgs or not msgs[0].get("content", "").startswith("Search:"):
            return False, "missing_Search_prefix"

    elif dist == "C":
        if len(msgs) < 2:
            return False, "missing_second_assistant_turn"
        if not msgs[0].get("content", "").startswith("Search:"):
            return False, "first_turn_missing_Search"
        if not msgs[1].get("content", "").startswith("Synthesis:"):
            return False, "second_turn_missing_Synthesis"

    elif dist == "D":
        # D has two shapes depending on d_type: comparison_no_gold is a
        # single refusal turn, bridge_failed_retrieval mirrors C's shape
        # (search, then retrieval, then refusal instead of synthesis).
        # Checking msgs[0] against "Answer:" unconditionally, as an
        # earlier draft did, would fail every bridge_failed_retrieval
        # record, since msgs[0] there is "Search:", not "Answer:".
        d_type = record.get("provenance", {}).get("d_type")
        if d_type == "comparison_no_gold":
            if not msgs or not msgs[0].get("content", "").startswith("Answer:"):
                return False, "missing_Answer_prefix"
        elif d_type == "bridge_failed_retrieval":
            if len(msgs) < 2:
                return False, "missing_second_assistant_turn"
            if not msgs[0].get("content", "").startswith("Search:"):
                return False, "first_turn_missing_Search"
            if not msgs[-1].get("content", "").startswith("Answer:"):
                return False, "last_turn_missing_Answer"
        else:
            return False, f"unknown_d_type:{d_type}"

    return True, "ok"

# ── Check: Query Purity (Distribution B only) ─────────────────────────────────
def check_query_purity(record: Dict) -> Tuple[bool, str]:
    msgs = [m for m in record.get("messages", []) if m.get("role") == "assistant"]
    if not msgs:
        return False, "no_assistant_turn"
    content = msgs[0].get("content", "")
    if not content.startswith("Search:"):
        return False, "not_a_search_turn"
    query = content[len("Search:"):].strip()

    for pat in QUERY_PROSE_PATTERNS:
        if re.search(pat, query, re.IGNORECASE):
            return False, f"prose_detected:{pat}"

    words = query.split()
    if len(words) > 12:
        return False, f"too_long_{len(words)}_words"

    prov = record.get("provenance", {})
    withheld = prov.get("withheld_title", "")
    if withheld and withheld.lower() not in query.lower():
        return False, "withheld_title_absent"

    return True, "ok"

# ── Check: Mode Leakage ────────────────────────────────────────────────────────
def check_mode_leakage(record: Dict, dist: str) -> Tuple[bool, str]:
    msgs = [m for m in record.get("messages", []) if m.get("role") == "assistant"]
    if not msgs:
        return False, "no_assistant_turn"

    if dist == "A":
        text = msgs[0].get("content", "")
        if "Search:" in text or "Synthesis:" in text:
            return False, "foreign_prefix_in_answer"
        if text.count("?") > 0:
            return False, "question_mark_in_answer"

    elif dist == "B":
        text = msgs[0].get("content", "")
        query = text[len("Search:"):].strip() if text.startswith("Search:") else text
        for pat in ANSWER_INDICATORS:
            if re.search(pat, query):
                return False, f"answer_language_in_query:{pat}"

    elif dist == "C":
        if len(msgs) >= 2:
            synth = msgs[1].get("content", "")
            body = synth[len("Synthesis:"):].strip() if synth.startswith("Synthesis:") else synth
            if "Search:" in body:
                return False, "search_prefix_inside_synthesis"
            if re.search(r"\b(what|where|when|who|why|how)\b.*\?", body.lower()):
                return False, "question_in_synthesis"

    elif dist == "D":
        # Refusal is always the LAST assistant turn regardless of
        # d_type (msgs[-1]), not necessarily msgs[0].
        refusal_text = msgs[-1].get("content", "")
        if re.search(r"\bis\s+[A-Z][a-zA-Z\s]+[.!?]", refusal_text):
            return False, "factual_assertion_in_refusal"

        d_type = record.get("provenance", {}).get("d_type")
        if d_type == "bridge_failed_retrieval":
            # Bonus check: bridge_failed_retrieval's search turn is the
            # same shape as B's, so it deserves the same leakage check.
            search_text = msgs[0].get("content", "")
            body = search_text[len("Search:"):].strip() if search_text.startswith("Search:") else search_text
            for pat in ANSWER_INDICATORS:
                if re.search(pat, body):
                    return False, f"answer_language_in_search_turn:{pat}"

    return True, "ok"

# ── Combined per-record structural audit ──────────────────────────────────────
def audit_record(record: Dict, dist: str) -> Tuple[bool, str]:
    """
    Runs all applicable structural checks for one record, in priority
    order, stopping at the first failure. Returns (is_clean, reason),
    where reason is "ok" or "<check_name>:<detail>".
    """
    ok, reason = check_prefix(record, dist)
    if not ok:
        return False, f"prefix:{reason}"

    ok, reason = check_chunk_count(record)
    if not ok:
        return False, f"chunk_count:{reason}"

    if dist == "B":
        ok, reason = check_query_purity(record)
        if not ok:
            return False, f"purity:{reason}"

    ok, reason = check_mode_leakage(record, dist)
    if not ok:
        return False, f"leak:{reason}"

    return True, "ok"

# ── Check: Entity Consistency (B -> C and B -> D-hop-failed) ──────────────────
def check_entity_consistency(
    b_records: List[Dict], c_records: List[Dict], d_records: List[Dict]
) -> Tuple[bool, List[str]]:
    """
    Verifies the entity B withheld matches the gold entity in whatever
    that question's retrieval attempt actually produced, whether it
    succeeded (landed in C) or failed (landed in D as
    bridge_failed_retrieval). comparison_no_gold D records have no
    hop_order/gold_chunk concept, so they're skipped.
    """
    b_map = {}
    for r in b_records:
        qid = r.get("provenance", {}).get("qid")
        withheld = r.get("provenance", {}).get("withheld_title")
        if qid and withheld:
            b_map[qid] = withheld

    failures = []

    for r in c_records:
        qid = r.get("provenance", {}).get("qid")
        gold_title = r.get("provenance", {}).get("gold_chunk", {}).get("title")
        if qid in b_map and b_map[qid] != gold_title:
            failures.append(f"{qid}: B_withheld='{b_map[qid]}' != C_gold='{gold_title}'")

    for r in d_records:
        if r.get("provenance", {}).get("d_type") != "bridge_failed_retrieval":
            continue
        qid = r.get("provenance", {}).get("qid")
        gold_title = r.get("provenance", {}).get("gold_chunk", {}).get("title")
        if qid in b_map and b_map[qid] != gold_title:
            failures.append(f"{qid}: B_withheld='{b_map[qid]}' != D_gold='{gold_title}'")

    return len(failures) == 0, failures

# ── Check: Refusal Precision ──────────────────────────────────────────────────
def check_refusal_precision(a: List[Dict], c: List[Dict], d: List[Dict]) -> Tuple[float, float, List[str]]:
    # False refusal: A or C's actual final answer turn (msgs[-1], the
    # Answer:/Synthesis: content) containing refusal language.
    false_refs = []
    for r in a + c:
        msgs = [m for m in r.get("messages", []) if m.get("role") == "assistant"]
        if not msgs:
            continue
        content = msgs[-1].get("content", "").lower()
        if any(p in content for p in REFUSAL_PHRASES):
            false_refs.append(r.get("provenance", {}).get("qid", "unknown"))

    # True refusal: D's actual final turn (msgs[-1]) must contain refusal
    # language, regardless of whether it's a 1-turn or 2-turn D record.
    true_count = 0
    d_failures = []
    for r in d:
        msgs = [m for m in r.get("messages", []) if m.get("role") == "assistant"]
        if msgs and any(p in msgs[-1].get("content", "").lower() for p in REFUSAL_PHRASES):
            true_count += 1
        else:
            d_failures.append(r.get("provenance", {}).get("qid", "unknown"))

    false_rate = len(false_refs) / max(1, len(a) + len(c))
    true_rate = true_count / max(1, len(d))
    return false_rate, true_rate, false_refs + d_failures

# ── Main audit loop (single pass per record) ──────────────────────────────────
print("Loading distributions ...")
dist_A = load_dist("A")
dist_B = load_dist("B")
dist_C = load_dist("C")
dist_D = load_dist("D")

total_records = len(dist_A) + len(dist_B) + len(dist_C) + len(dist_D)
print(f"A: {len(dist_A):,} | B: {len(dist_B):,} | C: {len(dist_C):,} | D: {len(dist_D):,}")

clean = {"A": [], "B": [], "C": [], "D": []}
contam = {"A": [], "B": [], "C": [], "D": []}
check_type_counts = Counter()
detailed_failures = Counter()

for name, records in [("A", dist_A), ("B", dist_B), ("C", dist_C), ("D", dist_D)]:
    for r in records:
        is_clean, reason = audit_record(r, name)
        if is_clean:
            clean[name].append(r)
        else:
            contam[name].append((r, reason))
            check_type = reason.split(":")[0]
            check_type_counts[check_type] += 1
            detailed_failures[f"{name}:{reason}"] += 1

# Cross-distribution checks
entity_ok, entity_failures = check_entity_consistency(dist_B, dist_C, dist_D)
false_ref_rate, true_ref_rate, refusal_failures = check_refusal_precision(dist_A, dist_C, dist_D)

# ── Pass rates ─────────────────────────────────────────────────────────────────
prefix_pass_rate = 100 * (1 - check_type_counts.get("prefix", 0) / max(1, total_records))
chunk_count_pass_rate = 100 * (1 - check_type_counts.get("chunk_count", 0) / max(1, total_records))
purity_pass_rate = 100 * (1 - check_type_counts.get("purity", 0) / max(1, len(dist_B)))
leak_pass_rate = 100 * (1 - check_type_counts.get("leak", 0) / max(1, total_records))

# ── Report ────────────────────────────────────────────────────────────────────
report = {
    "counts": {
        "A_total": len(dist_A), "B_total": len(dist_B), "C_total": len(dist_C), "D_total": len(dist_D),
        "A_clean": len(clean["A"]), "B_clean": len(clean["B"]), "C_clean": len(clean["C"]), "D_clean": len(clean["D"]),
        "A_contam": len(contam["A"]), "B_contam": len(contam["B"]), "C_contam": len(contam["C"]), "D_contam": len(contam["D"]),
    },
    "prefix_adherence": {
        "failures": {k: v for k, v in detailed_failures.items() if ":prefix:" in k},
        "pass_rate_pct": round(prefix_pass_rate, 2),
    },
    "chunk_count_consistency": {
        "expected_chunks": EXPECTED_CHUNK_COUNT,
        "failures": {k: v for k, v in detailed_failures.items() if ":chunk_count:" in k},
        "pass_rate_pct": round(chunk_count_pass_rate, 2),
    },
    "query_purity": {
        "failures": {k: v for k, v in detailed_failures.items() if ":purity:" in k},
        "pass_rate_pct": round(purity_pass_rate, 2),
    },
    "mode_leakage": {
        "failures": {k: v for k, v in detailed_failures.items() if ":leak:" in k},
        "pass_rate_pct": round(leak_pass_rate, 2),
    },
    "entity_consistency": {
        "passed": entity_ok,
        "failure_count": len(entity_failures),
        "failures": entity_failures[:10],
    },
    "refusal_precision": {
        "false_refusal_rate_pct": round(false_ref_rate * 100, 2),
        "true_refusal_rate_pct": round(true_ref_rate * 100, 2),
        "failure_count": len(refusal_failures),
    },
}

with open(AUDIT_REPORT_PATH, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

# ── Persist clean / contaminated splits ───────────────────────────────────────
for name in ["A", "B", "C", "D"]:
    with open(CLEAN_DIR / f"distribution_{name}.jsonl", "w", encoding="utf-8") as f:
        for r in clean[name]:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    with open(CONTAM_DIR / f"distribution_{name}.jsonl", "w", encoding="utf-8") as f:
        for r, reason in contam[name]:
            r_out = dict(r)
            r_out["_audit_failure"] = reason
            f.write(json.dumps(r_out, ensure_ascii=False) + "\n")

# ── Verdict ───────────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print("PHASE 4 AUDIT REPORT")
print(f"{'='*60}")
print(f"Prefix adherence:   {report['prefix_adherence']['pass_rate_pct']}% "
      f"({check_type_counts.get('prefix', 0)} failures)")
print(f"Chunk count == {EXPECTED_CHUNK_COUNT}:    {report['chunk_count_consistency']['pass_rate_pct']}% "
      f"({check_type_counts.get('chunk_count', 0)} failures)")
print(f"Query purity:       {report['query_purity']['pass_rate_pct']}% "
      f"({check_type_counts.get('purity', 0)} failures)")
print(f"Mode leakage:       {report['mode_leakage']['pass_rate_pct']}% "
      f"({check_type_counts.get('leak', 0)} failures)")
print(f"Entity consistency: {'PASS' if entity_ok else 'FAIL'} "
      f"({len(entity_failures)} mismatches, B->C and B->D)")
print(f"False refusal:      {report['refusal_precision']['false_refusal_rate_pct']}%")
print(f"True refusal:       {report['refusal_precision']['true_refusal_rate_pct']}%")

print(f"\nClean counts:  A={len(clean['A'])} B={len(clean['B'])} "
      f"C={len(clean['C'])} D={len(clean['D'])}")
print(f"Contam counts: A={len(contam['A'])} B={len(contam['B'])} "
      f"C={len(contam['C'])} D={len(contam['D'])}")

print(f"\n{'='*60}")
print("GO / NO-GO VERDICT")
print(f"{'='*60}")

gates = [
    (report['prefix_adherence']['pass_rate_pct'] >= 99.0, "Prefix adherence >= 99%"),
    (report['chunk_count_consistency']['pass_rate_pct'] >= 99.0, f"Chunk count == {EXPECTED_CHUNK_COUNT} >= 99%"),
    (report['query_purity']['pass_rate_pct'] >= 95.0, "Query purity >= 95%"),
    (report['mode_leakage']['pass_rate_pct'] >= 98.0, "Mode leakage <= 2%"),
    (entity_ok, "Entity consistency 100% (B->C and B->D)"),
    (report['refusal_precision']['false_refusal_rate_pct'] <= 5.0, "False refusal <= 5%"),
    (report['refusal_precision']['true_refusal_rate_pct'] >= 80.0, "True refusal >= 80%"),
]

all_pass = True
for passed, desc in gates:
    status = "✅" if passed else "❌"
    print(f"{status} {desc}")
    if not passed:
        all_pass = False

if all_pass:
    print(f"\n✅ ALL GATES PASSED. Proceed to Phase 5.")
else:
    print(f"\n❌ AUDIT FAILED. Inspect contaminated examples and fix Phase 3.")

print(f"\nReport:   {AUDIT_REPORT_PATH}")
print(f"Clean:    {CLEAN_DIR}")
print(f"Contam:   {CONTAM_DIR}")

Loading distributions ...
A: 30,012 | B: 24,482 | C: 21,220 | D: 4,981

PHASE 4 AUDIT REPORT
Prefix adherence:   100.0% (0 failures)
Chunk count == 5:    100.0% (0 failures)
Query purity:       99.94% (14 failures)
Mode leakage:       99.99% (10 failures)
Entity consistency: PASS (0 mismatches, B->C and B->D)
False refusal:      0.0%
True refusal:       100.0%

Clean counts:  A=30005 B=24468 C=21217 D=4981
Contam counts: A=7 B=14 C=3 D=0

GO / NO-GO VERDICT
✅ Prefix adherence >= 99%
✅ Chunk count == 5 >= 99%
✅ Query purity >= 95%
✅ Mode leakage <= 2%
✅ Entity consistency 100% (B->C and B->D)
✅ False refusal <= 5%
✅ True refusal >= 80%

✅ ALL GATES PASSED. Proceed to Phase 5.

Report:   /content/drive/MyDrive/hotpotqa-rag/data/audit_report.json
Clean:    /content/drive/MyDrive/hotpotqa-rag/data/distributions/clean
Contam:   /content/drive/MyDrive/hotpotqa-rag/data/distributions/contaminated


In [ ]:
import json
from pathlib import Path
from typing import Dict

DIST_DIR = Path("/content/drive/MyDrive/hotpotqa-rag/data/distributions")


def dedup_file(path: Path) -> None:
    if not path.exists():
        print(f"{path.name}: does not exist, skipping.")
        return

    seen: Dict[str, Dict] = {}
    kept = []
    dropped_qids = []
    mismatched_qids = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            record = json.loads(line)
            qid = record.get("provenance", {}).get("qid")

            if qid in seen:
                dropped_qids.append(qid)
                if record != seen[qid]:
                    mismatched_qids.append(qid)
                continue

            seen[qid] = record
            kept.append(record)

    if not dropped_qids:
        print(f"{path.name}: no duplicates found ({len(kept):,} records).")
        return

    with open(path, "w", encoding="utf-8") as f:
        for r in kept:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

    print(f"{path.name}: removed {len(dropped_qids):,} duplicate record(s) across "
          f"{len(set(dropped_qids)):,} qid(s), kept {len(kept):,}.")
    if mismatched_qids:
        print(f"  [WARN] {len(mismatched_qids)} duplicate(s) were NOT identical to the "
              f"kept copy: {mismatched_qids}.")
    else:
        print(f"  All duplicates were identical to the kept copy, as expected.")


for name in ["A", "B", "C", "D"]:
    dedup_file(DIST_DIR / f"distribution_{name}.jsonl")

distribution_A.jsonl: removed 12 duplicate record(s) across 12 qid(s), kept 30,000.
  All duplicates were identical to the kept copy, as expected.
distribution_B.jsonl: removed 12 duplicate record(s) across 12 qid(s), kept 24,470.
  [WARN] 3 duplicate(s) were NOT identical to the kept copy: ['04437f2be7f0e27abf9ba27679ff49fe', '39d3d52462044f1ad9f23238382c5e05', 'cc1be54836c9a5b0e596f61cebe3135e'].
distribution_C.jsonl: no duplicates found (21,220 records).
distribution_D.jsonl: removed 101 duplicate record(s) across 101 qid(s), kept 4,880.
  [WARN] 34 duplicate(s) were NOT identical to the kept copy: ['8106432e34b437f3d377a94f276b7cae', '3802d6075a972a69cf081c118ee84fa3', 'a4e5f3352e6a1875d1246c6afd99ade8', '8ae5fab53a040a321eeec73489ecbab8', '819678e3bbf673cdcb8036867cf61a42', '083dededa27a2d717a0c692c9e01a6a1', 'ad38cabc34901479413aa55bbd1856b7', 'e1bdbaa5a939601ea32dfd4c6ef24d20', 'c45e8d5c09171a5db512929f375145e5', '89250a14db97a48ca584f504bc353db3', '0511de461bdf14ed3fa376f3c5128

In [ ]:
# Cell 13: Phase 4 Audit
# Structural checks (prefix, chunk count), content checks (query purity,
# mode leakage), duplicate-qid detection, and cross-distribution checks
# (entity consistency, refusal precision). Loads distribution JSONL
# files, validates every record in a single pass, splits clean vs.
# contaminated, and reports.

import json
import re
from pathlib import Path
from typing import Dict, List, Tuple
from collections import Counter

# ── Configuration ─────────────────────────────────────────────────────────────
DIST_DIR = Path("/content/drive/MyDrive/hotpotqa-rag/data/distributions")
AUDIT_REPORT_PATH = Path("/content/drive/MyDrive/hotpotqa-rag/data/audit_report.json")
CLEAN_DIR = DIST_DIR / "clean"
CONTAM_DIR = DIST_DIR / "contaminated"

CLEAN_DIR.mkdir(parents=True, exist_ok=True)
CONTAM_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_CHUNK_COUNT = 5

QUERY_PROSE_PATTERNS = [
    r"\bi need\b", r"\bi want\b", r"\bplease\b", r"\bfind information\b",
    r"\bsearch for\b", r"\blook for\b", r"\baccording to\b", r"\bi think\b",
    r"\bi believe\b", r"\bcan you\b", r"\bcould you\b", r"\bhelp me\b",
    r"\btell me\b", r"\bgive me\b", r"\bwhat is\b", r"\bwhere is\b",
]

ANSWER_INDICATORS = [r"\bis\s+[A-Z]", r"\bare\s+[A-Z]", r"\bwas\s+[A-Z]", r"\bwere\s+[A-Z]"]

REFUSAL_PHRASES = ["cannot determine", "i cannot", "not enough information", "unable to answer"]


def load_dist(name: str) -> List[Dict]:
    path = DIST_DIR / f"distribution_{name}.jsonl"
    if not path.exists():
        print(f"[WARN] {path} does not exist yet, treating Distribution {name} as empty.")
        return []
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


def count_chunks_in_first_turn(record: Dict) -> int:
    messages = record.get("messages", [])
    if not messages:
        return -1
    first_turn = messages[0].get("content", "")
    return len(re.findall(r'^\[\d+\]', first_turn, re.MULTILINE))


def check_chunk_count(record: Dict, expected: int = EXPECTED_CHUNK_COUNT) -> Tuple[bool, str]:
    count = count_chunks_in_first_turn(record)
    if count != expected:
        return False, f"expected_{expected}_got_{count}"
    return True, "ok"


def check_prefix(record: Dict, dist: str) -> Tuple[bool, str]:
    msgs = [m for m in record.get("messages", []) if m.get("role") == "assistant"]

    if dist == "A":
        if not msgs or not msgs[0].get("content", "").startswith("Answer:"):
            return False, "missing_Answer_prefix"

    elif dist == "B":
        if not msgs or not msgs[0].get("content", "").startswith("Search:"):
            return False, "missing_Search_prefix"

    elif dist == "C":
        if len(msgs) < 2:
            return False, "missing_second_assistant_turn"
        if not msgs[0].get("content", "").startswith("Search:"):
            return False, "first_turn_missing_Search"
        if not msgs[1].get("content", "").startswith("Synthesis:"):
            return False, "second_turn_missing_Synthesis"

    elif dist == "D":
        d_type = record.get("provenance", {}).get("d_type")
        if d_type == "comparison_no_gold":
            if not msgs or not msgs[0].get("content", "").startswith("Answer:"):
                return False, "missing_Answer_prefix"
        elif d_type == "bridge_failed_retrieval":
            if len(msgs) < 2:
                return False, "missing_second_assistant_turn"
            if not msgs[0].get("content", "").startswith("Search:"):
                return False, "first_turn_missing_Search"
            if not msgs[-1].get("content", "").startswith("Answer:"):
                return False, "last_turn_missing_Answer"
        else:
            return False, f"unknown_d_type:{d_type}"

    return True, "ok"


def check_query_purity(record: Dict) -> Tuple[bool, str]:
    msgs = [m for m in record.get("messages", []) if m.get("role") == "assistant"]
    if not msgs:
        return False, "no_assistant_turn"
    content = msgs[0].get("content", "")
    if not content.startswith("Search:"):
        return False, "not_a_search_turn"
    query = content[len("Search:"):].strip()

    for pat in QUERY_PROSE_PATTERNS:
        if re.search(pat, query, re.IGNORECASE):
            return False, f"prose_detected:{pat}"

    words = query.split()
    if len(words) > 12:
        return False, f"too_long_{len(words)}_words"

    prov = record.get("provenance", {})
    withheld = prov.get("withheld_title", "")
    if withheld and withheld.lower() not in query.lower():
        return False, "withheld_title_absent"

    return True, "ok"


def check_mode_leakage(record: Dict, dist: str) -> Tuple[bool, str]:
    msgs = [m for m in record.get("messages", []) if m.get("role") == "assistant"]
    if not msgs:
        return False, "no_assistant_turn"

    if dist == "A":
        text = msgs[0].get("content", "")
        if "Search:" in text or "Synthesis:" in text:
            return False, "foreign_prefix_in_answer"
        if text.count("?") > 0:
            return False, "question_mark_in_answer"

    elif dist == "B":
        text = msgs[0].get("content", "")
        query = text[len("Search:"):].strip() if text.startswith("Search:") else text
        for pat in ANSWER_INDICATORS:
            if re.search(pat, query):
                return False, f"answer_language_in_query:{pat}"

    elif dist == "C":
        if len(msgs) >= 2:
            synth = msgs[1].get("content", "")
            body = synth[len("Synthesis:"):].strip() if synth.startswith("Synthesis:") else synth
            if "Search:" in body:
                return False, "search_prefix_inside_synthesis"
            if re.search(r"\b(what|where|when|who|why|how)\b.*\?", body.lower()):
                return False, "question_in_synthesis"

    elif dist == "D":
        refusal_text = msgs[-1].get("content", "")
        if re.search(r"\bis\s+[A-Z][a-zA-Z\s]+[.!?]", refusal_text):
            return False, "factual_assertion_in_refusal"

        d_type = record.get("provenance", {}).get("d_type")
        if d_type == "bridge_failed_retrieval":
            search_text = msgs[0].get("content", "")
            body = search_text[len("Search:"):].strip() if search_text.startswith("Search:") else search_text
            for pat in ANSWER_INDICATORS:
                if re.search(pat, body):
                    return False, f"answer_language_in_search_turn:{pat}"

    return True, "ok"


def audit_record(record: Dict, dist: str) -> Tuple[bool, str]:
    ok, reason = check_prefix(record, dist)
    if not ok:
        return False, f"prefix:{reason}"

    ok, reason = check_chunk_count(record)
    if not ok:
        return False, f"chunk_count:{reason}"

    if dist == "B":
        ok, reason = check_query_purity(record)
        if not ok:
            return False, f"purity:{reason}"

    ok, reason = check_mode_leakage(record, dist)
    if not ok:
        return False, f"leak:{reason}"

    return True, "ok"


def check_entity_consistency(
    b_records: List[Dict], c_records: List[Dict], d_records: List[Dict]
) -> Tuple[bool, List[str]]:
    b_map = {}
    for r in b_records:
        qid = r.get("provenance", {}).get("qid")
        withheld = r.get("provenance", {}).get("withheld_title")
        if qid and withheld:
            b_map[qid] = withheld

    failures = []

    for r in c_records:
        qid = r.get("provenance", {}).get("qid")
        gold_title = r.get("provenance", {}).get("gold_chunk", {}).get("title")
        if qid in b_map and b_map[qid] != gold_title:
            failures.append(f"{qid}: B_withheld='{b_map[qid]}' != C_gold='{gold_title}'")

    for r in d_records:
        if r.get("provenance", {}).get("d_type") != "bridge_failed_retrieval":
            continue
        qid = r.get("provenance", {}).get("qid")
        gold_title = r.get("provenance", {}).get("gold_chunk", {}).get("title")
        if qid in b_map and b_map[qid] != gold_title:
            failures.append(f"{qid}: B_withheld='{b_map[qid]}' != D_gold='{gold_title}'")

    return len(failures) == 0, failures


def check_refusal_precision(a: List[Dict], c: List[Dict], d: List[Dict]) -> Tuple[float, float, List[str]]:
    false_refs = []
    for r in a + c:
        msgs = [m for m in r.get("messages", []) if m.get("role") == "assistant"]
        if not msgs:
            continue
        content = msgs[-1].get("content", "").lower()
        if any(p in content for p in REFUSAL_PHRASES):
            false_refs.append(r.get("provenance", {}).get("qid", "unknown"))

    true_count = 0
    d_failures = []
    for r in d:
        msgs = [m for m in r.get("messages", []) if m.get("role") == "assistant"]
        if msgs and any(p in msgs[-1].get("content", "").lower() for p in REFUSAL_PHRASES):
            true_count += 1
        else:
            d_failures.append(r.get("provenance", {}).get("qid", "unknown"))

    false_rate = len(false_refs) / max(1, len(a) + len(c))
    true_rate = true_count / max(1, len(d))
    return false_rate, true_rate, false_refs + d_failures


print("Loading distributions ...")
dist_A = load_dist("A")
dist_B = load_dist("B")
dist_C = load_dist("C")
dist_D = load_dist("D")

total_records = len(dist_A) + len(dist_B) + len(dist_C) + len(dist_D)
print(f"A: {len(dist_A):,} | B: {len(dist_B):,} | C: {len(dist_C):,} | D: {len(dist_D):,}")

clean = {"A": [], "B": [], "C": [], "D": []}
contam = {"A": [], "B": [], "C": [], "D": []}
check_type_counts = Counter()
detailed_failures = Counter()

for name, records in [("A", dist_A), ("B", dist_B), ("C", dist_C), ("D", dist_D)]:
    seen_qids = set()
    for r in records:
        qid = r.get("provenance", {}).get("qid")

        if qid in seen_qids:
            contam[name].append((r, "duplicate_qid"))
            check_type_counts["duplicate_qid"] += 1
            detailed_failures[f"{name}:duplicate_qid"] += 1
            continue
        seen_qids.add(qid)

        is_clean, reason = audit_record(r, name)
        if is_clean:
            clean[name].append(r)
        else:
            contam[name].append((r, reason))
            check_type = reason.split(":")[0]
            check_type_counts[check_type] += 1
            detailed_failures[f"{name}:{reason}"] += 1

entity_ok, entity_failures = check_entity_consistency(clean["B"], clean["C"], clean["D"])
false_ref_rate, true_ref_rate, refusal_failures = check_refusal_precision(clean["A"], clean["C"], clean["D"])

duplicate_count = check_type_counts.get("duplicate_qid", 0)
prefix_pass_rate = 100 * (1 - check_type_counts.get("prefix", 0) / max(1, total_records))
chunk_count_pass_rate = 100 * (1 - check_type_counts.get("chunk_count", 0) / max(1, total_records))
purity_pass_rate = 100 * (1 - check_type_counts.get("purity", 0) / max(1, len(dist_B)))
leak_pass_rate = 100 * (1 - check_type_counts.get("leak", 0) / max(1, total_records))

report = {
    "counts": {
        "A_total": len(dist_A), "B_total": len(dist_B), "C_total": len(dist_C), "D_total": len(dist_D),
        "A_clean": len(clean["A"]), "B_clean": len(clean["B"]), "C_clean": len(clean["C"]), "D_clean": len(clean["D"]),
        "A_contam": len(contam["A"]), "B_contam": len(contam["B"]), "C_contam": len(contam["C"]), "D_contam": len(contam["D"]),
    },
    "duplicate_qids": {
        "count": duplicate_count,
    },
    "prefix_adherence": {
        "failures": {k: v for k, v in detailed_failures.items() if ":prefix:" in k},
        "pass_rate_pct": round(prefix_pass_rate, 2),
    },
    "chunk_count_consistency": {
        "expected_chunks": EXPECTED_CHUNK_COUNT,
        "failures": {k: v for k, v in detailed_failures.items() if ":chunk_count:" in k},
        "pass_rate_pct": round(chunk_count_pass_rate, 2),
    },
    "query_purity": {
        "failures": {k: v for k, v in detailed_failures.items() if ":purity:" in k},
        "pass_rate_pct": round(purity_pass_rate, 2),
    },
    "mode_leakage": {
        "failures": {k: v for k, v in detailed_failures.items() if ":leak:" in k},
        "pass_rate_pct": round(leak_pass_rate, 2),
    },
    "entity_consistency": {
        "passed": entity_ok,
        "failure_count": len(entity_failures),
        "failures": entity_failures[:10],
    },
    "refusal_precision": {
        "false_refusal_rate_pct": round(false_ref_rate * 100, 2),
        "true_refusal_rate_pct": round(true_ref_rate * 100, 2),
        "failure_count": len(refusal_failures),
    },
}

with open(AUDIT_REPORT_PATH, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

for name in ["A", "B", "C", "D"]:
    with open(CLEAN_DIR / f"distribution_{name}.jsonl", "w", encoding="utf-8") as f:
        for r in clean[name]:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    with open(CONTAM_DIR / f"distribution_{name}.jsonl", "w", encoding="utf-8") as f:
        for r, reason in contam[name]:
            r_out = dict(r)
            r_out["_audit_failure"] = reason
            f.write(json.dumps(r_out, ensure_ascii=False) + "\n")

print(f"\n{'='*60}")
print("PHASE 4 AUDIT REPORT")
print(f"{'='*60}")
print(f"Duplicate qids:     {duplicate_count} found")
print(f"Prefix adherence:   {report['prefix_adherence']['pass_rate_pct']}% "
      f"({check_type_counts.get('prefix', 0)} failures)")
print(f"Chunk count == {EXPECTED_CHUNK_COUNT}:    {report['chunk_count_consistency']['pass_rate_pct']}% "
      f"({check_type_counts.get('chunk_count', 0)} failures)")
print(f"Query purity:       {report['query_purity']['pass_rate_pct']}% "
      f"({check_type_counts.get('purity', 0)} failures)")
print(f"Mode leakage:       {report['mode_leakage']['pass_rate_pct']}% "
      f"({check_type_counts.get('leak', 0)} failures)")
print(f"Entity consistency: {'PASS' if entity_ok else 'FAIL'} "
      f"({len(entity_failures)} mismatches, B->C and B->D, checked on the clean set)")
print(f"False refusal:      {report['refusal_precision']['false_refusal_rate_pct']}%")
print(f"True refusal:       {report['refusal_precision']['true_refusal_rate_pct']}%")

print(f"\nClean counts:  A={len(clean['A'])} B={len(clean['B'])} "
      f"C={len(clean['C'])} D={len(clean['D'])}")
print(f"Contam counts: A={len(contam['A'])} B={len(contam['B'])} "
      f"C={len(contam['C'])} D={len(contam['D'])}")

print(f"\n{'='*60}")
print("GO / NO-GO VERDICT")
print(f"{'='*60}")

gates = [
    (duplicate_count == 0, "No duplicate qids (0 found)"),
    (report['prefix_adherence']['pass_rate_pct'] >= 99.0, "Prefix adherence >= 99%"),
    (report['chunk_count_consistency']['pass_rate_pct'] >= 99.0, f"Chunk count == {EXPECTED_CHUNK_COUNT} >= 99%"),
    (report['query_purity']['pass_rate_pct'] >= 95.0, "Query purity >= 95%"),
    (report['mode_leakage']['pass_rate_pct'] >= 98.0, "Mode leakage <= 2%"),
    (entity_ok, "Entity consistency 100% (B->C and B->D, clean set)"),
    (report['refusal_precision']['false_refusal_rate_pct'] <= 5.0, "False refusal <= 5%"),
    (report['refusal_precision']['true_refusal_rate_pct'] >= 80.0, "True refusal >= 80%"),
]

all_pass = True
for passed, desc in gates:
    status = "✅" if passed else "❌"
    print(f"{status} {desc}")
    if not passed:
        all_pass = False

if all_pass:
    print(f"\n✅ ALL GATES PASSED. Proceed to Phase 5.")
else:
    print(f"\n❌ AUDIT FAILED. Inspect contaminated examples and fix Phase 3.")

print(f"\nReport:   {AUDIT_REPORT_PATH}")
print(f"Clean:    {CLEAN_DIR}")
print(f"Contam:   {CONTAM_DIR}")

Loading distributions ...
A: 30,000 | B: 24,470 | C: 21,220 | D: 4,880

PHASE 4 AUDIT REPORT
Duplicate qids:     0 found
Prefix adherence:   100.0% (0 failures)
Chunk count == 5:    100.0% (0 failures)
Query purity:       99.94% (14 failures)
Mode leakage:       99.99% (10 failures)
Entity consistency: PASS (0 mismatches, B->C and B->D, checked on the clean set)
False refusal:      0.0%
True refusal:       100.0%

Clean counts:  A=29993 B=24456 C=21217 D=4880
Contam counts: A=7 B=14 C=3 D=0

GO / NO-GO VERDICT
✅ No duplicate qids (0 found)
✅ Prefix adherence >= 99%
✅ Chunk count == 5 >= 99%
✅ Query purity >= 95%
✅ Mode leakage <= 2%
✅ Entity consistency 100% (B->C and B->D, clean set)
✅ False refusal <= 5%
✅ True refusal >= 80%

✅ ALL GATES PASSED. Proceed to Phase 5.

Report:   /content/drive/MyDrive/hotpotqa-rag/data/audit_report.json
Clean:    /content/drive/MyDrive/hotpotqa-rag/data/distributions/clean
Contam:   /content/drive/MyDrive/hotpotqa-rag/data/distributions/contaminated


In [ ]:
import json
from pathlib import Path
from typing import Dict

DIST_DIR = Path("/content/drive/MyDrive/hotpotqa-rag/data/distributions")
REMOVED_DIR = DIST_DIR / "removed_duplicates"
REMOVED_DIR.mkdir(parents=True, exist_ok=True)


def dedup_file(path: Path) -> None:
    if not path.exists():
        print(f"{path.name}: does not exist, skipping.")
        return

    seen: Dict[str, Dict] = {}
    kept = []
    removed = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            record = json.loads(line)
            qid = record.get("provenance", {}).get("qid")

            if qid is None:
                print(f"  [WARN] record with no/None qid found in {path.name}, "
                      f"keeping it as-is, not deduping it against anything.")
                kept.append(record)
                continue

            if qid in seen:
                removed.append({"kept_copy": seen[qid], "discarded_copy": record})
                continue

            seen[qid] = record
            kept.append(record)

    if not removed:
        print(f"{path.name}: no duplicates found ({len(kept):,} records).")
        return

    with open(path, "w", encoding="utf-8") as f:
        for r in kept:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

    removed_path = REMOVED_DIR / path.name
    with open(removed_path, "w", encoding="utf-8") as f:
        for pair in removed:
            f.write(json.dumps(pair, ensure_ascii=False) + "\n")

    print(f"{path.name}: removed {len(removed):,} duplicate(s), kept {len(kept):,}. "
          f"Both copies of each saved to {removed_path} for inspection.")


for name in ["A", "B", "C", "D"]:
    dedup_file(DIST_DIR / f"distribution_{name}.jsonl")

distribution_A.jsonl: no duplicates found (30,000 records).
distribution_B.jsonl: no duplicates found (24,470 records).
distribution_C.jsonl: no duplicates found (21,220 records).
distribution_D.jsonl: no duplicates found (4,880 records).


In [ ]:
import json
from pathlib import Path
from typing import Dict

DIST_DIR = Path("/content/drive/MyDrive/hotpotqa-rag/data/distributions")
REMOVED_DIR = DIST_DIR / "removed_duplicates"
REMOVED_DIR.mkdir(parents=True, exist_ok=True)


def dedup_file(path: Path) -> None:
    if not path.exists():
        print(f"{path.name}: does not exist, skipping.")
        return

    seen: Dict[str, Dict] = {}
    kept = []
    removed = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            record = json.loads(line)
            qid = record.get("provenance", {}).get("qid")

            if qid is None:
                print(f"  [WARN] record with no/None qid found in {path.name}, "
                      f"keeping it as-is, not deduping it against anything.")
                kept.append(record)
                continue

            if qid in seen:
                removed.append({"kept_copy": seen[qid], "discarded_copy": record})
                continue

            seen[qid] = record
            kept.append(record)

    if not removed:
        print(f"{path.name}: no duplicates found ({len(kept):,} records).")
        return

    with open(path, "w", encoding="utf-8") as f:
        for r in kept:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

    removed_path = REMOVED_DIR / path.name
    with open(removed_path, "w", encoding="utf-8") as f:
        for pair in removed:
            f.write(json.dumps(pair, ensure_ascii=False) + "\n")

    print(f"{path.name}: removed {len(removed):,} duplicate(s), kept {len(kept):,}. "
          f"Both copies of each saved to {removed_path} for inspection.")


for name in ["A", "B", "C", "D"]:
    dedup_file(DIST_DIR / f"distribution_{name}.jsonl")

distribution_A.jsonl: no duplicates found (30,000 records).
distribution_B.jsonl: no duplicates found (24,470 records).
distribution_C.jsonl: no duplicates found (21,220 records).
distribution_D.jsonl: no duplicates found (4,880 records).


In [ ]:
import json
from pathlib import Path

TRAIN_SUBSAMPLE_PATH = Path("/content/drive/MyDrive/hotpotqa-rag/data/train_subsample.json")
DIST_DIR = Path("/content/drive/MyDrive/hotpotqa-rag/data/distributions")

with open(TRAIN_SUBSAMPLE_PATH, "r", encoding="utf-8") as f:
    valid_qids = set(json.load(f).keys())

print(f"Valid qids in train_subsample: {len(valid_qids):,}")

for name in ["A", "B", "C", "D"]:
    path = DIST_DIR / f"distribution_{name}.jsonl"
    seen_in_file = set()
    unknown = set()
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rec = json.loads(line)
            qid = rec.get("provenance", {}).get("qid")
            seen_in_file.add(qid)
            if qid not in valid_qids:
                unknown.add(qid)
    print(f"{name}: {len(seen_in_file):,} distinct qids, {len(unknown)} not found in train_subsample.json")
    if unknown:
        print(f"  Examples: {list(unknown)[:10]}")

Valid qids in train_subsample: 30,000
A: 30,000 distinct qids, 0 not found in train_subsample.json
B: 24,470 distinct qids, 0 not found in train_subsample.json
C: 21,220 distinct qids, 0 not found in train_subsample.json
D: 4,880 distinct qids, 0 not found in train_subsample.json


In [ ]:
b_qids = set()
with open(DIST_DIR / "distribution_B.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            b_qids.add(json.loads(line)["provenance"]["qid"])

d_bridge_qids = set()
with open(DIST_DIR / "distribution_D.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        rec = json.loads(line)
        if rec["provenance"].get("d_type") == "bridge_failed_retrieval":
            d_bridge_qids.add(rec["provenance"]["qid"])

orphaned = d_bridge_qids - b_qids
print(f"D bridge_failed_retrieval qids with no matching B record: {len(orphaned)}")
if orphaned:
    print(f"  Examples: {list(orphaned)[:10]}")

D bridge_failed_retrieval qids with no matching B record: 0


Fine Tuning Qwen2.5 1.5B Instruct model using Unsloth

In [ ]:
import sys, torch
print("Python:", sys.version)
print("Torch:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())


Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
Torch: 2.11.0+cu128
CUDA Available: True


In [ ]:
!pip install -q uv
!uv pip install --system -q unsloth

# Remove xformers immediately, before anything imports it. Its backward
# kernel for this model's attention layout has no working path on a T4,
# confirmed four times now across different torch/transformers builds
# with the identical error. It never gets installed-then-used again.
!pip uninstall -y -q xformers

import importlib.util
assert importlib.util.find_spec("xformers") is None, (
    "xformers is still present after uninstall. Stop here and paste this output."
)
print("Environment ready: unsloth installed, xformers removed, nothing imported yet.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.8/17.8 MB 77.9 MB/s eta 0:00:00
Environment ready: unsloth installed, xformers removed, nothing imported yet.


Delete the trained adapters and checkpoints

In [ ]:
import shutil
from pathlib import Path

if not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")

OUT_DIR = Path("/content/drive/MyDrive/hotpotqa-rag/finetune")

targets = [
    OUT_DIR / "checkpoints",
    OUT_DIR / "checkpoints_epoch2",
    OUT_DIR / "final_adapter",
    OUT_DIR / "final_adapter_epoch2",
    OUT_DIR / "monitor_log.jsonl",
    OUT_DIR / "holdout_records.jsonl",
    OUT_DIR / "run_config.json",
    OUT_DIR / "run_config_epoch2.json",
]
for p in targets:
    if p.is_dir():
        shutil.rmtree(p)
        print(f"Removed directory: {p}")
    elif p.exists():
        p.unlink()
        print(f"Removed file: {p}")
    else:
        print(f"Already absent: {p}")

Already absent: /content/drive/MyDrive/hotpotqa-rag/finetune/checkpoints
Already absent: /content/drive/MyDrive/hotpotqa-rag/finetune/checkpoints_epoch2
Already absent: /content/drive/MyDrive/hotpotqa-rag/finetune/final_adapter
Already absent: /content/drive/MyDrive/hotpotqa-rag/finetune/final_adapter_epoch2
Already absent: /content/drive/MyDrive/hotpotqa-rag/finetune/monitor_log.jsonl
Already absent: /content/drive/MyDrive/hotpotqa-rag/finetune/holdout_records.jsonl
Already absent: /content/drive/MyDrive/hotpotqa-rag/finetune/run_config.json
Already absent: /content/drive/MyDrive/hotpotqa-rag/finetune/run_config_epoch2.json


In [ ]:
# Cell 15: Unsloth LoRA fine-tune of Qwen2.5-1.5B-Instruct on the clean distributions (Colab T4)

import importlib.util
assert importlib.util.find_spec("xformers") is None, (
    "xformers is present. Run Cell 14 first, in this same session, before this cell."
)

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from unsloth import FastLanguageModel, is_bfloat16_supported

import dataclasses
import gc
import json
import math
import random
from collections import Counter, defaultdict
from pathlib import Path

import torch
from torch.utils.data import Dataset
from transformers import (
    DataCollatorForSeq2Seq,
    Trainer,
    TrainerCallback,
    TrainingArguments,
)

torch.backends.cuda.enable_mem_efficient_sdp(True)

for _name in ("trainer", "model"):
    if _name in globals():
        del globals()[_name]
gc.collect()
torch.cuda.empty_cache()

assert torch.cuda.is_available(), "No GPU detected. Runtime > Change runtime type > T4 GPU."

if not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
BASE_DIR = Path("/content/drive/MyDrive/hotpotqa-rag")
CLEAN_DIR = BASE_DIR / "data" / "distributions" / "clean"
OUT_DIR = BASE_DIR / "finetune"
CKPT_DIR = OUT_DIR / "checkpoints"
FINAL_DIR = OUT_DIR / "final_adapter"
MONITOR_LOG = OUT_DIR / "monitor_log.jsonl"
HOLDOUT_PATH = OUT_DIR / "holdout_records.jsonl"
for d in (OUT_DIR, CKPT_DIR):
    d.mkdir(parents=True, exist_ok=True)

BASE_MODEL = "unsloth/Qwen2.5-1.5B-Instruct"
MAX_SEQ_LEN = 2048
SEED = 3407

N_HOLDOUT_QIDS = 400
TRAIN_COUNTS = {"A": 4000, "B": 0, "C": 4000, "Dh": 1000, "Dc": 500}
VAL_COUNTS = {"A": 80, "C": 90, "Dh": 40, "Dc": 20}

LORA_R = 16
LEARNING_RATE = 1e-4
GRAD_ACCUM = 16

AMP_DTYPE = torch.bfloat16 if is_bfloat16_supported() else torch.float16

# ---------------------------------------------------------------------------
# Load model and tokenizer
# ---------------------------------------------------------------------------
print(f"Loading {BASE_MODEL} ...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
)
# Check the banner just printed above: it must say
# "FA [Xformers = None. FA2 = False]".

IM_END_ID = tokenizer.convert_tokens_to_ids("<|im_end|>")
ANSWER_PREFIX_IDS = tokenizer.encode("Answer:", add_special_tokens=False)
ANSWER_TOK = ANSWER_PREFIX_IDS[0]
ANSWER_PREFIX_LEN = len(ANSWER_PREFIX_IDS)
REFUSAL_TOK = tokenizer.encode(
    "Answer: I cannot determine the answer from the provided context.",
    add_special_tokens=False,
)[ANSWER_PREFIX_LEN]
assert tokenizer.decode([REFUSAL_TOK]).strip() == "I", "Unexpected tokenization of the refusal sentence."

# ---------------------------------------------------------------------------
# Tokenization with explicit assistant-only labels
# ---------------------------------------------------------------------------
ASSIST_MARK = "<|im_start|>assistant\n"
END_MARK = "<|im_end|>"


def encode_example(messages):
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    enc = tokenizer(text, add_special_tokens=False, return_offsets_mapping=True)
    ids = enc["input_ids"]
    offsets = enc["offset_mapping"]
    if len(ids) > MAX_SEQ_LEN:
        return None

    spans = []
    pos = 0
    while True:
        s = text.find(ASSIST_MARK, pos)
        if s == -1:
            break
        content_start = s + len(ASSIST_MARK)
        e = text.find(END_MARK, content_start)
        if e == -1:
            raise RuntimeError("Assistant turn without <|im_end|>, malformed example.")
        spans.append((content_start, e + len(END_MARK)))
        pos = e + len(END_MARK)
    if not spans:
        raise RuntimeError("No assistant turn found in an example.")

    labels = [-100] * len(ids)
    span_starts = []
    si = 0
    for i, (a, b) in enumerate(offsets):
        while si < len(spans) and a >= spans[si][1]:
            si += 1
        if si == len(spans):
            break
        cs, ce = spans[si]
        if a >= cs and b <= ce and b > a:
            labels[i] = ids[i]
            if len(span_starts) == si:
                span_starts.append(i)

    n_end = sum(1 for l in labels if l == IM_END_ID)
    if len(span_starts) != len(spans) or n_end != len(spans):
        raise RuntimeError(
            "Label masking check failed: the model would not learn to stop, or a turn got no labels."
        )
    return {"input_ids": ids, "labels": labels, "span_starts": span_starts}


# ---------------------------------------------------------------------------
# Load clean distributions, hold out whole questions, sample the mix
# ---------------------------------------------------------------------------
def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


print("Loading clean distributions ...")
raw = {name: load_jsonl(CLEAN_DIR / f"distribution_{name}.jsonl") for name in "ABCD"}
pools = {
    "A": raw["A"],
    "B": raw["B"],
    "C": raw["C"],
    "Dh": [r for r in raw["D"] if r["provenance"].get("d_type") == "bridge_failed_retrieval"],
    "Dc": [r for r in raw["D"] if r["provenance"].get("d_type") == "comparison_no_gold"],
}
print({k: len(v) for k, v in pools.items()})


def first_turn_key(rec):
    m = rec["messages"]
    return (hash(m[0]["content"]), hash(m[1]["content"]))


hop_keys = {first_turn_key(r) for r in pools["C"] + pools["Dh"]}
b_dup = sum(1 for r in pools["B"] if first_turn_key(r) in hop_keys)
print(f"B records identical to the first turn of a C/Dh record: {b_dup:,} / {len(pools['B']):,}")

rng = random.Random(SEED)
all_qids = sorted({r["provenance"]["qid"] for r in pools["A"]})
holdout_qids = set(rng.sample(all_qids, N_HOLDOUT_QIDS))

with open(HOLDOUT_PATH, "w", encoding="utf-8") as f:
    for tag, pool in pools.items():
        for r in pool:
            if r["provenance"]["qid"] in holdout_qids:
                f.write(json.dumps(
                    {"tag": tag, "messages": r["messages"], "provenance": r["provenance"]},
                    ensure_ascii=False,
                ) + "\n")

train_tagged = []
for tag, n in TRAIN_COUNTS.items():
    pool = [r for r in pools[tag] if r["provenance"]["qid"] not in holdout_qids]
    chosen = rng.sample(pool, min(n, len(pool)))
    train_tagged.extend((tag, r) for r in chosen)

val_tagged = []
for tag, n in VAL_COUNTS.items():
    pool = [r for r in pools[tag] if r["provenance"]["qid"] in holdout_qids]
    chosen = rng.sample(pool, min(n, len(pool)))
    val_tagged.extend((tag, r) for r in chosen)

del raw, pools


def build_rows(tagged, title):
    rows, meta, dropped = [], [], Counter()
    for tag, rec in tagged:
        enc = encode_example(rec["messages"])
        if enc is None:
            dropped[tag] += 1
            continue
        rows.append({
            "input_ids": enc["input_ids"],
            "attention_mask": [1] * len(enc["input_ids"]),
            "labels": enc["labels"],
        })
        meta.append({"tag": tag, "input_ids": enc["input_ids"], "span_starts": enc["span_starts"]})

    by_tag = defaultdict(list)
    for row, m in zip(rows, meta):
        by_tag[m["tag"]].append((len(row["input_ids"]), sum(1 for l in row["labels"] if l != -100)))
    total_tokens = sum(len(r["input_ids"]) for r in rows)
    print(f"\n{title}: {len(rows):,} examples, {total_tokens:,} tokens, "
          f"dropped for length > {MAX_SEQ_LEN}: {dict(dropped) if dropped else 'none'}")
    for tag in sorted(by_tag):
        lens = [x[0] for x in by_tag[tag]]
        sup = [x[1] for x in by_tag[tag]]
        print(f"  {tag:>3}: n={len(lens):>5,}  mean_tokens={sum(lens)/len(lens):>7.0f}  "
              f"max={max(lens):>5}  mean_supervised={sum(sup)/len(sup):>5.1f}")
    return rows, meta


train_rows, train_meta = build_rows(train_tagged, "TRAIN")
val_rows, val_meta = build_rows(val_tagged, "VAL")

demo = next((i for i, m in enumerate(train_meta) if m["tag"] == "C"), None)
if demo is not None:
    sup_ids = [t for t, l in zip(train_rows[demo]["input_ids"], train_rows[demo]["labels"]) if l != -100]
    print("\nSupervised text of one C example (everything else is masked):")
    print(tokenizer.decode(sup_ids)[:400])


class ListDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        return self.rows[i]


train_ds = ListDataset(train_rows)
val_ds = ListDataset(val_rows)

# ---------------------------------------------------------------------------
# LoRA
# ---------------------------------------------------------------------------
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=LORA_R,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    use_rslora=False,
    loftq_config=None,
)
model.print_trainable_parameters()


# ---------------------------------------------------------------------------
# Decision monitor: teacher-forced check of the tokens where "mode" is decided
# ---------------------------------------------------------------------------
class MonitorAndSave(TrainerCallback):
    def __init__(self, examples):
        self.examples = examples

    def on_save(self, args, state, control, **kwargs):
        ckpt = Path(args.output_dir) / f"checkpoint-{state.global_step}"
        if ckpt.exists():
            tokenizer.save_pretrained(str(ckpt))

    def on_evaluate(self, args, state, control, model=None, **kwargs):
        if model is None:
            return
        stats = defaultdict(lambda: [0, 0])
        device = next(model.parameters()).device
        for ex in self.examples:
            ids = ex["input_ids"]
            x = torch.tensor([ids], device=device)
            with torch.no_grad(), torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
                logits = model(input_ids=x, use_cache=False).logits[0]
            pred = logits.argmax(-1).tolist()
            del logits
            for k, s in enumerate(ex["span_starts"]):
                key = f"{ex['tag']}:t{k + 1}"
                stats[key][0] += int(pred[s - 1] == ids[s])
                stats[key][1] += 1
                if ids[s] == ANSWER_TOK:
                    r = s + ANSWER_PREFIX_LEN
                    if r < len(ids):
                        true_ref = ids[r] == REFUSAL_TOK
                        pred_ref = pred[r - 1] == REFUSAL_TOK
                        rkey = key + ":refuse"
                        stats[rkey][0] += int(true_ref == pred_ref)
                        stats[rkey][1] += 1

        rates = {k: v[0] / v[1] for k, v in stats.items() if v[1] > 0}
        worst = min(rates.values()) if rates else 0.0
        status = "OK" if worst >= 0.95 else ("WATCH" if worst >= 0.85 else "RED")
        line = " | ".join(f"{k} {rates[k] * 100:.0f}%" for k in sorted(rates))
        print(f"\n[decision monitor] step {state.global_step} | {line} | worst {worst * 100:.0f}% -> {status}")
        with open(MONITOR_LOG, "a", encoding="utf-8") as f:
            f.write(json.dumps({"step": state.global_step, "status": status, "rates": rates}) + "\n")
        torch.cuda.empty_cache()


# ---------------------------------------------------------------------------
# Trainer
# ---------------------------------------------------------------------------
steps_per_epoch = math.ceil(len(train_ds) / GRAD_ACCUM)
eval_every = max(25, steps_per_epoch // 6)
warmup_steps = max(10, int(0.05 * steps_per_epoch))

requested_args = dict(
    output_dir=str(CKPT_DIR),
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_steps=warmup_steps,
    num_train_epochs=1,
    weight_decay=0.01,
    max_grad_norm=1.0,
    optim="adamw_8bit",
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=eval_every,
    save_strategy="steps",
    save_steps=eval_every,
    save_total_limit=None,
    seed=SEED,
    report_to="none",
    remove_unused_columns=False,
    label_names=["labels"],
    dataloader_num_workers=0,
)

valid_arg_names = {f.name for f in dataclasses.fields(TrainingArguments)}
essential_args = {
    "output_dir", "per_device_train_batch_size", "gradient_accumulation_steps",
    "learning_rate", "num_train_epochs", "eval_strategy", "eval_steps",
    "save_strategy", "save_steps", "fp16", "bf16",
}
dropped_args = [k for k in requested_args if k not in valid_arg_names]
missing_essential = sorted(set(dropped_args) & essential_args)
if missing_essential:
    raise RuntimeError(f"This transformers version no longer accepts essential arguments: {missing_essential}")
if dropped_args:
    print(f"Ignoring TrainingArguments not supported by this transformers version: {dropped_args}")

training_args = TrainingArguments(**{k: v for k, v in requested_args.items() if k in valid_arg_names})

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=DataCollatorForSeq2Seq(
        tokenizer=tokenizer, padding=True, label_pad_token_id=-100, pad_to_multiple_of=8
    ),
    callbacks=[MonitorAndSave(val_meta)],
)

print(f"\nSteps per epoch: {steps_per_epoch} | eval/save every {eval_every} steps | "
      f"warmup {warmup_steps} steps | MAX_SEQ_LEN={MAX_SEQ_LEN}")

resume = True if list(CKPT_DIR.glob("checkpoint-*")) else None
if resume:
    print("Found existing checkpoints, resuming from the latest one.")

trainer.train(resume_from_checkpoint=resume)

model.save_pretrained(str(FINAL_DIR))
tokenizer.save_pretrained(str(FINAL_DIR))
with open(OUT_DIR / "run_config.json", "w", encoding="utf-8") as f:
    json.dump({
        "base_model": BASE_MODEL, "max_seq_len": MAX_SEQ_LEN, "seed": SEED,
        "train_counts": TRAIN_COUNTS, "val_counts": VAL_COUNTS, "n_holdout_qids": N_HOLDOUT_QIDS,
        "lora_r": LORA_R, "learning_rate": LEARNING_RATE, "grad_accum": GRAD_ACCUM,
        "train_examples": len(train_ds), "val_examples": len(val_ds),
    }, f, indent=2)

print(f"\nDone. Final adapter: {FINAL_DIR}")
print(f"Peak GPU memory: {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loading unsloth/Qwen2.5-1.5B-Instruct ...
==((====))==  Unsloth 2026.9.14: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loading clean distributions ...
{'A': 29993, 'B': 24456, 'C': 21217, 'Dh': 3252, 'Dc': 1628}
B records identical to the first turn of a C/Dh record: 24,416 / 24,456

TRAIN: 9,162 examples, 10,567,335 tokens, dropped for length > 2048: {'C': 237, 'Dh': 101}
    A: n=4,000  mean_tokens=    729  max= 1597  mean_supervised=  6.8
    C: n=3,763  mean_tokens=   1543  max= 2047  mean_supervised= 19.0
   Dc: n=  500  mean_tokens=    736  max= 1628  mean_supervised= 13.0
   Dh: n=  899  mean_tokens=   1640  max= 2044  mean_supervised= 23.2

VAL: 221 examples, 268,342 tokens, dropped for length > 2048: {'C': 8, 'Dh': 1}
    A: n=   80  mean_tokens=    757  max= 1295  mean_supervised=  6.8
    C: n=   82  mean_tokens=   1570  max= 2045  mean_supervised= 18.6
   Dc: n=   20  mean_tokens=    739  max= 1292  mean_supervised= 13.0
   Dh: n=   39  mean_tokens=   1649  max= 1983  mean_supervised= 23.0

Supervised text of one C example (everything else is masked):
Search: John de Lacy, 2nd Earl of Linco

Unsloth 2026.9.14 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820

Steps per epoch: 573 | eval/save every 95 steps | warmup 28 steps | MAX_SEQ_LEN=2048


...


[decision monitor] step 570 | A:t1 75% | A:t1:refuse 98% | C:t1 85% | C:t2 100% | Dc:t1 95% | Dc:t1:refuse 90% | Dh:t1 85% | Dh:t2 95% | Dh:t2:refuse 100% | worst 75% -> RED


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/hotpotqa-rag/finetune/checkpoints/checkpoint-570/tokenizer_config.json.



[decision monitor] step 573 | A:t1 75% | A:t1:refuse 98% | C:t1 85% | C:t2 100% | Dc:t1 95% | Dc:t1:refuse 90% | Dh:t1 85% | Dh:t2 95% | Dh:t2:refuse 100% | worst 75% -> RED


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/hotpotqa-rag/finetune/checkpoints/checkpoint-573/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/hotpotqa-rag/finetune/final_adapter/tokenizer_config.json.



Done. Final adapter: /content/drive/MyDrive/hotpotqa-rag/finetune/final_adapter
Peak GPU memory: 3.6 GB


In [ ]:
# Cell 15b: Continue training from a saved LoRA adapter for one more epoch (Colab T4)
# Same session as Cell 15, no restart needed, environment already verified clean  using uv .

import importlib.util
assert importlib.util.find_spec("xformers") is None, (
    "xformers is present. This should not happen in the same session as Cell 15."
)

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from unsloth import FastLanguageModel, is_bfloat16_supported

import dataclasses
import gc
import json
import math
import random
from collections import Counter, defaultdict
from pathlib import Path

import torch
from peft import PeftModel
from torch.utils.data import Dataset
from transformers import (
    DataCollatorForSeq2Seq,
    Trainer,
    TrainerCallback,
    TrainingArguments,
)

torch.backends.cuda.enable_mem_efficient_sdp(True)

for _name in ("trainer", "model"):
    if _name in globals():
        del globals()[_name]
gc.collect()
torch.cuda.empty_cache()

BASE_DIR = Path("/content/drive/MyDrive/hotpotqa-rag")
CLEAN_DIR = BASE_DIR / "data" / "distributions" / "clean"
OUT_DIR = BASE_DIR / "finetune"
MONITOR_LOG = OUT_DIR / "monitor_log.jsonl"
HOLDOUT_PATH = OUT_DIR / "holdout_records.jsonl"

RUN_TAG = "epoch2"
PREV_ADAPTER_DIR = OUT_DIR / "final_adapter"
CKPT_DIR = OUT_DIR / f"checkpoints_{RUN_TAG}"
FINAL_DIR = OUT_DIR / f"final_adapter_{RUN_TAG}"
CKPT_DIR.mkdir(parents=True, exist_ok=True)

BASE_MODEL = "unsloth/Qwen2.5-1.5B-Instruct"
MAX_SEQ_LEN = 2048
SEED = 3407
SAMPLE_SEED = 5407

N_HOLDOUT_QIDS = 400
TRAIN_COUNTS = {"A": 4000, "B": 0, "C": 4000, "Dh": 1000, "Dc": 500}
VAL_COUNTS = {"A": 80, "C": 90, "Dh": 40, "Dc": 20}

LORA_R = 16
LEARNING_RATE = 5e-5
GRAD_ACCUM = 16

AMP_DTYPE = torch.bfloat16 if is_bfloat16_supported() else torch.float16

print(f"Loading base model {BASE_MODEL} ...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
)

print(f"Attaching adapter from {PREV_ADAPTER_DIR} (trainable) ...")
model = PeftModel.from_pretrained(model, str(PREV_ADAPTER_DIR), is_trainable=True)
model.print_trainable_parameters()

IM_END_ID = tokenizer.convert_tokens_to_ids("<|im_end|>")
ANSWER_PREFIX_IDS = tokenizer.encode("Answer:", add_special_tokens=False)
ANSWER_TOK = ANSWER_PREFIX_IDS[0]
ANSWER_PREFIX_LEN = len(ANSWER_PREFIX_IDS)
REFUSAL_TOK = tokenizer.encode(
    "Answer: I cannot determine the answer from the provided context.",
    add_special_tokens=False,
)[ANSWER_PREFIX_LEN]
assert tokenizer.decode([REFUSAL_TOK]).strip() == "I", "Unexpected tokenization of the refusal sentence."

ASSIST_MARK = "<|im_start|>assistant\n"
END_MARK = "<|im_end|>"


def encode_example(messages):
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    enc = tokenizer(text, add_special_tokens=False, return_offsets_mapping=True)
    ids = enc["input_ids"]
    offsets = enc["offset_mapping"]
    if len(ids) > MAX_SEQ_LEN:
        return None

    spans = []
    pos = 0
    while True:
        s = text.find(ASSIST_MARK, pos)
        if s == -1:
            break
        content_start = s + len(ASSIST_MARK)
        e = text.find(END_MARK, content_start)
        if e == -1:
            raise RuntimeError("Assistant turn without <|im_end|>, malformed example.")
        spans.append((content_start, e + len(END_MARK)))
        pos = e + len(END_MARK)
    if not spans:
        raise RuntimeError("No assistant turn found in an example.")

    labels = [-100] * len(ids)
    span_starts = []
    si = 0
    for i, (a, b) in enumerate(offsets):
        while si < len(spans) and a >= spans[si][1]:
            si += 1
        if si == len(spans):
            break
        cs, ce = spans[si]
        if a >= cs and b <= ce and b > a:
            labels[i] = ids[i]
            if len(span_starts) == si:
                span_starts.append(i)

    n_end = sum(1 for l in labels if l == IM_END_ID)
    if len(span_starts) != len(spans) or n_end != len(spans):
        raise RuntimeError(
            "Label masking check failed: the model would not learn to stop, or a turn got no labels."
        )
    return {"input_ids": ids, "labels": labels, "span_starts": span_starts}


def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


print("Loading clean distributions ...")
raw = {name: load_jsonl(CLEAN_DIR / f"distribution_{name}.jsonl") for name in "ABCD"}
pools = {
    "A": raw["A"],
    "B": raw["B"],
    "C": raw["C"],
    "Dh": [r for r in raw["D"] if r["provenance"].get("d_type") == "bridge_failed_retrieval"],
    "Dc": [r for r in raw["D"] if r["provenance"].get("d_type") == "comparison_no_gold"],
}

rng = random.Random(SEED)
all_qids = sorted({r["provenance"]["qid"] for r in pools["A"]})
holdout_qids = set(rng.sample(all_qids, N_HOLDOUT_QIDS))

val_tagged = []
for tag, n in VAL_COUNTS.items():
    pool = [r for r in pools[tag] if r["provenance"]["qid"] in holdout_qids]
    chosen = rng.sample(pool, min(n, len(pool)))
    val_tagged.extend((tag, r) for r in chosen)

rng2 = random.Random(SAMPLE_SEED)
train_tagged = []
for tag, n in TRAIN_COUNTS.items():
    pool = [r for r in pools[tag] if r["provenance"]["qid"] not in holdout_qids]
    chosen = rng2.sample(pool, min(n, len(pool)))
    train_tagged.extend((tag, r) for r in chosen)

del raw, pools


def build_rows(tagged, title):
    rows, meta, dropped = [], [], Counter()
    for tag, rec in tagged:
        enc = encode_example(rec["messages"])
        if enc is None:
            dropped[tag] += 1
            continue
        rows.append({
            "input_ids": enc["input_ids"],
            "attention_mask": [1] * len(enc["input_ids"]),
            "labels": enc["labels"],
        })
        meta.append({"tag": tag, "input_ids": enc["input_ids"], "span_starts": enc["span_starts"]})

    by_tag = defaultdict(list)
    for row, m in zip(rows, meta):
        by_tag[m["tag"]].append((len(row["input_ids"]), sum(1 for l in row["labels"] if l != -100)))
    total_tokens = sum(len(r["input_ids"]) for r in rows)
    print(f"\n{title}: {len(rows):,} examples, {total_tokens:,} tokens, "
          f"dropped for length > {MAX_SEQ_LEN}:

SyntaxError: unterminated f-string literal (detected at line 197) (2125864473.py, line 197)

In [ ]:
# Cell 15b: Continue training from a saved LoRA adapter for one more epoch (Colab T4)
# Same session as Cell 15, no restart needed, environment already verified clean.

import importlib.util
assert importlib.util.find_spec("xformers") is None, (
    "xformers is present. This should not happen in the same session as Cell 15."
)

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from unsloth import FastLanguageModel, is_bfloat16_supported

import dataclasses
import gc
import json
import math
import random
from collections import Counter, defaultdict
from pathlib import Path

import torch
from peft import PeftModel
from torch.utils.data import Dataset
from transformers import (
    DataCollatorForSeq2Seq,
    Trainer,
    TrainerCallback,
    TrainingArguments,
)

torch.backends.cuda.enable_mem_efficient_sdp(True)

for _name in ("trainer", "model"):
    if _name in globals():
        del globals()[_name]
gc.collect()
torch.cuda.empty_cache()

BASE_DIR = Path("/content/drive/MyDrive/hotpotqa-rag")
CLEAN_DIR = BASE_DIR / "data" / "distributions" / "clean"
OUT_DIR = BASE_DIR / "finetune"
MONITOR_LOG = OUT_DIR / "monitor_log.jsonl"
HOLDOUT_PATH = OUT_DIR / "holdout_records.jsonl"

RUN_TAG = "epoch2"
PREV_ADAPTER_DIR = OUT_DIR / "final_adapter"
CKPT_DIR = OUT_DIR / f"checkpoints_{RUN_TAG}"
FINAL_DIR = OUT_DIR / f"final_adapter_{RUN_TAG}"
CKPT_DIR.mkdir(parents=True, exist_ok=True)

BASE_MODEL = "unsloth/Qwen2.5-1.5B-Instruct"
MAX_SEQ_LEN = 2048
SEED = 3407
SAMPLE_SEED = 5407

N_HOLDOUT_QIDS = 400
TRAIN_COUNTS = {"A": 4000, "B": 0, "C": 4000, "Dh": 1000, "Dc": 500}
VAL_COUNTS = {"A": 80, "C": 90, "Dh": 40, "Dc": 20}

LORA_R = 16
LEARNING_RATE = 5e-5
GRAD_ACCUM = 16

AMP_DTYPE = torch.bfloat16 if is_bfloat16_supported() else torch.float16

print(f"Loading base model {BASE_MODEL} ...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
)

print(f"Attaching adapter from {PREV_ADAPTER_DIR} (trainable) ...")
model = PeftModel.from_pretrained(model, str(PREV_ADAPTER_DIR), is_trainable=True)
model.print_trainable_parameters()

IM_END_ID = tokenizer.convert_tokens_to_ids("<|im_end|>")
ANSWER_PREFIX_IDS = tokenizer.encode("Answer:", add_special_tokens=False)
ANSWER_TOK = ANSWER_PREFIX_IDS[0]
ANSWER_PREFIX_LEN = len(ANSWER_PREFIX_IDS)
REFUSAL_TOK = tokenizer.encode(
    "Answer: I cannot determine the answer from the provided context.",
    add_special_tokens=False,
)[ANSWER_PREFIX_LEN]
assert tokenizer.decode([REFUSAL_TOK]).strip() == "I", "Unexpected tokenization of the refusal sentence."

ASSIST_MARK = "<|im_start|>assistant\n"
END_MARK = "<|im_end|>"


def encode_example(messages):
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    enc = tokenizer(text, add_special_tokens=False, return_offsets_mapping=True)
    ids = enc["input_ids"]
    offsets = enc["offset_mapping"]
    if len(ids) > MAX_SEQ_LEN:
        return None

    spans = []
    pos = 0
    while True:
        s = text.find(ASSIST_MARK, pos)
        if s == -1:
            break
        content_start = s + len(ASSIST_MARK)
        e = text.find(END_MARK, content_start)
        if e == -1:
            raise RuntimeError("Assistant turn without <|im_end|>, malformed example.")
        spans.append((content_start, e + len(END_MARK)))
        pos = e + len(END_MARK)
    if not spans:
        raise RuntimeError("No assistant turn found in an example.")

    labels = [-100] * len(ids)
    span_starts = []
    si = 0
    for i, (a, b) in enumerate(offsets):
        while si < len(spans) and a >= spans[si][1]:
            si += 1
        if si == len(spans):
            break
        cs, ce = spans[si]
        if a >= cs and b <= ce and b > a:
            labels[i] = ids[i]
            if len(span_starts) == si:
                span_starts.append(i)

    n_end = sum(1 for l in labels if l == IM_END_ID)
    if len(span_starts) != len(spans) or n_end != len(spans):
        raise RuntimeError(
            "Label masking check failed: the model would not learn to stop, or a turn got no labels."
        )
    return {"input_ids": ids, "labels": labels, "span_starts": span_starts}


def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


print("Loading clean distributions ...")
raw = {name: load_jsonl(CLEAN_DIR / f"distribution_{name}.jsonl") for name in "ABCD"}
pools = {
    "A": raw["A"],
    "B": raw["B"],
    "C": raw["C"],
    "Dh": [r for r in raw["D"] if r["provenance"].get("d_type") == "bridge_failed_retrieval"],
    "Dc": [r for r in raw["D"] if r["provenance"].get("d_type") == "comparison_no_gold"],
}

rng = random.Random(SEED)
all_qids = sorted({r["provenance"]["qid"] for r in pools["A"]})
holdout_qids = set(rng.sample(all_qids, N_HOLDOUT_QIDS))

val_tagged = []
for tag, n in VAL_COUNTS.items():
    pool = [r for r in pools[tag] if r["provenance"]["qid"] in holdout_qids]
    chosen = rng.sample(pool, min(n, len(pool)))
    val_tagged.extend((tag, r) for r in chosen)

rng2 = random.Random(SAMPLE_SEED)
train_tagged = []
for tag, n in TRAIN_COUNTS.items():
    pool = [r for r in pools[tag] if r["provenance"]["qid"] not in holdout_qids]
    chosen = rng2.sample(pool, min(n, len(pool)))
    train_tagged.extend((tag, r) for r in chosen)

del raw, pools


def build_rows(tagged, title):
    rows, meta, dropped = [], [], Counter()
    for tag, rec in tagged:
        enc = encode_example(rec["messages"])
        if enc is None:
            dropped[tag] += 1
            continue
        rows.append({
            "input_ids": enc["input_ids"],
            "attention_mask": [1] * len(enc["input_ids"]),
            "labels": enc["labels"],
        })
        meta.append({"tag": tag, "input_ids": enc["input_ids"], "span_starts": enc["span_starts"]})

    by_tag = defaultdict(list)
    for row, m in zip(rows, meta):
        by_tag[m["tag"]].append((len(row["input_ids"]), sum(1 for l in row["labels"] if l != -100)))
    total_tokens = sum(len(r["input_ids"]) for r in rows)
    print(f"\n{title}: {len(rows):,} examples, {total_tokens:,} tokens, "
          f"dropped for length > {MAX_SEQ_LEN}: {dict(dropped) if dropped else 'none'}")
    for tag in sorted(by_tag):
        lens = [x[0] for x in by_tag[tag]]
        sup = [x[1] for x in by_tag[tag]]
        print(f"  {tag:>3}: n={len(lens):>5,}  mean_tokens={sum(lens)/len(lens):>7.0f}  "
              f"max={max(lens):>5}  mean_supervised={sum(sup)/len(sup):>5.1f}")
    return rows, meta


train_rows, train_meta = build_rows(train_tagged, "TRAIN")
val_rows, val_meta = build_rows(val_tagged, "VAL")


class ListDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        return self.rows[i]


train_ds = ListDataset(train_rows)
val_ds = ListDataset(val_rows)


class MonitorAndSave(TrainerCallback):
    def __init__(self, examples):
        self.examples = examples

    def on_save(self, args, state, control, **kwargs):
        ckpt = Path(args.output_dir) / f"checkpoint-{state.global_step}"
        if ckpt.exists():
            tokenizer.save_pretrained(str(ckpt))

    def on_evaluate(self, args, state, control, model=None, **kwargs):
        if model is None:
            return
        stats = defaultdict(lambda: [0, 0])
        device = next(model.parameters()).device
        for ex in self.examples:
            ids = ex["input_ids"]
            x = torch.tensor([ids], device=device)
            with torch.no_grad(), torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
                logits = model(input_ids=x, use_cache=False).logits[0]
            pred = logits.argmax(-1).tolist()
            del logits
            for k, s in enumerate(ex["span_starts"]):
                key = f"{ex['tag']}:t{k + 1}"
                stats[key][0] += int(pred[s - 1] == ids[s])
                stats[key][1] += 1
                if ids[s] == ANSWER_TOK:
                    r = s + ANSWER_PREFIX_LEN
                    if r < len(ids):
                        true_ref = ids[r] == REFUSAL_TOK
                        pred_ref = pred[r - 1] == REFUSAL_TOK
                        rkey = key + ":refuse"
                        stats[rkey][0] += int(true_ref == pred_ref)
                        stats[rkey][1] += 1

        rates = {k: v[0] / v[1] for k, v in stats.items() if v[1] > 0}
        worst = min(rates.values()) if rates else 0.0
        status = "OK" if worst >= 0.95 else ("WATCH" if worst >= 0.85 else "RED")
        line = " | ".join(f"{k} {rates[k] * 100:.0f}%" for k in sorted(rates))
        print(f"\n[decision monitor:{RUN_TAG}] step {state.global_step} | {line} | worst {worst * 100:.0f}% -> {status}")
        with open(MONITOR_LOG, "a", encoding="utf-8") as f:
            f.write(json.dumps({"run": RUN_TAG, "step": state.global_step, "status": status, "rates": rates}) + "\n")
        torch.cuda.empty_cache()


steps_per_epoch = math.ceil(len(train_ds) / GRAD_ACCUM)
eval_every = max(25, steps_per_epoch // 6)
warmup_steps = max(10, int(0.05 * steps_per_epoch))

requested_args = dict(
    output_dir=str(CKPT_DIR),
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_steps=warmup_steps,
    num_train_epochs=1,
    weight_decay=0.01,
    max_grad_norm=1.0,
    optim="adamw_8bit",
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=eval_every,
    save_strategy="steps",
    save_steps=eval_every,
    save_total_limit=None,
    seed=SEED,
    report_to="none",
    remove_unused_columns=False,
    label_names=["labels"],
    dataloader_num_workers=0,
)

valid_arg_names = {f.name for f in dataclasses.fields(TrainingArguments)}
essential_args = {
    "output_dir", "per_device_train_batch_size", "gradient_accumulation_steps",
    "learning_rate", "num_train_epochs", "eval_strategy", "eval_steps",
    "save_strategy", "save_steps", "fp16", "bf16",
}
dropped_args = [k for k in requested_args if k not in valid_arg_names]
missing_essential = sorted(set(dropped_args) & essential_args)
if missing_essential:
    raise RuntimeError(f"This transformers version no longer accepts essential arguments: {missing_essential}")
if dropped_args:
    print(f"Ignoring TrainingArguments not supported by this transformers version: {dropped_args}")

training_args = TrainingArguments(**{k: v for k, v in requested_args.items() if k in valid_arg_names})

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=DataCollatorForSeq2Seq(
        tokenizer=tokenizer, padding=True, label_pad_token_id=-100, pad_to_multiple_of=8
    ),
    callbacks=[MonitorAndSave(val_meta)],
)

print(f"\n[{RUN_TAG}] Steps per epoch: {steps_per_epoch} | eval/save every {eval_every} steps | "
      f"warmup {warmup_steps} steps | MAX_SEQ_LEN={MAX_SEQ_LEN}")

resume = True if list(CKPT_DIR.glob("checkpoint-*")) else None
if resume:
    print(f"Found existing checkpoints for {RUN_TAG}, resuming from the latest one.")

trainer.train(resume_from_checkpoint=resume)

model.save_pretrained(str(FINAL_DIR))
tokenizer.save_pretrained(str(FINAL_DIR))
with open(OUT_DIR / f"run_config_{RUN_TAG}.json", "w", encoding="utf-8") as f:
    json.dump({
        "run_tag": RUN_TAG, "warm_started_from": str(PREV_ADAPTER_DIR),
        "base_model": BASE_MODEL, "max_seq_len": MAX_SEQ_LEN, "seed": SEED, "sample_seed": SAMPLE_SEED,
        "train_counts": TRAIN_COUNTS, "val_counts": VAL_COUNTS, "n_holdout_qids": N_HOLDOUT_QIDS,
        "lora_r": LORA_R, "learning_rate": LEARNING_RATE, "grad_accum": GRAD_ACCUM,
        "train_examples": len(train_ds), "val_examples": len(val_ds),
    }, f, indent=2)

print(f"\nDone. Adapter for {RUN_TAG}: {FINAL_DIR}")
print(f"Peak GPU memory: {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")

Loading base model unsloth/Qwen2.5-1.5B-Instruct ...
==((====))==  Unsloth 2026.9.14: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Attaching adapter from /content/drive/MyDrive/hotpotqa-rag/finetune/final_adapter (trainable) ...
trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820
Loading clean distributions ...

TRAIN: 9,155 examples, 10,547,101 tokens, dropped for length > 2048: {'C': 246, 'Dh': 99}
    A: n=4,000  mean_tokens=    730  max= 1882  mean_supervised=  6.9
    C: n=3,754  mean_tokens=   1539  max= 2048  mean_supervised= 18.8
   Dc: n=  500  mean_tokens=    738  max= 1638  mean_supervised= 13.0
   Dh: n=  901  mean_tokens=   1646  max= 2043  mean_supervised= 23.1

VAL: 218 examples, 259,680 tokens, dropped for length > 2048: {'C': 11, 'Dh': 1}
    A: n=   80  mean_tokens=    749  max= 1339  mean_supervised=  7.4
    C: n=   79  mean_tokens=   1528  max= 2040  mean_supervised= 19.0
   Dc: n=   20  mean_tokens=    753  max= 1292  mean_supervised= 13.0
   Dh: n=   39  mean_tokens=   1641  max= 1983  mean_supervised= 23.0

[epoch2] Steps per epoch: 573 | eval/save every 95 steps

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 9,155 | Num Epochs = 1 | Total steps = 573
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 16
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 16 x 1) = 16
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)


Step,Training Loss,Validation Loss
95,0.486904,0.446507
190,0.469100,0.421505


...


[decision monitor:epoch2] step 570 | A:t1 79% | A:t1:refuse 99% | C:t1 91% | C:t2 99% | Dc:t1 80% | Dc:t1:refuse 90% | Dh:t1 87% | Dh:t2 97% | Dh:t2:refuse 100% | worst 79% -> RED


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/hotpotqa-rag/finetune/checkpoints_epoch2/checkpoint-570/tokenizer_config.json.



[decision monitor:epoch2] step 573 | A:t1 79% | A:t1:refuse 99% | C:t1 91% | C:t2 99% | Dc:t1 80% | Dc:t1:refuse 90% | Dh:t1 87% | Dh:t2 97% | Dh:t2:refuse 100% | worst 79% -> RED


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/hotpotqa-rag/finetune/checkpoints_epoch2/checkpoint-573/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/hotpotqa-rag/finetune/final_adapter_epoch2/tokenizer_config.json.



Done. Adapter for epoch2: /content/drive/MyDrive/hotpotqa-rag/finetune/final_adapter_epoch2
Peak GPU memory: 14.5 GB


In [ ]:
# Cell 16: Behavior evaluation by generation on held-out questions

from unsloth import FastLanguageModel

import gc
import json
import random
import re
import string
from collections import defaultdict
from pathlib import Path

import torch

for _name in ("trainer", "model"):
    if _name in globals():
        del globals()[_name]
gc.collect()
torch.cuda.empty_cache()

if not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")

BASE_DIR = Path("/content/drive/MyDrive/hotpotqa-rag")
OUT_DIR = BASE_DIR / "finetune"
HOLDOUT_PATH = OUT_DIR / "holdout_records.jsonl"

# Evaluating epoch2, the adapter we just finished and want to decide on.
# Point this at OUT_DIR / "final_adapter" to compare back against epoch 1,
# or at a specific checkpoint-N folder under checkpoints_epoch2.
CHECKPOINT_TO_EVAL = OUT_DIR / "final_adapter_epoch2"

MAX_SEQ_LEN = 2048  # matches what epoch2 was actually trained at
PER_BUCKET = 50
MAX_NEW_TOKENS = 48
SEED = 7

print(f"Loading {CHECKPOINT_TO_EVAL} ...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=str(CHECKPOINT_TO_EVAL),
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)
model.generation_config.max_length = None

IM_END_ID = tokenizer.convert_tokens_to_ids("<|im_end|>")
PROSE = re.compile(
    r"\b(i need|i want|please|find information|search for|look for|according to|i think|"
    r"can you|could you|help me|tell me|what is|where is)\b",
    re.IGNORECASE,
)
PUNCT = set(string.punctuation)

by_tag = defaultdict(list)
with open(HOLDOUT_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            rec = json.loads(line)
            by_tag[rec["tag"]].append(rec)

rng = random.Random(SEED)
sampled = {tag: rng.sample(recs, min(PER_BUCKET, len(recs))) for tag, recs in by_tag.items()}

# name, tag, number of messages in the prompt, expected prefix, expects refusal
BUCKETS = [
    ("A:t1", "A", 1, "Answer:", False),
    ("B:t1", "B", 1, "Search:", None),
    ("C:t1", "C", 1, "Search:", None),
    ("C:t2", "C", 3, "Synthesis:", False),
    ("Dh:t1", "Dh", 1, "Search:", None),
    ("Dh:t2", "Dh", 3, "Answer:", True),
    ("Dc:t1", "Dc", 1, "Answer:", True),
]


def norm(s):
    s = "".join(ch for ch in s.lower() if ch not in PUNCT)
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(s.split())


def generate(messages):
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt", add_special_tokens=False).to("cuda")
    if inputs["input_ids"].shape[1] > MAX_SEQ_LEN - MAX_NEW_TOKENS:
        return None
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            eos_token_id=IM_END_ID,
            pad_token_id=tokenizer.pad_token_id,
            use_cache=True,
        )
    new_tokens = out[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


results = defaultdict(list)
skipped = 0
for name, tag, n_prompt, prefix, expects_refusal in BUCKETS:
    print(f"Generating {name} ...")
    for rec in sampled.get(tag, []):
        msgs = rec["messages"]
        out = generate(msgs[:n_prompt])
        if out is None:
            skipped += 1
            continue

        prefix_ok = out.startswith(prefix)
        body = out[len(prefix):].strip() if prefix_ok else out
        item = {
            "out": out,
            "prefix_ok": prefix_ok,
            "refusal": "cannot determine" in out.lower(),
            "leak": prefix != "Search:" and "Search:" in body,
        }

        if prefix == "Search:":
            prov = rec["provenance"]
            withheld = prov.get("withheld_title") or (prov.get("hop_order") or ["", ""])[1]
            item["pure"] = (
                prefix_ok
                and not PROSE.search(body)
                and len(body.split()) <= 12
                and withheld.lower() in body.lower()
            )

        if name in ("A:t1", "C:t2"):
            target = msgs[n_prompt]["content"]
            gold_prefix = "Answer:" if tag == "A" else "Synthesis:"
            gold = target[len(gold_prefix):].strip()
            item["em"] = norm(body) == norm(gold)
            item["contains"] = norm(gold) in norm(body)

        results[name].append(item)


def rate(items, key):
    vals = [i[key] for i in items if key in i]
    return 100.0 * sum(vals) / len(vals) if vals else float("nan")


def pooled(names):
    return [i for n in names for i in results[n]]


print(f"\n{'=' * 78}")
print(f"BEHAVIOR EVAL: {CHECKPOINT_TO_EVAL.name} | skipped (too long): {skipped}")
print(f"{'=' * 78}")
print(f"{'bucket':<8}{'n':>5}{'prefix':>9}{'pure':>8}{'leak':>8}{'refuse':>9}{'EM':>8}{'contains':>10}")
for name, *_ in BUCKETS:
    items = results[name]
    print(f"{name:<8}{len(items):>5}{rate(items, 'prefix_ok'):>8.0f}%"
          f"{rate(items, 'pure'):>7.0f}%{rate(items, 'leak'):>7.0f}%{rate(items, 'refusal'):>8.0f}%"
          f"{rate(items, 'em'):>7.0f}%{rate(items, 'contains'):>9.0f}%")

all_names = [b[0] for b in BUCKETS]
answer_names = ["A:t1", "C:t2", "Dh:t2", "Dc:t1"]
gates = [
    ("Prefix adherence >= 98%", rate(pooled(all_names), "prefix_ok"), lambda v: v >= 98),
    ("Query purity >= 95%", rate(pooled(["B:t1", "C:t1", "Dh:t1"]), "pure"), lambda v: v >= 95),
    ("Search leaking into answers <= 2%", rate(pooled(answer_names), "leak"), lambda v: v <= 2),
    ("False refusal on answerable <= 5%", rate(pooled(["A:t1", "C:t2"]), "refusal"), lambda v: v <= 5),
    ("Refusal on unanswerable >= 90%", rate(pooled(["Dh:t2", "Dc:t1"]), "refusal"), lambda v: v >= 90),
]
print(f"\n{'-' * 78}")
for desc, value, ok_fn in gates:
    print(f"{'PASS' if ok_fn(value) else 'FAIL'}  {desc}: {value:.1f}%")
qa = pooled(["A:t1", "C:t2"])
print(f"\nAnswer quality on answerable questions: EM {rate(qa, 'em'):.1f}% | "
      f"gold answer contained {rate(qa, 'contains'):.1f}%")

print(f"\n{'-' * 78}\nSample outputs (first 2 per bucket):")
for name, *_ in BUCKETS:
    for item in results[name][:2]:
        print(f"[{name}] {item['out'][:160]}")

with open(OUT_DIR / f"behavior_eval_{CHECKPOINT_TO_EVAL.name}.json", "w", encoding="utf-8") as f:
    json.dump({name: results[name] for name in all_names}, f, ensure_ascii=False, indent=2)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loading /content/drive/MyDrive/hotpotqa-rag/finetune/final_adapter_epoch2 ...
==((====))==  Unsloth 2026.9.14: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Unsloth 2026.9.14 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Generating A:t1 ...
Generating B:t1 ...
Generating C:t1 ...
Generating C:t2 ...
Generating Dh:t1 ...
Generating Dh:t2 ...
Generating Dc:t1 ...

BEHAVIOR EVAL: final_adapter_epoch2 | skipped (too long): 4
bucket      n   prefix    pure    leak   refuse      EM  contains
A:t1       50      80%    nan%     20%       0%     52%       58%
B:t1       50      88%     46%      0%       0%    nan%      nan%
C:t1       50      86%     46%      0%       0%    nan%      nan%
C:t2       47     100%    nan%      0%       0%     64%       79%
Dh:t1      42      88%     43%      0%       2%    nan%      nan%
Dh:t2      41      93%    nan%      0%      95%    nan%      nan%
Dc:t1      21      81%    nan%     19%      76%    nan%      nan%

------------------------------------------------------------------------------
FAIL  Prefix adherence >= 98%: 88.4%
FAIL  Query purity >= 95%: 45.1%
FAIL  Search leaking into answers <= 2%: 8.8%
PASS  False refusal on answerable <= 5%: 0.0%
FAIL  Refusal on unanswera

In [ ]:
!pip install -q uv
!uv pip install --system -q unsloth faiss-cpu sentence-transformers fastapi "uvicorn[standard]"

# Remove xformers immediately, before anything imports it. Its backward
# kernel for this model's attention layout has no working path on a T4,
# confirmed four times now across different torch/transformers builds
# with the identical error. Stripping it here also means Cell 17's
# generation calls use the same SDPA path as training, rather than an
# unverified assumption that xformers' forward kernels behave fine on
# this GPU even though its backward kernels don't.
!pip uninstall -y -q xformers

import importlib.util
assert importlib.util.find_spec("xformers") is None, (
    "xformers is still present after uninstall. Stop here and paste this output."
)
print("Environment ready: unsloth, faiss, sentence-transformers, fastapi, and "
      "uvicorn installed, xformers removed, nothing imported yet.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.6/20.6 MB 68.2 MB/s eta 0:00:00
Environment ready: unsloth, faiss, sentence-transformers, fastapi, and uvicorn installed, xformers removed, nothing imported yet.


In [ ]:
import gc
import json
import threading
import time
from pathlib import Path
from typing import Dict, List, Tuple

import faiss
import torch
import uvicorn
from fastapi import FastAPI
from pydantic import BaseModel
from sentence_transformers import CrossEncoder, SentenceTransformer
from unsloth import FastLanguageModel

for _name in ("model", "bge_model", "reranker_model"):
    if _name in globals():
        del globals()[_name]
gc.collect()
torch.cuda.empty_cache()

if not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")

# ---------------------------------------------------------------------------
# Configuration, matching Cell 12 exactly
# ---------------------------------------------------------------------------
BASE_DIR = Path("/content/drive/MyDrive/hotpotqa-rag")
CHUNK_INDEX_PATH = BASE_DIR / "data" / "chunk_index.json"
FAISS_INDEX_PATH = BASE_DIR / "data" / "faiss_index.ivfpq"
CHUNK_ID_MAP_PATH = BASE_DIR / "data" / "chunk_id_map.json"
ADAPTER_DIR = BASE_DIR / "finetune" / "final_adapter_epoch2"

BASE_MODEL = "unsloth/Qwen2.5-1.5B-Instruct"
BGE_MODEL = "BAAI/bge-large-en-v1.5"
RERANKER_MODEL = "BAAI/bge-reranker-large"
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "

FAISS_NPROBE = 32
ANN_CANDIDATE_POOL = 50
TOP_K = 5
CROSS_ENCODER_BATCH_SIZE = 32

MAX_SEQ_LEN = 2048
MAX_NEW_TOKENS = 48

# ---------------------------------------------------------------------------
# Load corpus, retrieval models
# ---------------------------------------------------------------------------
print("Loading chunk index and FAISS ...")
with open(CHUNK_INDEX_PATH, "r", encoding="utf-8") as f:
    chunk_index: Dict[str, List[Dict]] = json.load(f)
with open(CHUNK_ID_MAP_PATH, "r", encoding="utf-8") as f:
    chunk_id_map: List[str] = json.load(f)
chunk_lookup: Dict[str, Dict] = {
    c["chunk_id"]: c for chunks_for_title in chunk_index.values() for c in chunks_for_title
}

faiss_index = faiss.read_index(str(FAISS_INDEX_PATH))
faiss_index.nprobe = FAISS_NPROBE
print(f"FAISS loaded: {faiss_index.ntotal:,} vectors, nprobe={FAISS_NPROBE}")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading BGE embedding model: {BGE_MODEL}")
bge_model = SentenceTransformer(BGE_MODEL, device=device)

print(f"Loading cross-encoder reranker: {RERANKER_MODEL}")
reranker_model = CrossEncoder(RERANKER_MODEL, max_length=512, device=device)

print(f"Loading fine-tuned model: {ADAPTER_DIR}")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=str(ADAPTER_DIR),
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)
model.generation_config.max_length = None  # avoid the max_new_tokens/max_length warning
IM_END_ID = tokenizer.convert_tokens_to_ids("<|im_end|>")

generation_lock = threading.Lock()

# ---------------------------------------------------------------------------
# Retrieval: identical mechanism for both turns, matching Cell 12's retrieve_for_c
# ---------------------------------------------------------------------------
def retrieve(query: str, ann_pool_size: int = ANN_CANDIDATE_POOL, final_k: int = TOP_K) -> List[Dict]:
    q_emb = bge_model.encode(
        [QUERY_PREFIX + query], normalize_embeddings=True, convert_to_numpy=True,
    ).astype("float32")

    scores, indices = faiss_index.search(q_emb, ann_pool_size)
    candidate_ids = [chunk_id_map[idx] for idx in indices[0] if idx != -1]
    candidate_chunks = [chunk_lookup[cid] for cid in candidate_ids]
    if not candidate_chunks:
        return []

    pairs = [[query, c["text"]] for c in candidate_chunks]
    rerank_scores = reranker_model.predict(
        pairs, batch_size=CROSS_ENCODER_BATCH_SIZE, show_progress_bar=False,
    )
    ranked = sorted(zip(candidate_chunks, rerank_scores), key=lambda x: x[1], reverse=True)
    return [c for c, _ in ranked[:final_k]]


def format_context(chunks: List[Dict], start_idx: int = 1) -> str:
    lines = [f"[{i}] {c['title']}:\n{c['text']}" for i, c in enumerate(chunks, start=start_idx)]
    return "\n\n".join(lines)


# ---------------------------------------------------------------------------
# Generation and output parsing
# ---------------------------------------------------------------------------
def generate(messages: List[Dict]) -> Tuple[str, bool]:
    """Returns (generated_text, fit_ok). fit_ok is False if the prompt
    itself was already too long to leave room for MAX_NEW_TOKENS."""
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt", add_special_tokens=False).to("cuda")
    if inputs["input_ids"].shape[1] > MAX_SEQ_LEN - MAX_NEW_TOKENS:
        return "", False
    with generation_lock, torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            eos_token_id=IM_END_ID,
            pad_token_id=tokenizer.pad_token_id,
            use_cache=True,
        )
    new_tokens = out[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip(), True


def parse_turn1(raw: str) -> Tuple[str, str]:
    if raw.startswith("Answer:"):
        return "answer", raw[len("Answer:"):].strip()
    if raw.startswith("Search:"):
        return "search", raw[len("Search:"):].strip()
    return "malformed", raw.strip()


def parse_turn2(raw: str) -> Tuple[str, str]:
    if raw.startswith("Synthesis:"):
        return "synthesis", raw[len("Synthesis:"):].strip()
    if raw.startswith("Answer:"):
        return "refusal_or_answer", raw[len("Answer:"):].strip()
    return "malformed", raw.strip()


def run_pipeline(question: str) -> Dict:
    trace: Dict = {"question": question}

    turn1_chunks = retrieve(question)
    trace["turn_1_retrieval"] = [{"title": c["title"], "chunk_id": c["chunk_id"]} for c in turn1_chunks]

    turn1_user_msg = f"{format_context(turn1_chunks)}\n\nQuestion: {question}"
    messages = [{"role": "user", "content": turn1_user_msg}]

    turn1_raw, fit_ok = generate(messages)
    if not fit_ok:
        trace["mode"] = "prompt_too_long"
        trace["final_answer"] = None
        return trace
    trace["turn_1_raw_output"] = turn1_raw

    mode1, body1 = parse_turn1(turn1_raw)

    if mode1 == "answer":
        trace["mode"] = "direct_answer"
        trace["final_answer"] = body1
        trace["refused"] = False
        return trace

    if mode1 == "malformed":
        trace["mode"] = "malformed_turn1"
        trace["final_answer"] = body1
        trace["refused"] = None
        return trace

    # mode1 == "search"
    search_query = body1
    trace["search_query"] = search_query
    messages.append({"role": "assistant", "content": turn1_raw})

    turn2_chunks = retrieve(search_query)
    trace["turn_2_retrieval"] = [{"title": c["title"], "chunk_id": c["chunk_id"]} for c in turn2_chunks]

    turn2_user_msg = "Retrieved information:\n" + format_context(turn2_chunks, start_idx=1)
    messages.append({"role": "user", "content": turn2_user_msg})

    turn2_raw, fit_ok = generate(messages)
    if not fit_ok:
        trace["mode"] = "prompt_too_long_turn2"
        trace["final_answer"] = None
        return trace
    trace["turn_2_raw_output"] = turn2_raw

    mode2, body2 = parse_turn2(turn2_raw)
    if mode2 == "synthesis":
        trace["mode"] = "search_then_synthesize"
        trace["final_answer"] = body2
        trace["refused"] = False
    elif mode2 == "refusal_or_answer":
        trace["mode"] = "search_then_refuse"
        trace["final_answer"] = body2
        trace["refused"] = True
    else:
        trace["mode"] = "malformed_turn2"
        trace["final_answer"] = body2
        trace["refused"] = None

    return trace


# ---------------------------------------------------------------------------
# FastAPI app
# ---------------------------------------------------------------------------
app = FastAPI(title="Multi-hop RAG test API")


class AskRequest(BaseModel):
    question: str


@app.get("/health")
def health():
    return {"status": "ok"}


@app.post("/ask")
def ask(req: AskRequest):
    try:
        return run_pipeline(req.question)
    except Exception as e:
        return {"error": str(e)}


def _run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")


server_thread = threading.Thread(target=_run_server, daemon=True)
server_thread.start()
time.sleep(4)
print("Server starting at http://127.0.0.1:8000 (reachable from within this Colab session).")
print("Do not rerun this cell in the same session, restart the runtime instead if you need to reload.")

/usr/local/lib/python3.13/dist-packages/torchao/quantization/quant_api.py:1558: SyntaxWarning: invalid escape sequence '\.'
  * regex for parameter names, must start with `re:`, e.g. `re:language\.layers\..+\.q_proj.weight`.
/usr/local/lib/python3.13/dist-packages/unsloth/__init__.py:1606: UserWarning: WARNING: Unsloth should be imported before [transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loading chunk index and FAISS ...
FAISS loaded: 240,571 vectors, nprobe=32
Loading BGE embedding model: BAAI/bge-large-en-v1.5


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

Loading fine-tuned model: /content/drive/MyDrive/hotpotqa-rag/finetune/final_adapter_epoch2
==((====))==  Unsloth 2026.9.14: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Unsloth 2026.9.14 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Server starting at http://127.0.0.1:8000 (reachable from within this Colab session).
Do not rerun this cell in the same session, restart the runtime instead if you need to reload.


In [ ]:
import json
import random
import time
from pathlib import Path

import requests

BASE_DIR = Path("/content/drive/MyDrive/hotpotqa-rag")
DEV_PATH = BASE_DIR / "data" / "dev_subsample.json"

with open(DEV_PATH, "r", encoding="utf-8") as f:
    dev_subsample = json.load(f)

rng = random.Random(123)
sample_qids = rng.sample(list(dev_subsample.keys()), 5)

for qid in sample_qids:
    rec = dev_subsample[qid]
    print("=" * 78)
    print(f"QID: {qid} | type: {rec['type']} | level: {rec['level']}")
    print(f"Question: {rec['question']}")
    print(f"Gold answer: {rec['answer']}")

    start = time.time()
    resp = requests.post("http://127.0.0.1:8000/ask", json={"question": rec["question"]}, timeout=120)
    elapsed = time.time() - start
    result = resp.json()

    print(f"\nMode: {result.get('mode')}")
    print(f"Turn 1 retrieval: {[c['title'] for c in result.get('turn_1_retrieval', [])]}")
    if "search_query" in result:
        print(f"Search query generated: {result['search_query']}")
        print(f"Turn 2 retrieval: {[c['title'] for c in result.get('turn_2_retrieval', [])]}")
    print(f"\nModel's final answer: {result.get('final_answer')}")
    print(f"({elapsed:.1f}s)")
    print()

QID: 1981f2c3c44461fd555e9a4f943d9008 | type: bridge | level: hard
Question: Tamanchey is a comedy film that starred the actress who works in what type of films?
Gold answer: Hindi

Mode: search_then_synthesize
Turn 1 retrieval: ['Tamanchey', 'Kristen Wiig', 'Bipasha Basu', 'Charlize Theron', 'Anushka Shetty']
Search query generated: Richa Chadda worked films type
Turn 2 retrieval: ['Richa Chadda', 'Dev.D', 'Parineeta (2005 film)', 'Dear Zindagi', 'Jazbaa']

Model's final answer: Hindi films
(11.4s)

QID: c5eb1a2121dcf4255e14ca9f0dec366e | type: bridge | level: hard
Question: The subject of McGinniss' 1983 book "Fatal Vision" was convicted of murder in what year?
Gold answer: 1979

Mode: direct_answer
Turn 1 retrieval: ['The Journalist and the Murderer', 'Fatal Vision (disambiguation)', 'Richard Cottingham', 'Fatal Vision controversy', 'Joe McGinniss']

Model's final answer: 1981
(3.0s)

QID: 76e6d6289e0c8af9b43b922ffa28f167 | type: bridge | level: hard
Question: Seth MacFarlane was th

In [ ]:

# Install system-level prerequisite
!apt-get update -qq
!apt-get install -y -qq libomp-dev

# Install uv
!pip install -q uv

# Install Unsloth and CPU FAISS into the same Python environment
!uv pip install --system -q unsloth faiss-cpu

# Remove xformers before importing model libraries
!uv pip uninstall --system -y -q xformers

# Verify the environment
import importlib.util
import sys

assert importlib.util.find_spec("xformers") is None, (
    "xformers is still installed."
)

import faiss
import torch

print("Python:", sys.executable)
print("FAISS version:", faiss.__version__)
print("FAISS GPU available:", hasattr(faiss, "StandardGpuResources"))
print("CUDA available:", torch.cuda.is_available())
print("Environment ready.")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libllvm18:amd64.
(Reading database ... 127332 files and directories currently installed.)
Preparing to unpack .../libllvm18_1%3a18.1.3-1ubuntu1_amd64.deb ...
Unpacking libllvm18:amd64 (1:18.1.3-1ubuntu1) ...
Selecting previously unselected package libomp5-18:amd64.
Preparing to unpack .../libomp5-18_1%3a18.1.3-1ubuntu1_amd64.deb ...
Unpacking libomp5-18:amd64 (1:18.1.3-1ubuntu1) ...
Selecting previously unselected package libomp-18-dev.
Preparing to unpack .../libomp-18-dev_1%3a18.1.3-1ubuntu1_amd64.deb ...
Unpacking libomp-18-dev (1:18.1.3-1ubuntu1) ...
Selecting previously unselected package libomp-dev:amd64.
Preparing to unpack .../libomp-dev_1%3a18.0-59~exp2_amd64.deb ...
Unpacking libomp-dev:amd64 (1:18.0-59~exp2) ...
Setting up libllvm18:amd64 (1:18.1.3-1

In [ ]:
# Cell 17: load everything, serve it a

import gc
import json
import threading
import time
from pathlib import Path
from typing import Dict, List, Tuple

import faiss
import torch
import uvicorn
from fastapi import FastAPI
from pydantic import BaseModel
from sentence_transformers import CrossEncoder, SentenceTransformer
from unsloth import FastLanguageModel

for _name in ("model", "bge_model", "reranker_model"):
    if _name in globals():
        del globals()[_name]
gc.collect()
torch.cuda.empty_cache()

if not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")

# ---------------------------------------------------------------------------
# Configuration, matching Cell 12 exactly
# ---------------------------------------------------------------------------
BASE_DIR = Path("/content/drive/MyDrive/hotpotqa-rag")
CHUNK_INDEX_PATH = BASE_DIR / "data" / "chunk_index.json"
FAISS_INDEX_PATH = BASE_DIR / "data" / "faiss_index.ivfpq"
CHUNK_ID_MAP_PATH = BASE_DIR / "data" / "chunk_id_map.json"
ADAPTER_DIR = BASE_DIR / "finetune" / "final_adapter_epoch2"

BASE_MODEL = "unsloth/Qwen2.5-1.5B-Instruct"
BGE_MODEL = "BAAI/bge-large-en-v1.5"
RERANKER_MODEL = "BAAI/bge-reranker-large"
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "

FAISS_NPROBE = 32
ANN_CANDIDATE_POOL = 50
TOP_K = 5
CROSS_ENCODER_BATCH_SIZE = 32

MAX_SEQ_LEN = 2048
MAX_NEW_TOKENS = 48

# ---------------------------------------------------------------------------
# Load corpus, retrieval models
# ---------------------------------------------------------------------------
print("Loading chunk index and FAISS ...")
with open(CHUNK_INDEX_PATH, "r", encoding="utf-8") as f:
    chunk_index: Dict[str, List[Dict]] = json.load(f)
with open(CHUNK_ID_MAP_PATH, "r", encoding="utf-8") as f:
    chunk_id_map: List[str] = json.load(f)
chunk_lookup: Dict[str, Dict] = {
    c["chunk_id"]: c for chunks_for_title in chunk_index.values() for c in chunks_for_title
}

faiss_index = faiss.read_index(str(FAISS_INDEX_PATH))
faiss_index.nprobe = FAISS_NPROBE
print(f"FAISS loaded: {faiss_index.ntotal:,} vectors, nprobe={FAISS_NPROBE}")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading BGE embedding model: {BGE_MODEL}")
bge_model = SentenceTransformer(BGE_MODEL, device=device)

print(f"Loading cross-encoder reranker: {RERANKER_MODEL}")
reranker_model = CrossEncoder(RERANKER_MODEL, max_length=512, device=device)

print(f"Loading fine-tuned model: {ADAPTER_DIR}")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=str(ADAPTER_DIR),
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)
model.generation_config.max_length = None  # avoid the max_new_tokens/max_length warning
IM_END_ID = tokenizer.convert_tokens_to_ids("<|im_end|>")

generation_lock = threading.Lock()

# ---------------------------------------------------------------------------
# Retrieval: identical mechanism for both turns, matching Cell 12's retrieve_for_c
# ---------------------------------------------------------------------------
def retrieve(query: str, ann_pool_size: int = ANN_CANDIDATE_POOL, final_k: int = TOP_K) -> List[Dict]:
    q_emb = bge_model.encode(
        [QUERY_PREFIX + query], normalize_embeddings=True, convert_to_numpy=True,
    ).astype("float32")

    scores, indices = faiss_index.search(q_emb, ann_pool_size)
    candidate_ids = [chunk_id_map[idx] for idx in indices[0] if idx != -1]
    candidate_chunks = [chunk_lookup[cid] for cid in candidate_ids]
    if not candidate_chunks:
        return []

    pairs = [[query, c["text"]] for c in candidate_chunks]
    rerank_scores = reranker_model.predict(
        pairs, batch_size=CROSS_ENCODER_BATCH_SIZE, show_progress_bar=False,
    )
    ranked = sorted(zip(candidate_chunks, rerank_scores), key=lambda x: x[1], reverse=True)
    return [c for c, _ in ranked[:final_k]]


def format_context(chunks: List[Dict], start_idx: int = 1) -> str:
    lines = [f"[{i}] {c['title']}:\n{c['text']}" for i, c in enumerate(chunks, start=start_idx)]
    return "\n\n".join(lines)


# ---------------------------------------------------------------------------
# Generation and output parsing
# ---------------------------------------------------------------------------
def generate(messages: List[Dict]) -> Tuple[str, bool]:
    """Returns (generated_text, fit_ok). fit_ok is False if the prompt
    itself was already too long to leave room for MAX_NEW_TOKENS."""
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt", add_special_tokens=False).to("cuda")
    if inputs["input_ids"].shape[1] > MAX_SEQ_LEN - MAX_NEW_TOKENS:
        return "", False
    with generation_lock, torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            eos_token_id=IM_END_ID,
            pad_token_id=tokenizer.pad_token_id,
            use_cache=True,
        )
    new_tokens = out[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip(), True


def parse_turn1(raw: str) -> Tuple[str, str]:
    if raw.startswith("Answer:"):
        return "answer", raw[len("Answer:"):].strip()
    if raw.startswith("Search:"):
        return "search", raw[len("Search:"):].strip()
    return "malformed", raw.strip()


def parse_turn2(raw: str) -> Tuple[str, str]:
    if raw.startswith("Synthesis:"):
        return "synthesis", raw[len("Synthesis:"):].strip()
    if raw.startswith("Answer:"):
        return "refusal_or_answer", raw[len("Answer:"):].strip()
    return "malformed", raw.strip()


def run_pipeline(question: str) -> Dict:
    trace: Dict = {"question": question}

    turn1_chunks = retrieve(question)
    trace["turn_1_retrieval"] = [{"title": c["title"], "chunk_id": c["chunk_id"]} for c in turn1_chunks]

    turn1_user_msg = f"{format_context(turn1_chunks)}\n\nQuestion: {question}"
    messages = [{"role": "user", "content": turn1_user_msg}]

    turn1_raw, fit_ok = generate(messages)
    if not fit_ok:
        trace["mode"] = "prompt_too_long"
        trace["final_answer"] = None
        return trace
    trace["turn_1_raw_output"] = turn1_raw

    mode1, body1 = parse_turn1(turn1_raw)

    if mode1 == "answer":
        trace["mode"] = "direct_answer"
        trace["final_answer"] = body1
        trace["refused"] = False
        return trace

    if mode1 == "malformed":
        trace["mode"] = "malformed_turn1"
        trace["final_answer"] = body1
        trace["refused"] = None
        return trace

    # mode1 == "search"
    search_query = body1
    trace["search_query"] = search_query
    messages.append({"role": "assistant", "content": turn1_raw})

    turn2_chunks = retrieve(search_query)
    trace["turn_2_retrieval"] = [{"title": c["title"], "chunk_id": c["chunk_id"]} for c in turn2_chunks]

    turn2_user_msg = "Retrieved information:\n" + format_context(turn2_chunks, start_idx=1)
    messages.append({"role": "user", "content": turn2_user_msg})

    turn2_raw, fit_ok = generate(messages)
    if not fit_ok:
        trace["mode"] = "prompt_too_long_turn2"
        trace["final_answer"] = None
        return trace
    trace["turn_2_raw_output"] = turn2_raw

    mode2, body2 = parse_turn2(turn2_raw)
    if mode2 == "synthesis":
        trace["mode"] = "search_then_synthesize"
        trace["final_answer"] = body2
        trace["refused"] = False
    elif mode2 == "refusal_or_answer":
        trace["mode"] = "search_then_refuse"
        trace["final_answer"] = body2
        trace["refused"] = True
    else:
        trace["mode"] = "malformed_turn2"
        trace["final_answer"] = body2
        trace["refused"] = None

    return trace


# ---------------------------------------------------------------------------
# FastAPI app
# ---------------------------------------------------------------------------
app = FastAPI(title="Multi-hop RAG test API")


class AskRequest(BaseModel):
    question: str


@app.get("/health")
def health():
    return {"status": "ok"}


@app.post("/ask")
def ask(req: AskRequest):
    try:
        return run_pipeline(req.question)
    except Exception as e:
        return {"error": str(e)}


def _run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")


server_thread = threading.Thread(target=_run_server, daemon=True)
server_thread.start()
time.sleep(4)
print("Server starting at http://127.0.0.1:8000 (reachable from within this Colab session).")
print("Do not rerun this cell in the same session, restart the runtime instead if you need to reload.")

/usr/local/lib/python3.13/dist-packages/torchao/quantization/quant_api.py:1558: SyntaxWarning: invalid escape sequence '\.'
  * regex for parameter names, must start with `re:`, e.g. `re:language\.layers\..+\.q_proj.weight`.
/usr/local/lib/python3.13/dist-packages/unsloth/__init__.py:1633: UserWarning: WARNING: Unsloth should be imported before [transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loading chunk index and FAISS ...
FAISS loaded: 240,571 vectors, nprobe=32
Loading BGE embedding model: BAAI/bge-large-en-v1.5


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

Loading fine-tuned model: /content/drive/MyDrive/hotpotqa-rag/finetune/final_adapter_epoch2
==((====))==  Unsloth 2026.10.5: Fast Qwen2 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Unsloth 2026.10.5 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Server starting at http://127.0.0.1:8000 (reachable from within this Colab session).
Do not rerun this cell in the same session, restart the runtime instead if you need to reload.


In [ ]:
# Cell 18: test the fine tuned Qwen2.5 instruct model against real, untouched dev questions

# This pulls from dev_subsample.json
# the official hard-only HotpotQA split that’s never been used anywhere in this pipeline until now
# and shows you the full trace plus the real gold answer for direct comparison.

import json
import random
import time
from pathlib import Path

import requests

BASE_DIR = Path("/content/drive/MyDrive/hotpotqa-rag")
DEV_PATH = BASE_DIR / "data" / "dev_subsample.json"

with open(DEV_PATH, "r", encoding="utf-8") as f:
    dev_subsample = json.load(f)

rng = random.Random(123)
sample_qids = rng.sample(list(dev_subsample.keys()), 20)

for qid in sample_qids:
    rec = dev_subsample[qid]
    print("=" * 78)
    print(f"QID: {qid} | type: {rec['type']} | level: {rec['level']}")
    print(f"Question: {rec['question']}")
    print(f"Gold answer: {rec['answer']}")

    start = time.time()
    resp = requests.post("http://127.0.0.1:8000/ask", json={"question": rec["question"]}, timeout=120)
    elapsed = time.time() - start
    result = resp.json()

    print(f"\nMode: {result.get('mode')}")
    print(f"Turn 1 retrieval: {[c['title'] for c in result.get('turn_1_retrieval', [])]}")
    if "search_query" in result:
        print(f"Search query generated: {result['search_query']}")
        print(f"Turn 2 retrieval: {[c['title'] for c in result.get('turn_2_retrieval', [])]}")
    print(f"\nModel's final answer: {result.get('final_answer')}")
    print(f"({elapsed:.1f}s)")
    print()

QID: 1981f2c3c44461fd555e9a4f943d9008 | type: bridge | level: hard
Question: Tamanchey is a comedy film that starred the actress who works in what type of films?
Gold answer: Hindi

Mode: search_then_synthesize
Turn 1 retrieval: ['Tamanchey', 'Kristen Wiig', 'Bipasha Basu', 'Charlize Theron', 'Anushka Shetty']
Search query generated: Richa Chadda worked films type
Turn 2 retrieval: ['Richa Chadda', 'Dev.D', 'Parineeta (2005 film)', 'Dear Zindagi', 'Jazbaa']

Model's final answer: Hindi films
(8.8s)

QID: c5eb1a2121dcf4255e14ca9f0dec366e | type: bridge | level: hard
Question: The subject of McGinniss' 1983 book "Fatal Vision" was convicted of murder in what year?
Gold answer: 1979

Mode: direct_answer
Turn 1 retrieval: ['The Journalist and the Murderer', 'Fatal Vision (disambiguation)', 'Richard Cottingham', 'Fatal Vision controversy', 'Joe McGinniss']

Model's final answer: 1981
(4.0s)

QID: 76e6d6289e0c8af9b43b922ffa28f167 | type: bridge | level: hard
Question: Seth MacFarlane was the

In [ ]:
# Cell 20: Attribute errors to retrieval/search versus the model's reading.
# Requires Cell 17 to have been run in this same session.

import json
import math
import random
import re
import string
from collections import Counter, defaultdict
from pathlib import Path

import torch
from tqdm.auto import tqdm

assert "run_pipeline" in globals() and "model" in globals(), "Run Cell 17 first, in this same session."

BASE_DIR = Path("/content/drive/MyDrive/hotpotqa-rag")
DEV_PATH = BASE_DIR / "data" / "dev_subsample.json"
OUT_PATH = BASE_DIR / "finetune" / "diagnosis_results.jsonl"

N_QUESTIONS = 100
SEED = 2026
BASE_INSTRUCTION = (
    "\n\nAnswer the question using the context above. "
    "Reply with only the answer, as briefly as possible."
)

_PUNCT = set(string.punctuation)


def norm(s):
    s = "".join(ch for ch in s.lower() if ch not in _PUNCT)
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(s.split())


def is_correct(pred, gold):
    # Gold phrase appears in the prediction, or every prediction token appears in the gold
    # (catches "Richard Llewellyn" vs "Richard Dafydd Vivian Llewellyn").
    # True aliases such as "Sam Neill" vs "Nigel John Dermot Neill" still need a manual look.
    if pred is None:
        return False
    p, g = norm(pred), norm(gold)
    if not p or not g:
        return False
    if g in ("yes", "no"):
        return p == g
    pt, gt = p.split(), g.split()
    n = len(gt)
    if any(pt[i:i + n] == gt for i in range(len(pt) - n + 1)):
        return True
    return set(pt) <= set(gt)


def wilson(k, n, z=1.96):
    if n == 0:
        return float("nan"), float("nan")
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return 100 * (c - h), 100 * (c + h)


def pct(k, n):
    return 100.0 * k / n if n else float("nan")


def group(r):
    if r["mode"] == "direct_answer":
        return "direct"
    if r["mode"] in ("search_then_synthesize", "search_then_refuse"):
        return "search"
    return "other"


def gold_chunks_for(rec):
    sft = rec.get("supporting_fact_texts", {})
    found = []
    for title in rec["gold_titles"]:
        targets = list(sft.get(title, {}).values())
        match = next(
            (ch for ch in chunk_index.get(title, []) if any(t in ch["text"] for t in targets)),
            None,
        )
        if match is None:
            return None
        found.append(match)
    return found


def build_oracle(rec, trace, gold, rng):
    # Gold chunks plus the highest-ranked real retrieval results that are not gold,
    # padded from the question's own distractor titles if needed, then shuffled.
    gold_ids = {c["chunk_id"] for c in gold}
    chunks = list(gold)
    for c in trace.get("turn_1_retrieval", []):
        if len(chunks) >= 5:
            break
        if c["chunk_id"] not in gold_ids:
            chunks.append(chunk_lookup[c["chunk_id"]])
    if len(chunks) < 5:
        have = {c["chunk_id"] for c in chunks}
        pool = [
            ch for t in rec["distractor_titles"] for ch in chunk_index.get(t, [])
            if ch["chunk_id"] not in have
        ]
        rng.shuffle(pool)
        chunks += pool[: 5 - len(chunks)]
    rng.shuffle(chunks)
    return [{"role": "user", "content": f"{format_context(chunks)}\n\nQuestion: {rec['question']}"}]


def generate_prefilled(messages, prefix="Answer:"):
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True) + prefix
    inputs = tokenizer(text, return_tensors="pt", add_special_tokens=False).to("cuda")
    if inputs["input_ids"].shape[1] > MAX_SEQ_LEN - MAX_NEW_TOKENS:
        return None
    with generation_lock, torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            eos_token_id=IM_END_ID,
            pad_token_id=tokenizer.pad_token_id,
            use_cache=True,
        )
    return tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()


def base_answer(messages):
    prompt = [{"role": "user", "content": messages[0]["content"] + BASE_INSTRUCTION}]
    with model.disable_adapter():
        raw, ok = generate(prompt)
    if not ok:
        return None
    raw = raw.strip()
    return raw[len("Answer:"):].strip() if raw.startswith("Answer:") else raw


with open(DEV_PATH, "r", encoding="utf-8") as f:
    dev = json.load(f)

# Check that disabling the adapter really changes behavior.
base_ok = True
try:
    probe_q = next(iter(dev.values()))["question"]
    probe = [{"role": "user", "content": f"{format_context(retrieve(probe_q))}\n\nQuestion: {probe_q}"}]
    with_adapter, _ = generate(probe)
    with model.disable_adapter():
        without_adapter, _ = generate(probe)
    if with_adapter == without_adapter:
        print("WARNING: identical outputs with and without the adapter. Base-model numbers are unreliable.")
except Exception as e:
    print(f"Base-model comparison disabled: {e}")
    base_ok = False

# ---------------------------------------------------------------------------
# Run (resumable)
# ---------------------------------------------------------------------------
done = {}
if OUT_PATH.exists():
    with open(OUT_PATH, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                row = json.loads(line)
                done[row["qid"]] = row

qids = random.Random(SEED).sample(list(dev.keys()), min(N_QUESTIONS, len(dev)))
pending = [q for q in qids if q not in done]
print(f"{len(done)} already done, {len(pending)} pending of {len(qids)}")

with open(OUT_PATH, "a", encoding="utf-8") as out_f:
    for qid in tqdm(pending):
        rec = dev[qid]
        gold_answer = rec["answer"]
        row = {
            "qid": qid,
            "type": rec["type"],
            "question": rec["question"],
            "gold": gold_answer,
            "numeric": bool(re.search(r"\d", gold_answer)),
        }

        gold = gold_chunks_for(rec)
        if gold is None:
            row["skipped"] = "gold chunk not resolvable"
            done[qid] = row
            out_f.write(json.dumps(row, ensure_ascii=False) + "\n")
            out_f.flush()
            continue
        gold_ids = {c["chunk_id"] for c in gold}

        try:
            trace = run_pipeline(rec["question"])
        except Exception as e:
            trace = {"mode": "error", "final_answer": None, "refused": None, "error": str(e)}

        t1 = {c["chunk_id"] for c in trace.get("turn_1_retrieval", [])}
        t2 = {c["chunk_id"] for c in trace.get("turn_2_retrieval", [])}
        answer = trace.get("final_answer")
        refused = bool(trace.get("refused"))
        row.update({
            "mode": trace.get("mode"),
            "pipeline_answer": answer,
            "refused": refused,
            "search_query": trace.get("search_query"),
            "gold_in_turn1": gold_ids <= t1,
            "gold_seen_final": gold_ids <= (t1 | t2),
            "pipeline_correct": (not refused) and is_correct(answer, gold_answer),
        })

        rng = random.Random(f"{SEED}:{qid}")
        msgs = build_oracle(rec, trace, gold, rng)
        free_raw, fit = generate(msgs)
        if fit:
            mode, body = parse_turn1(free_raw)
            row["oracle_free_mode"] = mode
            forced = generate_prefilled(msgs)
            if forced is not None:
                row["oracle_forced_answer"] = forced
                row["oracle_forced_correct"] = is_correct(forced, gold_answer)
                if base_ok:
                    try:
                        b = base_answer(msgs)
                        row["oracle_base_answer"] = b
                        row["oracle_base_correct"] = is_correct(b, gold_answer)
                    except Exception as e:
                        row["oracle_base_answer"] = f"error: {e}"
                        row["oracle_base_correct"] = False

        done[qid] = row
        out_f.write(json.dumps(row, ensure_ascii=False) + "\n")
        out_f.flush()

# ---------------------------------------------------------------------------
# Report
# ---------------------------------------------------------------------------
rows = [done[q] for q in qids if q in done and not done[q].get("skipped")]
skipped = [done[q] for q in qids if q in done and done[q].get("skipped")]
n = len(rows)
orows = [r for r in rows if "oracle_forced_correct" in r]

print(f"\n{'=' * 78}\nDIAGNOSIS on {n} hard dev questions ({len(skipped)} skipped)\n{'=' * 78}")

print("\n1. HEADLINE")
k = sum(r["pipeline_correct"] for r in rows)
lo, hi = wilson(k, n)
print(f"   End-to-end pipeline:                             {pct(k, n):5.1f}%  (95% CI {lo:.0f} to {hi:.0f}, n={n})")
if orows:
    k = sum(r["oracle_forced_correct"] for r in orows)
    lo, hi = wilson(k, len(orows))
    print(f"   Reading ceiling, fine-tuned (gold guaranteed):   {pct(k, len(orows)):5.1f}%  (95% CI {lo:.0f} to {hi:.0f}, n={len(orows)})")
    brows = [r for r in orows if "oracle_base_correct" in r]
    if brows:
        k = sum(r["oracle_base_correct"] for r in brows)
        lo, hi = wilson(k, len(brows))
        print(f"   Reading ceiling, base model (adapter off):       {pct(k, len(brows)):5.1f}%  (95% CI {lo:.0f} to {hi:.0f}, n={len(brows)})")
    k = sum(r.get("oracle_free_mode") == "answer" for r in orows)
    print(f"   With full evidence, chooses to answer directly:  {pct(k, len(orows)):5.1f}%")

print("\n2. WHERE END-TO-END ERRORS COME FROM")
seen = [r for r in rows if r["gold_seen_final"]]
unseen = [r for r in rows if not r["gold_seen_final"]]
print(f"   Evidence reached the model on {len(seen)}/{n} questions ({pct(len(seen), n):.0f}%)")
print(f"   Accuracy when evidence reached the model: {pct(sum(r['pipeline_correct'] for r in seen), len(seen)):5.1f}%  (n={len(seen)})")
print(f"   Accuracy when it never did:               {pct(sum(r['pipeline_correct'] for r in unseen), len(unseen)):5.1f}%  (n={len(unseen)})")
cats = Counter()
examples = defaultdict(list)
for r in rows:
    if r["pipeline_correct"]:
        cats["correct"] += 1
        continue
    g = group(r)
    if r["gold_seen_final"]:
        if r["refused"]:
            key = "READING: refused although the evidence was in context"
        elif g == "direct":
            key = "READING: wrong direct answer, evidence in context"
        elif g == "search":
            key = "READING: wrong answer after search, evidence in context"
        else:
            key = f"OTHER ({r['mode']}), evidence in context"
    else:
        if g == "direct":
            key = "SEARCH: answered directly while evidence was missing"
        elif g == "search":
            key = "SEARCH: the search never retrieved the missing evidence"
        else:
            key = f"OTHER ({r['mode']}), evidence missing"
    cats[key] += 1
    examples[key].append(r)
for key, cnt in cats.most_common():
    print(f"   {cnt:>4}  {key}")

print("\n3. MODE DECISION vs WHAT THE FIRST RETRIEVAL ALREADY HELD")
for have, label in ((True, "first retrieval already had all gold"), (False, "first retrieval was missing gold")):
    sub = [r for r in rows if r["gold_in_turn1"] == have]
    print(f"   {label} (n={len(sub)})")
    for g_name in ("direct", "search", "other"):
        s = [r for r in sub if group(r) == g_name]
        if s:
            print(f"      {g_name:<7} n={len(s):<3} accuracy {pct(sum(r['pipeline_correct'] for r in s), len(s)):5.1f}%")

print("\n4. SEARCH QUALITY")
needed = [r for r in rows if group(r) == "search" and not r["gold_in_turn1"]]
completed = [r for r in needed if r["gold_seen_final"]]
failed = [r for r in needed if not r["gold_seen_final"]]
print(f"   Searches that were needed: {len(needed)}")
print(f"   Second retrieval completed the evidence: {len(completed)} ({pct(len(completed), len(needed)):.0f}%), "
      f"accuracy {pct(sum(r['pipeline_correct'] for r in completed), len(completed)):.0f}%")
print(f"   Second retrieval missed the evidence:    {len(failed)} ({pct(len(failed), len(needed)):.0f}%), "
      f"accuracy {pct(sum(r['pipeline_correct'] for r in failed), len(failed)):.0f}%")
unneeded = [r for r in rows if group(r) == "search" and r["gold_in_turn1"]]
missed = [r for r in rows if group(r) == "direct" and not r["gold_in_turn1"]]
print(f"   Unneeded searches (evidence already present): {len(unneeded)}, "
      f"accuracy {pct(sum(r['pipeline_correct'] for r in unneeded), len(unneeded)):.0f}%")
print(f"   Answered directly while evidence was missing:  {len(missed)}, "
      f"accuracy {pct(sum(r['pipeline_correct'] for r in missed), len(missed)):.0f}%")


def rate_str(sub, key):
    s = [r for r in sub if key in r]
    return f"{pct(sum(r[key] for r in s), len(s)):5.1f}% (n={len(s)})" if s else "n/a"


print("\n5. READING ABILITY with gold guaranteed in context (answer forced)")
for label, sub in (
    ("numeric gold answer", [r for r in orows if r["numeric"]]),
    ("non-numeric gold answer", [r for r in orows if not r["numeric"]]),
    ("bridge", [r for r in orows if r["type"] == "bridge"]),
    ("comparison", [r for r in orows if r["type"] == "comparison"]),
):
    print(f"   {label:<24} fine-tuned {rate_str(sub, 'oracle_forced_correct'):<18} base {rate_str(sub, 'oracle_base_correct')}")

print("\n6. READING FAILURES WITH GOLD GUARANTEED (up to 10)")
shown = 0
for r in orows:
    if not r["oracle_forced_correct"] and shown < 10:
        print(f"   Q: {r['question']}")
        print(f"      gold: {r['gold']} | fine-tuned: {r.get('oracle_forced_answer')} | base: {r.get('oracle_base_answer')}")
        shown += 1

print("\n7. FAILED SEARCHES (up to 8)")
shown = 0
for key, group_rows in examples.items():
    if key.startswith("SEARCH") and "search never" in key:
        for r in group_rows:
            if shown < 8:
                print(f"   Q: {r['question']}")
                print(f"      query: {r['search_query']} | gold: {r['gold']} | model: {r['pipeline_answer']}")
                shown += 1

print(f"\nPer-question results saved to: {OUT_PATH}")

0 already done, 100 pending of 100


  0%|          | 0/100 [00:00<?, ?it/s]


DIAGNOSIS on 100 hard dev questions (0 skipped)

1. HEADLINE
   End-to-end pipeline:                              51.0%  (95% CI 41 to 61, n=100)
   Reading ceiling, fine-tuned (gold guaranteed):    68.0%  (95% CI 58 to 76, n=100)
   Reading ceiling, base model (adapter off):        48.0%  (95% CI 38 to 58, n=100)
   With full evidence, chooses to answer directly:   89.0%

2. WHERE END-TO-END ERRORS COME FROM
   Evidence reached the model on 61/100 questions (61%)
   Accuracy when evidence reached the model:  82.0%  (n=61)
   Accuracy when it never did:                 2.6%  (n=39)
     51  correct
     25  SEARCH: the search never retrieved the missing evidence
      9  SEARCH: answered directly while evidence was missing
      7  READING: wrong direct answer, evidence in context
      4  OTHER (prompt_too_long_turn2), evidence missing
      3  READING: wrong answer after search, evidence in context
      1  READING: refused although the evidence was in context

3. MODE DECISION vs W

In [ ]:
# Cell 21: Is the entity in each training "Search:" target actually visible in the context shown?

import json
import re
from pathlib import Path

CLEAN_B = Path("/content/drive/MyDrive/hotpotqa-rag/data/distributions/clean/distribution_B.jsonl")


def base_title(t):
    return re.sub(r"\s*\(.*?\)", "", t).strip().lower()


total = visible = 0
not_visible_examples = []
with open(CLEAN_B, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        rec = json.loads(line)
        withheld = rec["provenance"]["withheld_title"]
        context, _, question = rec["messages"][0]["content"].rpartition("\n\nQuestion:")
        total += 1
        if base_title(withheld) in context.lower():
            visible += 1
        elif len(not_visible_examples) < 6:
            not_visible_examples.append((question.strip(), withheld, rec["messages"][1]["content"]))

print(f"B examples: {total:,}")
print(f"Entity named in the Search: target is visible in the shown context: {visible:,} ({100 * visible / total:.1f}%)")
print(f"Not visible anywhere in the context: {total - visible:,} ({100 * (total - visible) / total:.1f}%)")
print("\nExamples where it is not visible:")
for q, w, target in not_visible_examples:
    print(f"  Q: {q}\n     withheld title: {w} | training target: {target}")

B examples: 24,456
Entity named in the Search: target is visible in the shown context: 21,119 (86.4%)
Not visible anywhere in the context: 3,337 (13.6%)

Examples where it is not visible:
  Q: Noritada Saneyoshi, is a former Japanese football player and manager, in 1993, he was a member of which Japan national football team?
     withheld title: Football at the Summer Universiade | training target: Search: Football at the Summer Universiade saneyoshi japanese noritada
  Q: What film for Disney was a 1992 American animated comedy musical romantic fantasy adventure film produced by Walt Disney Feature Animation that had an American animation studio founded by Terry and Sue Shakespeare with David Molina in 1989 that was an uncredited additional animation facility for the film?
     withheld title: Creative Capers Entertainment | training target: Search: Creative Capers Entertainment shakespeare uncredited additional
  Q: Deez Nuts was a satirical candidate, portrayed by Brady C. Olson, wh

In [ ]:
# Cell 22: Retrieval-only screen of second-hop retrieval strategies.
# Requires Cell 17 in this same session. No LLM generation, only embedding and reranking.

import json
from collections import Counter
from pathlib import Path

assert "retrieve" in globals() and "reranker_model" in globals(), "Run Cell 17 first, in this same session."

BASE_DIR = Path("/content/drive/MyDrive/hotpotqa-rag")
DEV_PATH = BASE_DIR / "data" / "dev_subsample.json"
RESULTS_PATH = BASE_DIR / "finetune" / "diagnosis_results.jsonl"

with open(DEV_PATH, "r", encoding="utf-8") as f:
    dev = json.load(f)

rows = []
with open(RESULTS_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            r = json.loads(line)
            if not r.get("skipped"):
                rows.append(r)


def gold_ids_for(rec):
    sft = rec.get("supporting_fact_texts", {})
    ids = set()
    for title in rec["gold_titles"]:
        targets = list(sft.get(title, {}).values())
        match = next(
            (ch for ch in chunk_index.get(title, []) if any(t in ch["text"] for t in targets)),
            None,
        )
        if match is None:
            return None
        ids.add(match["chunk_id"])
    return ids


def ann_pool(text, size):
    q_emb = bge_model.encode(
        [QUERY_PREFIX + text], normalize_embeddings=True, convert_to_numpy=True
    ).astype("float32")
    _, idx = faiss_index.search(q_emb, size)
    return [chunk_id_map[i] for i in idx[0] if i != -1]


def rank_by(rank_query, ids, k=5):
    if not ids:
        return []
    chunks = [chunk_lookup[i] for i in ids]
    scores = reranker_model.predict(
        [[rank_query, c["text"]] for c in chunks],
        batch_size=CROSS_ENCODER_BATCH_SIZE,
        show_progress_bar=False,
    )
    ranked = sorted(zip(ids, scores), key=lambda x: x[1], reverse=True)
    return [i for i, _ in ranked[:k]]


first = Counter()
needed = []
n_all = 0
for r in rows:
    gold = gold_ids_for(dev[r["qid"]])
    if gold is None:
        continue
    n_all += 1
    q = r["question"]
    pool100 = ann_pool(q, 100)
    top5 = [c["chunk_id"] for c in retrieve(q)]
    first["top5"] += gold <= set(top5)
    first["pool50"] += gold <= set(pool100[:50])
    first["pool100"] += gold <= set(pool100)
    if r.get("search_query") and not gold <= set(top5):
        needed.append((r, gold, top5, pool100))

STRATS = [
    "S0 current: rerank the model's query pool by the query",
    "S1 same pool, rerank by the question",
    "S2 query pool + question pool, minus turn-1 chunks, rerank by question",
    "S3 same as S2, rerank by question + query",
]
hits = [Counter() for _ in STRATS]
pool_hits = Counter()
by_missing = Counter()
stored_seen = 0

for r, gold, top5, pool100 in needed:
    q, m = r["question"], r["search_query"]
    missing = gold - set(top5)
    k_missing = len(missing)
    by_missing[k_missing] += 1
    stored_seen += bool(r["gold_seen_final"])

    pool_m = ann_pool(m, 50)
    seen = set(top5)
    union = [i for i in dict.fromkeys(pool_m + pool100) if i not in seen]

    pool_hits["query pool (50)"] += missing <= set(pool_m)
    pool_hits["question pool (100), minus seen"] += missing <= set(i for i in pool100 if i not in seen)
    pool_hits["union"] += missing <= set(union)

    tops = [
        rank_by(m, pool_m),
        rank_by(q, pool_m),
        rank_by(q, union),
        rank_by(f"{q} {m}", union),
    ]
    for s, top in enumerate(tops):
        ok = missing <= set(top)
        hits[s]["all"] += ok
        hits[s][k_missing] += ok

n = len(needed)
print(f"\n{'=' * 72}\nRETRIEVAL SCREEN on {n_all} questions\n{'=' * 72}")
print("\nA. FIRST RETRIEVAL: all gold chunks found for the question")
print(f"   top 5 (what the model sees):     {first['top5']}/{n_all}")
print(f"   ANN pool of 50, before rerank:   {first['pool50']}/{n_all}")
print(f"   ANN pool of 100, before rerank:  {first['pool100']}/{n_all}")

print(f"\nB. SECOND RETRIEVAL on the {n} questions where a search was needed")
print(f"   Gold chunks still missing after retrieval 1: one chunk in {by_missing[1]} questions, "
      f"two chunks in {by_missing[2]} questions")
print("   Missing chunks present in the candidate pool (the ceiling for any reranker):")
for name, v in pool_hits.items():
    print(f"      {name:<34} {v}/{n}")
print("   Evidence completed within the 5 chunks shown to the model:")
for s, name in enumerate(STRATS):
    line = f"      {name}: {hits[s]['all']}/{n}"
    if by_missing[1]:
        line += f" | one missing: {hits[s][1]}/{by_missing[1]}"
    if by_missing[2]:
        line += f" | two missing: {hits[s][2]}/{by_missing[2]}"
    print(line)
print(f"\nSanity check: S0 completed {hits[0]['all']}/{n} here, Cell 20 recorded {stored_seen}/{n} "
      f"for the same questions. These should match closely.")


RETRIEVAL SCREEN on 100 questions

A. FIRST RETRIEVAL: all gold chunks found for the question
   top 5 (what the model sees):     49/100
   ANN pool of 50, before rerank:   51/100
   ANN pool of 100, before rerank:  60/100

B. SECOND RETRIEVAL on the 42 questions where a search was needed
   Gold chunks still missing after retrieval 1: one chunk in 34 questions, two chunks in 8 questions
   Missing chunks present in the candidate pool (the ceiling for any reranker):
      query pool (50)                    14/42
      question pool (100), minus seen    11/42
      union                              19/42
   Evidence completed within the 5 chunks shown to the model:
      S0 current: rerank the model's query pool by the query: 12/42 | one missing: 12/34 | two missing: 0/8
      S1 same pool, rerank by the question: 12/42 | one missing: 12/34 | two missing: 0/8
      S2 query pool + question pool, minus turn-1 chunks, rerank by question: 15/42 | one missing: 15/34 | two missing: 0/8
   